In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
from importlib import metadata
import json
import os
import platform
import sys
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"

if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist: {ROOT}"
    )

RESULTS.mkdir(parents=True, exist_ok=True)


def package_version(package_name: str) -> str:
    """
    Return the installed package version or 'not installed'.
    """
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return "not installed"
    except Exception as exc:
        return f"unable to determine: {exc}"


# ---------------------------------------------------------
# Operating-system and Python information
# ---------------------------------------------------------

environment_rows: list[dict[str, str]] = [
    {
        "category": "operating_system",
        "field": "system",
        "value": platform.system(),
    },
    {
        "category": "operating_system",
        "field": "release",
        "value": platform.release(),
    },
    {
        "category": "operating_system",
        "field": "version",
        "value": platform.version(),
    },
    {
        "category": "operating_system",
        "field": "platform",
        "value": platform.platform(),
    },
    {
        "category": "operating_system",
        "field": "machine_architecture",
        "value": platform.machine(),
    },
    {
        "category": "python",
        "field": "python_version",
        "value": platform.python_version(),
    },
    {
        "category": "python",
        "field": "python_implementation",
        "value": platform.python_implementation(),
    },
    {
        "category": "python",
        "field": "python_executable",
        "value": sys.executable,
    },
    {
        "category": "python",
        "field": "python_bit_width",
        "value": platform.architecture()[0],
    },
]


# ---------------------------------------------------------
# Processor and memory information
# ---------------------------------------------------------

processor_value = (
    platform.processor()
    or os.environ.get("PROCESSOR_IDENTIFIER")
    or "not reported"
)

environment_rows.append(
    {
        "category": "hardware",
        "field": "processor",
        "value": processor_value,
    }
)

environment_rows.append(
    {
        "category": "hardware",
        "field": "logical_cpu_count",
        "value": str(os.cpu_count()),
    }
)

try:
    import psutil

    physical_cores = psutil.cpu_count(logical=False)
    logical_cores = psutil.cpu_count(logical=True)
    total_memory_bytes = psutil.virtual_memory().total
    total_memory_gib = total_memory_bytes / (1024 ** 3)

    environment_rows.extend(
        [
            {
                "category": "hardware",
                "field": "physical_cpu_cores",
                "value": str(physical_cores),
            },
            {
                "category": "hardware",
                "field": "logical_cpu_cores_psutil",
                "value": str(logical_cores),
            },
            {
                "category": "hardware",
                "field": "installed_memory_gib",
                "value": f"{total_memory_gib:.2f}",
            },
            {
                "category": "software",
                "field": "psutil_version",
                "value": package_version("psutil"),
            },
        ]
    )

except ImportError:
    environment_rows.extend(
        [
            {
                "category": "hardware",
                "field": "physical_cpu_cores",
                "value": "psutil not installed",
            },
            {
                "category": "hardware",
                "field": "installed_memory_gib",
                "value": "psutil not installed",
            },
        ]
    )


# ---------------------------------------------------------
# Scientific and notebook package versions
# ---------------------------------------------------------

packages = [
    "numpy",
    "pandas",
    "scipy",
    "matplotlib",
    "jupyter",
    "jupyterlab",
    "notebook",
    "ipykernel",
    "statsmodels",
    "scikit-learn",
    "threadpoolctl",
]

for package in packages:
    environment_rows.append(
        {
            "category": "software",
            "field": f"{package}_version",
            "value": package_version(package),
        }
    )


# ---------------------------------------------------------
# Thread-related environment settings
# ---------------------------------------------------------

thread_environment_variables = [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
]

for variable in thread_environment_variables:
    environment_rows.append(
        {
            "category": "threading",
            "field": variable,
            "value": os.environ.get(variable, "not explicitly set"),
        }
    )


# ---------------------------------------------------------
# Inspect active numerical-library thread pools
# ---------------------------------------------------------

threadpool_details: list[dict] = []

try:
    from threadpoolctl import threadpool_info

    threadpool_details = threadpool_info()

    if threadpool_details:
        for index, entry in enumerate(threadpool_details, start=1):
            environment_rows.extend(
                [
                    {
                        "category": "threadpool",
                        "field": f"pool_{index}_user_api",
                        "value": str(entry.get("user_api", "")),
                    },
                    {
                        "category": "threadpool",
                        "field": f"pool_{index}_internal_api",
                        "value": str(entry.get("internal_api", "")),
                    },
                    {
                        "category": "threadpool",
                        "field": f"pool_{index}_num_threads",
                        "value": str(entry.get("num_threads", "")),
                    },
                    {
                        "category": "threadpool",
                        "field": f"pool_{index}_prefix",
                        "value": str(entry.get("prefix", "")),
                    },
                    {
                        "category": "threadpool",
                        "field": f"pool_{index}_version",
                        "value": str(entry.get("version", "")),
                    },
                ]
            )
    else:
        environment_rows.append(
            {
                "category": "threadpool",
                "field": "active_threadpools",
                "value": "none reported",
            }
        )

except Exception as exc:
    environment_rows.append(
        {
            "category": "threadpool",
            "field": "threadpool_inspection",
            "value": f"unable to inspect: {exc}",
        }
    )


# ---------------------------------------------------------
# Project and notebook execution paths
# ---------------------------------------------------------

environment_rows.extend(
    [
        {
            "category": "project",
            "field": "project_root",
            "value": str(ROOT),
        },
        {
            "category": "project",
            "field": "current_working_directory",
            "value": str(Path.cwd()),
        },
        {
            "category": "project",
            "field": "results_directory",
            "value": str(RESULTS),
        },
        {
            "category": "runtime",
            "field": "random_seed",
            "value": "2026",
        },
        {
            "category": "runtime",
            "field": "runtime_interpretation",
            "value": (
                "Local within-environment comparison; "
                "not a hardware-independent guarantee."
            ),
        },
    ]
)


# ---------------------------------------------------------
# Construct and save the environment table
# ---------------------------------------------------------

environment_df = pd.DataFrame(environment_rows)

environment_df = (
    environment_df
    .sort_values(["category", "field"])
    .reset_index(drop=True)
)

display(environment_df)

csv_output = (
    RESULTS
    / "methodology_software_hardware_environment.csv"
)

json_output = (
    RESULTS
    / "methodology_software_hardware_environment.json"
)

environment_df.to_csv(csv_output, index=False)

json_record = {
    "environment": environment_rows,
    "threadpool_details": threadpool_details,
}

with json_output.open("w", encoding="utf-8") as handle:
    json.dump(
        json_record,
        handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nEnvironment audit saved to:")
print(csv_output)
print(json_output)

In [ ]:
required_fields = [
    "system",
    "release",
    "version",
    "machine_architecture",
    "processor",
    "physical_cpu_cores",
    "logical_cpu_count",
    "installed_memory_gib",
    "python_version",
    "python_implementation",
    "numpy_version",
    "pandas_version",
    "scipy_version",
    "matplotlib_version",
    "jupyter_version",
    "jupyterlab_version",
    "notebook_version",
    "ipykernel_version",
    "threadpoolctl_version",
    "project_root",
    "current_working_directory",
    "random_seed",
]

manuscript_environment = environment_df[
    environment_df["field"].isin(required_fields)
].copy()

display(manuscript_environment)

manuscript_output = (
    RESULTS
    / "methodology_manuscript_environment_summary.csv"
)

manuscript_environment.to_csv(
    manuscript_output,
    index=False,
)

print("\nManuscript environment summary saved to:")
print(manuscript_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import ast
import json
import pandas as pd


ROOT = Path(PROJECT_ROOT)
PYTHON_DIR = ROOT / "03_Python_Optimization"
RESULTS_DIR = ROOT / "05_Results"
FIGURES_DIR = ROOT / "06_Figures"
TABLES_DIR = ROOT / "07_Tables"

for directory in (
    ROOT,
    PYTHON_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
):
    if not directory.exists():
        raise FileNotFoundError(
            f"Expected project directory not found: {directory}"
        )


# ---------------------------------------------------------
# 1. Inspect Python source modules and their function names
# ---------------------------------------------------------

source_records = []

for path in sorted(PYTHON_DIR.rglob("*.py")):
    relative_path = str(path.relative_to(ROOT))

    try:
        source_text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        tree = ast.parse(source_text)

        functions = [
            node.name
            for node in ast.walk(tree)
            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef),
            )
        ]

        classes = [
            node.name
            for node in ast.walk(tree)
            if isinstance(node, ast.ClassDef)
        ]

        lower_name = path.name.lower()

        if "development" in lower_name or "backup" in lower_name:
            source_role = "development_backup"
        elif path.name == "mst_core.py":
            source_role = "authoritative_core_module"
        else:
            source_role = "supporting_python_source"

        source_records.append(
            {
                "source_role": source_role,
                "file_name": path.name,
                "relative_path": relative_path,
                "size_bytes": path.stat().st_size,
                "function_count": len(functions),
                "functions": " | ".join(sorted(set(functions))),
                "class_count": len(classes),
                "classes": " | ".join(sorted(set(classes))),
                "parse_status": "PASS",
            }
        )

    except Exception as exc:
        source_records.append(
            {
                "source_role": "unreadable_python_source",
                "file_name": path.name,
                "relative_path": relative_path,
                "size_bytes": path.stat().st_size,
                "function_count": None,
                "functions": "",
                "class_count": None,
                "classes": "",
                "parse_status": f"ERROR: {exc}",
            }
        )

source_inventory = pd.DataFrame(source_records)

print("PYTHON SOURCE INVENTORY")
display(source_inventory)


# ---------------------------------------------------------
# 2. Inspect notebooks for imports of mst_core
# ---------------------------------------------------------

notebook_records = []

for path in sorted(PYTHON_DIR.rglob("*.ipynb")):
    relative_path = str(path.relative_to(ROOT))

    try:
        notebook = json.loads(
            path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        )

        code_cells = [
            cell
            for cell in notebook.get("cells", [])
            if cell.get("cell_type") == "code"
        ]

        complete_code = "\n".join(
            "".join(cell.get("source", []))
            for cell in code_cells
        )

        imports_mst_core = (
            "import mst_core" in complete_code
            or "from mst_core import" in complete_code
        )

        referenced_functions = [
            function_name
            for function_name in (
                "exact_mst",
                "greedy_mst",
                "progressive_mst_step",
                "progressive_mst_sequence",
                "compute_optimality_gap",
                "generate_synthetic_items",
                "run_scalability_study",
            )
            if function_name in complete_code
        ]

        lower_name = path.name.lower()

        if "audit" in lower_name:
            notebook_role = "methodology_or_result_audit"
        elif "checkpoint" in lower_name or "backup" in lower_name:
            notebook_role = "development_or_checkpoint"
        else:
            notebook_role = "experiment_notebook"

        notebook_records.append(
            {
                "notebook_role": notebook_role,
                "file_name": path.name,
                "relative_path": relative_path,
                "code_cell_count": len(code_cells),
                "imports_mst_core": imports_mst_core,
                "referenced_core_functions": " | ".join(
                    referenced_functions
                ),
                "read_status": "PASS",
            }
        )

    except Exception as exc:
        notebook_records.append(
            {
                "notebook_role": "unreadable_notebook",
                "file_name": path.name,
                "relative_path": relative_path,
                "code_cell_count": None,
                "imports_mst_core": False,
                "referenced_core_functions": "",
                "read_status": f"ERROR: {exc}",
            }
        )

notebook_inventory = pd.DataFrame(notebook_records)

print("\nNOTEBOOK INVENTORY")
display(notebook_inventory)


# ---------------------------------------------------------
# 3. Verify important evidence files named in the manuscript
# ---------------------------------------------------------

expected_result_files = [
    "progressive_mst_validation_sequence.csv",
    "progressive_mst_property_validation.csv",
    "exact_vs_greedy_counterexample.csv",
    "normalized_mst_property_validation.csv",
    "normalized_progressive_mst_sequence.csv",
    "synthetic_regime_validation.csv",
    "corrected_generator_metadata.csv",
    "corrected_generator_validation_raw.csv",
    "corrected_generator_validation_summary.csv",
    "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv",
    "coverage_threshold_denominator_audit.csv",
    "coverage_extreme_optimality_gap_rows.csv",
    "coverage_greedy_failure_mechanism_summary.csv",
    "coverage_greedy_failure_mechanism_by_regime.csv",
    "coverage_large_gap_mechanism_attribution.csv",
    "coverage_optimality_gap_tail_summary.csv",
    "coverage_threshold_denominator_descriptive_summary.csv",
    "coverage_threshold_paired_adjacent_contrasts.csv",
    "coverage_threshold_paired_bootstrap_replicates.csv",
    "corrected_v2_algorithm_comparison_long.csv",
    "corrected_v2_greedy_failure_mechanism_rows.csv",
    "corrected_v2_dynamic_static_ratio_regime_threshold_descriptive.csv",
    "corrected_v2_dynamic_vs_static_ratio_regime_threshold_bootstrap_replicates.csv",
    "methodology_exact_weight_profiles.csv",
    "methodology_weight_profile_consistency_audit.csv",
    "methodology_software_hardware_environment.csv",
    "methodology_software_hardware_environment.json",
    "methodology_manuscript_environment_summary.csv",
]

file_records = []

for file_name in expected_result_files:
    direct_path = RESULTS_DIR / file_name

    if direct_path.exists():
        found_paths = [direct_path]
    else:
        found_paths = list(ROOT.rglob(file_name))

    if found_paths:
        for found_path in found_paths:
            file_records.append(
                {
                    "expected_file_name": file_name,
                    "exists": True,
                    "relative_path": str(found_path.relative_to(ROOT)),
                    "size_bytes": found_path.stat().st_size,
                    "extension": found_path.suffix.lower(),
                }
            )
    else:
        file_records.append(
            {
                "expected_file_name": file_name,
                "exists": False,
                "relative_path": "",
                "size_bytes": None,
                "extension": Path(file_name).suffix.lower(),
            }
        )

evidence_file_audit = pd.DataFrame(file_records)

print("\nEXPECTED EVIDENCE-FILE AUDIT")
display(evidence_file_audit)


# ---------------------------------------------------------
# 4. Save all audit outputs
# ---------------------------------------------------------

source_output = (
    RESULTS_DIR
    / "methodology_python_source_inventory.csv"
)

notebook_output = (
    RESULTS_DIR
    / "methodology_notebook_inventory.csv"
)

evidence_output = (
    RESULTS_DIR
    / "methodology_expected_evidence_file_audit.csv"
)

source_inventory.to_csv(source_output, index=False)
notebook_inventory.to_csv(notebook_output, index=False)
evidence_file_audit.to_csv(evidence_output, index=False)

print("\nSaved:")
print(source_output)
print(notebook_output)
print(evidence_output)


# ---------------------------------------------------------
# 5. Show only items requiring attention
# ---------------------------------------------------------

print("\nAUTHORITATIVE OR DEVELOPMENT PYTHON SOURCES")

display(
    source_inventory[
        source_inventory["source_role"].isin(
            [
                "authoritative_core_module",
                "development_backup",
                "unreadable_python_source",
            ]
        )
    ]
)

print("\nNOTEBOOKS IMPORTING THE FINAL CORE MODULE")

display(
    notebook_inventory[
        notebook_inventory["imports_mst_core"]
        | (
            notebook_inventory["read_status"]
            != "PASS"
        )
    ]
)

print("\nMISSING EXPECTED EVIDENCE FILES")

missing_files = evidence_file_audit[
    ~evidence_file_audit["exists"]
]

if missing_files.empty:
    print("PASS: All expected evidence files were found.")
else:
    display(missing_files)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS_DIR = ROOT / "05_Results"

if not ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {ROOT}")


notebook_columns = [
    "notebook_role",
    "file_name",
    "relative_path",
    "code_cell_count",
    "imports_mst_core",
    "referenced_core_functions",
    "contains_development_backup_reference",
    "read_status",
]

notebook_records = []

for path in sorted(ROOT.rglob("*.ipynb")):

    # Ignore automatic Jupyter checkpoint copies
    if ".ipynb_checkpoints" in path.parts:
        continue

    relative_path = str(path.relative_to(ROOT))

    try:
        notebook = json.loads(
            path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        )

        code_cells = [
            cell
            for cell in notebook.get("cells", [])
            if cell.get("cell_type") == "code"
        ]

        complete_code = "\n".join(
            "".join(cell.get("source", []))
            for cell in code_cells
        )

        imports_mst_core = (
            "import mst_core" in complete_code
            or "from mst_core import" in complete_code
        )

        contains_backup_reference = (
            "mst_core_development_backup"
            in complete_code
        )

        core_function_names = [
            "validate_explanation_items",
            "subset_metrics",
            "evaluate_subset",
            "exact_mst",
            "greedy_mst",
            "progressive_mst_step",
            "progressive_mst_sequence",
            "compute_optimality_gap",
            "generate_synthetic_items",
            "run_scalability_study",
        ]

        referenced_functions = [
            function_name
            for function_name in core_function_names
            if function_name in complete_code
        ]

        lower_name = path.name.lower()

        if "methodology_audit" in lower_name:
            notebook_role = "methodology_audit"
        elif "audit" in lower_name:
            notebook_role = "other_audit"
        elif (
            "checkpoint" in lower_name
            or "backup" in lower_name
            or "development" in lower_name
        ):
            notebook_role = "development_or_checkpoint"
        else:
            notebook_role = "experiment_notebook"

        notebook_records.append(
            {
                "notebook_role": notebook_role,
                "file_name": path.name,
                "relative_path": relative_path,
                "code_cell_count": len(code_cells),
                "imports_mst_core": imports_mst_core,
                "referenced_core_functions": " | ".join(
                    referenced_functions
                ),
                "contains_development_backup_reference":
                    contains_backup_reference,
                "read_status": "PASS",
            }
        )

    except Exception as exc:
        notebook_records.append(
            {
                "notebook_role": "unreadable_notebook",
                "file_name": path.name,
                "relative_path": relative_path,
                "code_cell_count": None,
                "imports_mst_core": False,
                "referenced_core_functions": "",
                "contains_development_backup_reference": False,
                "read_status": f"ERROR: {exc}",
            }
        )


# Explicit columns prevent a KeyError when no records are found
notebook_inventory = pd.DataFrame(
    notebook_records,
    columns=notebook_columns,
)

print("NOTEBOOK INVENTORY")

if notebook_inventory.empty:
    print(
        "No notebooks were found inside the MST project. "
        "Save the current notebook under "
        "03_Python_Optimization and run this cell again."
    )
else:
    display(notebook_inventory)


print("\nNOTEBOOKS IMPORTING THE AUTHORITATIVE CORE MODULE")

if notebook_inventory.empty:
    print("No project notebooks were available for import inspection.")
else:
    importing_notebooks = notebook_inventory[
        notebook_inventory["imports_mst_core"]
        | (
            notebook_inventory["read_status"]
            != "PASS"
        )
    ]

    if importing_notebooks.empty:
        print(
            "No project notebook explicitly imports mst_core. "
            "Some notebooks may contain self-contained code."
        )
    else:
        display(importing_notebooks)


print("\nNOTEBOOKS REFERENCING THE DEVELOPMENT BACKUP")

if notebook_inventory.empty:
    print("No project notebooks were available.")
else:
    backup_references = notebook_inventory[
        notebook_inventory[
            "contains_development_backup_reference"
        ]
    ]

    if backup_references.empty:
        print(
            "PASS: No project notebook references "
            "mst_core_development_backup.py."
        )
    else:
        display(backup_references)


output_file = (
    RESULTS_DIR
    / "methodology_notebook_inventory.csv"
)

notebook_inventory.to_csv(
    output_file,
    index=False,
)

print("\nNotebook inventory saved to:")
print(output_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(PROJECT_ROOT)

SOURCE = (
    ROOT
    / "05_Results"
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

if not SOURCE.exists():
    raise FileNotFoundError(f"Benchmark not found: {SOURCE}")

df = pd.read_csv(SOURCE, low_memory=False)

print("SOURCE")
print(SOURCE)

print("\nSHAPE")
print(df.shape)

print("\nALL COLUMNS")
for index, column in enumerate(df.columns, start=1):
    print(f"{index:02d}. {column}")

# Identify columns relevant to the reported results
keywords = (
    "status",
    "feasible",
    "gap",
    "exact",
    "greedy",
    "selected",
    "count",
    "jaccard",
    "objective",
    "disclosure",
    "runtime",
    "threshold",
    "tau",
    "regime",
    "weight",
    "matched",
)

relevant_columns = [
    column
    for column in df.columns
    if any(keyword in column.lower() for keyword in keywords)
]

print("\nRELEVANT COLUMN AUDIT")

audit_rows = []

for column in relevant_columns:
    series = df[column]
    non_null = series.dropna()

    row = {
        "column": column,
        "dtype": str(series.dtype),
        "total_rows": len(series),
        "non_null_rows": len(non_null),
        "missing_rows": int(series.isna().sum()),
        "unique_values": int(non_null.nunique()),
    }

    numeric = pd.to_numeric(non_null, errors="coerce")
    finite_numeric = numeric[np.isfinite(numeric)]

    if len(finite_numeric) > 0:
        row.update(
            {
                "finite_numeric_rows": len(finite_numeric),
                "minimum": finite_numeric.min(),
                "maximum": finite_numeric.max(),
                "mean": finite_numeric.mean(),
            }
        )
    else:
        values = non_null.astype(str).unique().tolist()
        row["example_values"] = " | ".join(values[:20])

    audit_rows.append(row)

column_audit = pd.DataFrame(audit_rows)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

display(column_audit)

# Basic benchmark structure
structure_rows = [
    {
        "measure": "total_rows",
        "value": len(df),
    },
    {
        "measure": "duplicate_complete_rows",
        "value": int(df.duplicated().sum()),
    },
]

for possible_column in (
    "matched_group_id",
    "regime",
    "tau",
    "threshold",
    "weight_profile",
    "exact_status",
    "greedy_status",
):
    if possible_column in df.columns:
        structure_rows.append(
            {
                "measure": f"unique_{possible_column}",
                "value": int(df[possible_column].nunique(dropna=True)),
            }
        )

structure_audit = pd.DataFrame(structure_rows)

print("\nBENCHMARK STRUCTURE")
display(structure_audit)

column_output = (
    ROOT
    / "05_Results"
    / "results_primary_benchmark_column_audit.csv"
)

structure_output = (
    ROOT
    / "05_Results"
    / "results_primary_benchmark_structure_audit.csv"
)

column_audit.to_csv(column_output, index=False)
structure_audit.to_csv(structure_output, index=False)

print("\nSaved:")
print(column_output)
print(structure_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(PROJECT_ROOT)

SOURCE = (
    ROOT
    / "05_Results"
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

df = pd.read_csv(SOURCE, low_memory=False)


# =========================================================
# 1. Helper for selected-item strings
# =========================================================

def parse_selected_items(value) -> frozenset[str]:
    """
    Convert a pipe-separated selected-item field into a set.
    """
    if pd.isna(value):
        return frozenset()

    text = str(value).strip()

    if text == "":
        return frozenset()

    return frozenset(
        item.strip()
        for item in text.split("|")
        if item.strip()
    )


exact_sets = df["exact_selected_items"].map(
    parse_selected_items
)

greedy_sets = df["greedy_selected_items"].map(
    parse_selected_items
)


# =========================================================
# 2. Reconstruct row-level measures
# =========================================================

reconstructed_absolute_gap = (
    df["greedy_disclosure_cost"]
    - df["exact_disclosure_cost"]
)

reconstructed_gap_percent = (
    100.0
    * reconstructed_absolute_gap
    / df["exact_disclosure_cost"]
)

reconstructed_count_difference = (
    df["greedy_selected_count"]
    - df["exact_selected_count"]
)

reconstructed_same_subset = pd.Series(
    [
        exact_set == greedy_set
        for exact_set, greedy_set in zip(
            exact_sets,
            greedy_sets,
        )
    ],
    index=df.index,
)

reconstructed_jaccard = pd.Series(
    [
        (
            1.0
            if len(exact_set | greedy_set) == 0
            else len(exact_set & greedy_set)
            / len(exact_set | greedy_set)
        )
        for exact_set, greedy_set in zip(
            exact_sets,
            greedy_sets,
        )
    ],
    index=df.index,
)


# =========================================================
# 3. Row-level consistency audit
# =========================================================

absolute_gap_residual = (
    reconstructed_absolute_gap
    - df["optimality_gap"]
).abs()

percentage_gap_residual = (
    reconstructed_gap_percent
    - df["optimality_gap_percent"]
).abs()

jaccard_residual = (
    reconstructed_jaccard
    - df["jaccard_similarity"]
).abs()

consistency_rows = [
    {
        "check": "exact_status_is_OPTIMAL",
        "failed_rows": int(
            (df["exact_status"] != "OPTIMAL").sum()
        ),
        "maximum_numeric_residual": np.nan,
    },
    {
        "check": "greedy_status_is_FEASIBLE",
        "failed_rows": int(
            (df["greedy_status"] != "FEASIBLE").sum()
        ),
        "maximum_numeric_residual": np.nan,
    },
    {
        "check": "exact_utility_reaches_tau",
        "failed_rows": int(
            (df["exact_utility"] < df["tau"]).sum()
        ),
        "maximum_numeric_residual": np.nan,
    },
    {
        "check": "greedy_utility_reaches_tau",
        "failed_rows": int(
            (df["greedy_utility"] < df["tau"]).sum()
        ),
        "maximum_numeric_residual": np.nan,
    },
    {
        "check": "stored_absolute_gap_matches_reconstruction",
        "failed_rows": int(
            (absolute_gap_residual > 1e-12).sum()
        ),
        "maximum_numeric_residual": float(
            absolute_gap_residual.max()
        ),
    },
    {
        "check": "stored_percentage_gap_matches_reconstruction",
        "failed_rows": int(
            (percentage_gap_residual > 1e-10).sum()
        ),
        "maximum_numeric_residual": float(
            percentage_gap_residual.max()
        ),
    },
    {
        "check": "selected_count_difference_matches",
        "failed_rows": int(
            (
                reconstructed_count_difference
                != df["selected_count_difference"]
            ).sum()
        ),
        "maximum_numeric_residual": np.nan,
    },
    {
        "check": "same_selected_subset_matches",
        "failed_rows": int(
            (
                reconstructed_same_subset
                != df["same_selected_subset"]
            ).sum()
        ),
        "maximum_numeric_residual": np.nan,
    },
    {
        "check": "jaccard_similarity_matches",
        "failed_rows": int(
            (jaccard_residual > 1e-12).sum()
        ),
        "maximum_numeric_residual": float(
            jaccard_residual.max()
        ),
    },
    {
        "check": "no_material_negative_optimality_gap",
        "failed_rows": int(
            (df["optimality_gap"] < -1e-12).sum()
        ),
        "maximum_numeric_residual": float(
            df["optimality_gap"].min()
        ),
    },
]

consistency_audit = pd.DataFrame(
    consistency_rows
)

print("ROW-LEVEL CONSISTENCY AUDIT")
display(consistency_audit)


# =========================================================
# 4. Overall descriptive results
# =========================================================

total_rows = len(df)

identical_count = int(
    df["same_selected_subset"].sum()
)

zero_gap_count = int(
    (df["optimality_gap_percent"] == 0.0).sum()
)

large_gap_count = int(
    (df["optimality_gap_percent"] > 10.0).sum()
)

overall_summary = pd.DataFrame(
    [
        {
            "measure": "configured_rows",
            "value": total_rows,
            "unit": "rows",
        },
        {
            "measure": "exact_optimal_rows",
            "value": int(
                (df["exact_status"] == "OPTIMAL").sum()
            ),
            "unit": "rows",
        },
        {
            "measure": "greedy_feasible_rows",
            "value": int(
                (df["greedy_status"] == "FEASIBLE").sum()
            ),
            "unit": "rows",
        },
        {
            "measure": "infeasible_rows",
            "value": int(
                (
                    (df["exact_status"] != "OPTIMAL")
                    | (df["greedy_status"] != "FEASIBLE")
                ).sum()
            ),
            "unit": "rows",
        },
        {
            "measure": "mean_relative_gap_percent",
            "value": df[
                "optimality_gap_percent"
            ].mean(),
            "unit": "percent",
        },
        {
            "measure": "median_relative_gap_percent",
            "value": df[
                "optimality_gap_percent"
            ].median(),
            "unit": "percent",
        },
        {
            "measure": "maximum_relative_gap_percent",
            "value": df[
                "optimality_gap_percent"
            ].max(),
            "unit": "percent",
        },
        {
            "measure": "minimum_relative_gap_percent",
            "value": df[
                "optimality_gap_percent"
            ].min(),
            "unit": "percent",
        },
        {
            "measure": "mean_absolute_objective_gap",
            "value": df["optimality_gap"].mean(),
            "unit": "objective",
        },
        {
            "measure": "median_absolute_objective_gap",
            "value": df["optimality_gap"].median(),
            "unit": "objective",
        },
        {
            "measure": "mean_exact_objective",
            "value": df[
                "exact_disclosure_cost"
            ].mean(),
            "unit": "objective",
        },
        {
            "measure": "mean_greedy_objective",
            "value": df[
                "greedy_disclosure_cost"
            ].mean(),
            "unit": "objective",
        },
        {
            "measure": "zero_gap_rows",
            "value": zero_gap_count,
            "unit": "rows",
        },
        {
            "measure": "zero_gap_percent",
            "value": 100.0
            * zero_gap_count
            / total_rows,
            "unit": "percent",
        },
        {
            "measure": "identical_subset_rows",
            "value": identical_count,
            "unit": "rows",
        },
        {
            "measure": "identical_subset_percent",
            "value": 100.0
            * identical_count
            / total_rows,
            "unit": "percent",
        },
        {
            "measure": "gap_above_10_percent_rows",
            "value": large_gap_count,
            "unit": "rows",
        },
        {
            "measure": "gap_above_10_percent_share",
            "value": 100.0
            * large_gap_count
            / total_rows,
            "unit": "percent",
        },
        {
            "measure": "gap_percentile_95",
            "value": df[
                "optimality_gap_percent"
            ].quantile(0.95),
            "unit": "percent",
        },
        {
            "measure": "gap_percentile_99",
            "value": df[
                "optimality_gap_percent"
            ].quantile(0.99),
            "unit": "percent",
        },
        {
            "measure": "mean_jaccard_similarity",
            "value": df[
                "jaccard_similarity"
            ].mean(),
            "unit": "proportion",
        },
        {
            "measure": "mean_selected_count_difference",
            "value": df[
                "selected_count_difference"
            ].mean(),
            "unit": "items",
        },
        {
            "measure": "mean_exact_selected_count",
            "value": df[
                "exact_selected_count"
            ].mean(),
            "unit": "items",
        },
        {
            "measure": "mean_greedy_selected_count",
            "value": df[
                "greedy_selected_count"
            ].mean(),
            "unit": "items",
        },
    ]
)

print("\nOVERALL BENCHMARK SUMMARY")
display(overall_summary)


# =========================================================
# 5. Failure-mechanism classification
# =========================================================

conditions = [
    df["same_selected_subset"],
    (
        ~df["same_selected_subset"]
        & (
            df["selected_count_difference"]
            == 0
        )
    ),
    (
        df["selected_count_difference"]
        > 0
    ),
    (
        df["selected_count_difference"]
        < 0
    ),
]

labels = [
    "identical_subset",
    "same_count_different_items",
    "greedy_more_items",
    "greedy_fewer_items",
]

df["reconstructed_failure_mechanism"] = np.select(
    conditions,
    labels,
    default="UNCLASSIFIED",
)

failure_summary = (
    df
    .groupby(
        "reconstructed_failure_mechanism",
        dropna=False,
    )
    .agg(
        rows=(
            "reconstructed_failure_mechanism",
            "size",
        ),
        mean_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),
        median_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),
        maximum_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),
        mean_absolute_gap=(
            "optimality_gap",
            "mean",
        ),
        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),
        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),
    )
    .reset_index()
)

failure_summary["share_percent"] = (
    100.0
    * failure_summary["rows"]
    / total_rows
)

failure_summary = failure_summary[
    [
        "reconstructed_failure_mechanism",
        "rows",
        "share_percent",
        "mean_gap_percent",
        "median_gap_percent",
        "maximum_gap_percent",
        "mean_absolute_gap",
        "mean_jaccard_similarity",
        "mean_selected_count_difference",
    ]
]

print("\nFAILURE-MECHANISM SUMMARY")
display(failure_summary)


# =========================================================
# 6. Large-gap attribution
# =========================================================

large_gap_rows = df[
    df["optimality_gap_percent"] > 10.0
].copy()

large_gap_attribution = (
    large_gap_rows
    .groupby(
        "reconstructed_failure_mechanism",
        dropna=False,
    )
    .size()
    .rename("large_gap_rows")
    .reset_index()
)

large_gap_attribution[
    "share_of_large_gap_rows_percent"
] = (
    100.0
    * large_gap_attribution["large_gap_rows"]
    / len(large_gap_rows)
)

print("\nLARGE-GAP MECHANISM ATTRIBUTION")
display(large_gap_attribution)


# =========================================================
# 7. Save verified tables
# =========================================================

RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

RESULTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

consistency_file = (
    RESULTS
    / "results_primary_benchmark_row_consistency_audit.csv"
)

overall_file = (
    TABLES
    / "results_primary_benchmark_overall_summary_verified.csv"
)

failure_file = (
    TABLES
    / "results_primary_benchmark_failure_mechanism_verified.csv"
)

large_gap_file = (
    TABLES
    / "results_primary_benchmark_large_gap_attribution_verified.csv"
)

consistency_audit.to_csv(
    consistency_file,
    index=False,
)

overall_summary.to_csv(
    overall_file,
    index=False,
)

failure_summary.to_csv(
    failure_file,
    index=False,
)

large_gap_attribution.to_csv(
    large_gap_file,
    index=False,
)

print("\nSaved:")
print(consistency_file)
print(overall_file)
print(failure_file)
print(large_gap_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

SOURCE = (
    RESULTS
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

df = pd.read_csv(SOURCE, low_memory=False)


# ---------------------------------------------------------
# Reconstruct failure mechanism
# ---------------------------------------------------------

df["failure_mechanism"] = np.select(
    [
        df["same_selected_subset"],
        (
            ~df["same_selected_subset"]
            & (df["selected_count_difference"] == 0)
        ),
        df["selected_count_difference"] > 0,
        df["selected_count_difference"] < 0,
    ],
    [
        "identical_subset",
        "same_count_different_items",
        "greedy_more_items",
        "greedy_fewer_items",
    ],
    default="UNCLASSIFIED",
)


# ---------------------------------------------------------
# Exact top-N tails
# ---------------------------------------------------------

def summarize_tail(
    data: pd.DataFrame,
    label: str,
) -> dict:
    return {
        "tail": label,
        "rows": len(data),
        "mean_gap_percent":
            data["optimality_gap_percent"].mean(),
        "median_gap_percent":
            data["optimality_gap_percent"].median(),
        "minimum_gap_percent":
            data["optimality_gap_percent"].min(),
        "maximum_gap_percent":
            data["optimality_gap_percent"].max(),
        "mean_absolute_gap":
            data["optimality_gap"].mean(),
        "mean_exact_objective":
            data["exact_disclosure_cost"].mean(),
        "mean_jaccard_similarity":
            data["jaccard_similarity"].mean(),
        "mean_selected_count_difference":
            data["selected_count_difference"].mean(),
        "greedy_more_items_rows":
            int(
                (
                    data["failure_mechanism"]
                    == "greedy_more_items"
                ).sum()
            ),
        "greedy_more_items_percent":
            100.0
            * (
                data["failure_mechanism"]
                == "greedy_more_items"
            ).mean(),
        "same_count_different_items_percent":
            100.0
            * (
                data["failure_mechanism"]
                == "same_count_different_items"
            ).mean(),
        "greedy_fewer_items_percent":
            100.0
            * (
                data["failure_mechanism"]
                == "greedy_fewer_items"
            ).mean(),
    }


top_5_count = int(round(len(df) * 0.05))
top_1_count = int(round(len(df) * 0.01))

top_5 = df.nlargest(
    top_5_count,
    "optimality_gap_percent",
)

top_1 = df.nlargest(
    top_1_count,
    "optimality_gap_percent",
)

tail_summary = pd.DataFrame(
    [
        summarize_tail(top_5, "top_5_percent"),
        summarize_tail(top_1, "top_1_percent"),
    ]
)

print("RECONSTRUCTED TAIL SUMMARY")
display(tail_summary)


# ---------------------------------------------------------
# Threshold-specific descriptive results
# ---------------------------------------------------------

threshold_summary = (
    df
    .groupby("tau", as_index=False)
    .agg(
        configured_rows=("tau", "size"),
        exact_optimal_rows=(
            "exact_status",
            lambda values:
                int((values == "OPTIMAL").sum()),
        ),
        greedy_feasible_rows=(
            "greedy_status",
            lambda values:
                int((values == "FEASIBLE").sum()),
        ),
        mean_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),
        median_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),
        mean_absolute_gap=(
            "optimality_gap",
            "mean",
        ),
        mean_exact_objective=(
            "exact_disclosure_cost",
            "mean",
        ),
        identical_subset_rows=(
            "same_selected_subset",
            "sum",
        ),
        large_gap_rows=(
            "optimality_gap_percent",
            lambda values:
                int((values > 10.0).sum()),
        ),
        greedy_more_items_rows=(
            "selected_count_difference",
            lambda values:
                int((values > 0).sum()),
        ),
        same_count_different_rows=(
            "failure_mechanism",
            lambda values:
                int(
                    (
                        values
                        == "same_count_different_items"
                    ).sum()
                ),
        ),
        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),
        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),
    )
)

threshold_summary[
    "identical_subset_percent"
] = (
    100.0
    * threshold_summary["identical_subset_rows"]
    / threshold_summary["configured_rows"]
)

threshold_summary[
    "large_gap_percent"
] = (
    100.0
    * threshold_summary["large_gap_rows"]
    / threshold_summary["configured_rows"]
)

threshold_summary[
    "greedy_more_items_percent"
] = (
    100.0
    * threshold_summary["greedy_more_items_rows"]
    / threshold_summary["configured_rows"]
)

threshold_summary[
    "same_count_different_percent"
] = (
    100.0
    * threshold_summary["same_count_different_rows"]
    / threshold_summary["configured_rows"]
)

print("\nRECONSTRUCTED THRESHOLD SUMMARY")
display(threshold_summary)


# ---------------------------------------------------------
# Read stored derived tables for comparison
# ---------------------------------------------------------

stored_tail_file = (
    TABLES
    / "coverage_optimality_gap_tail_summary.csv"
)

stored_threshold_file = (
    TABLES
    / "coverage_threshold_denominator_descriptive_summary.csv"
)

stored_denominator_file = (
    RESULTS
    / "coverage_threshold_denominator_audit.csv"
)

print("\nSTORED TAIL SUMMARY")
stored_tail = pd.read_csv(
    stored_tail_file,
    low_memory=False,
)
display(stored_tail)

print("\nSTORED THRESHOLD DESCRIPTIVE SUMMARY")
stored_threshold = pd.read_csv(
    stored_threshold_file,
    low_memory=False,
)
display(stored_threshold)

print("\nSTORED THRESHOLD DENOMINATOR AUDIT")
stored_denominators = pd.read_csv(
    stored_denominator_file,
    low_memory=False,
)
display(stored_denominators)


# ---------------------------------------------------------
# Save reconstructed verification tables
# ---------------------------------------------------------

tail_output = (
    TABLES
    / "results_gap_tail_summary_verified.csv"
)

threshold_output = (
    TABLES
    / "results_threshold_summary_verified.csv"
)

tail_summary.to_csv(
    tail_output,
    index=False,
)

threshold_summary.to_csv(
    threshold_output,
    index=False,
)

print("\nSaved:")
print(tail_output)
print(threshold_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

CONTRAST_FILE = (
    TABLES
    / "coverage_threshold_paired_adjacent_contrasts.csv"
)

BOOTSTRAP_FILE = (
    RESULTS
    / "coverage_threshold_paired_bootstrap_replicates.csv"
)

DENOMINATOR_FILE = (
    RESULTS
    / "coverage_threshold_denominator_audit.csv"
)

for path in (
    CONTRAST_FILE,
    BOOTSTRAP_FILE,
    DENOMINATOR_FILE,
):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


contrasts = pd.read_csv(
    CONTRAST_FILE,
    low_memory=False,
)

bootstrap = pd.read_csv(
    BOOTSTRAP_FILE,
    low_memory=False,
)

denominators = pd.read_csv(
    DENOMINATOR_FILE,
    low_memory=False,
)


# =========================================================
# 1. File shapes and column names
# =========================================================

print("=" * 90)
print("PAIRED ADJACENT-THRESHOLD CONTRASTS")
print("=" * 90)

print("\nFile:")
print(CONTRAST_FILE)

print("\nShape:")
print(contrasts.shape)

print("\nColumns:")
for index, column in enumerate(
    contrasts.columns,
    start=1,
):
    print(f"{index:02d}. {column}")

print("\nComplete contrast table:")
display(contrasts)


print("\n" + "=" * 90)
print("BOOTSTRAP REPLICATES")
print("=" * 90)

print("\nFile:")
print(BOOTSTRAP_FILE)

print("\nShape:")
print(bootstrap.shape)

print("\nColumns:")
for index, column in enumerate(
    bootstrap.columns,
    start=1,
):
    print(f"{index:02d}. {column}")

print("\nFirst 20 bootstrap rows:")
display(bootstrap.head(20))


print("\n" + "=" * 90)
print("THRESHOLD DENOMINATOR AUDIT")
print("=" * 90)

print("\nFile:")
print(DENOMINATOR_FILE)

print("\nComplete denominator table:")
display(denominators)


# =========================================================
# 2. Unique-value audit
# =========================================================

def unique_value_audit(
    data: pd.DataFrame,
    maximum_values: int = 30,
) -> pd.DataFrame:

    rows = []

    for column in data.columns:
        series = data[column]
        non_null = series.dropna()

        row = {
            "column": column,
            "dtype": str(series.dtype),
            "rows": len(series),
            "non_null_rows": len(non_null),
            "missing_rows": int(series.isna().sum()),
            "unique_values": int(non_null.nunique()),
        }

        numeric = pd.to_numeric(
            non_null,
            errors="coerce",
        )

        finite = numeric[
            np.isfinite(numeric)
        ]

        if (
            len(finite) == len(non_null)
            and len(finite) > 0
        ):
            row.update(
                {
                    "minimum": float(finite.min()),
                    "maximum": float(finite.max()),
                    "mean": float(finite.mean()),
                }
            )

        if non_null.nunique() <= maximum_values:
            values = sorted(
                non_null.astype(str).unique()
            )

            row["values"] = " | ".join(values)

        rows.append(row)

    return pd.DataFrame(rows)


contrast_column_audit = unique_value_audit(
    contrasts
)

bootstrap_column_audit = unique_value_audit(
    bootstrap
)

print("\nCONTRAST COLUMN AND VALUE AUDIT")
display(contrast_column_audit)

print("\nBOOTSTRAP COLUMN AND VALUE AUDIT")
display(bootstrap_column_audit)


# =========================================================
# 3. Generic bootstrap grouping audit
# =========================================================

# Candidate identifier columns are non-numeric columns and
# low-cardinality numeric columns, excluding likely replicate
# and numerical-result columns.

identifier_keywords = (
    "contrast",
    "metric",
    "threshold",
    "tau",
    "lower",
    "higher",
    "comparison",
)

replicate_keywords = (
    "replicate",
    "bootstrap_id",
    "iteration",
    "draw",
)

value_keywords = (
    "difference",
    "estimate",
    "value",
    "statistic",
)

identifier_columns = [
    column
    for column in bootstrap.columns
    if any(
        keyword in column.lower()
        for keyword in identifier_keywords
    )
]

replicate_columns = [
    column
    for column in bootstrap.columns
    if any(
        keyword in column.lower()
        for keyword in replicate_keywords
    )
]

candidate_value_columns = [
    column
    for column in bootstrap.columns
    if any(
        keyword in column.lower()
        for keyword in value_keywords
    )
    and column not in identifier_columns
    and column not in replicate_columns
]

print("\nDetected identifier columns:")
print(identifier_columns)

print("\nDetected replicate columns:")
print(replicate_columns)

print("\nDetected candidate bootstrap-value columns:")
print(candidate_value_columns)


if identifier_columns:
    bootstrap_group_counts = (
        bootstrap
        .groupby(
            identifier_columns,
            dropna=False,
        )
        .size()
        .rename("bootstrap_rows")
        .reset_index()
    )

    print("\nBOOTSTRAP ROWS PER CONTRAST/METRIC GROUP")
    display(bootstrap_group_counts)

else:
    bootstrap_group_counts = pd.DataFrame()

    print(
        "\nNo identifier columns were detected automatically."
    )


# =========================================================
# 4. Save schema audit outputs
# =========================================================

contrast_audit_output = (
    RESULTS
    / "results_threshold_contrast_column_audit.csv"
)

bootstrap_audit_output = (
    RESULTS
    / "results_threshold_bootstrap_column_audit.csv"
)

bootstrap_groups_output = (
    RESULTS
    / "results_threshold_bootstrap_group_counts.csv"
)

contrast_column_audit.to_csv(
    contrast_audit_output,
    index=False,
)

bootstrap_column_audit.to_csv(
    bootstrap_audit_output,
    index=False,
)

bootstrap_group_counts.to_csv(
    bootstrap_groups_output,
    index=False,
)

print("\nSaved:")
print(contrast_audit_output)
print(bootstrap_audit_output)
print(bootstrap_groups_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

PRIMARY_FILE = (
    RESULTS
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

CONTRAST_FILE = (
    TABLES
    / "coverage_threshold_paired_adjacent_contrasts.csv"
)

BOOTSTRAP_FILE = (
    RESULTS
    / "coverage_threshold_paired_bootstrap_replicates.csv"
)

for path in (
    PRIMARY_FILE,
    CONTRAST_FILE,
    BOOTSTRAP_FILE,
):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


primary = pd.read_csv(PRIMARY_FILE, low_memory=False)
contrasts = pd.read_csv(CONTRAST_FILE, low_memory=False)
bootstrap = pd.read_csv(BOOTSTRAP_FILE, low_memory=False)


# =========================================================
# 1. Reconstruct row-level mechanism indicators
# =========================================================

primary["absolute_disclosure_cost_gap"] = (
    primary["greedy_disclosure_cost"]
    - primary["exact_disclosure_cost"]
)

primary["exact_cost_recovery"] = (
    primary["optimality_gap_percent"] == 0.0
)

primary["large_gap"] = (
    primary["optimality_gap_percent"] > 10.0
)

primary["more_items"] = (
    primary["selected_count_difference"] > 0
)

primary["same_count_different_items"] = (
    ~primary["same_selected_subset"]
    & (primary["selected_count_difference"] == 0)
)


# =========================================================
# 2. Direct threshold summaries from primary benchmark
# =========================================================

threshold_direct = (
    primary
    .groupby("tau", as_index=False)
    .agg(
        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),
        mean_absolute_disclosure_cost_gap=(
            "absolute_disclosure_cost_gap",
            "mean",
        ),
        mean_exact_disclosure_cost=(
            "exact_disclosure_cost",
            "mean",
        ),
        exact_cost_recovery_percent=(
            "exact_cost_recovery",
            lambda x: 100.0 * x.mean(),
        ),
        large_gap_percent=(
            "large_gap",
            lambda x: 100.0 * x.mean(),
        ),
        more_items_percent=(
            "more_items",
            lambda x: 100.0 * x.mean(),
        ),
        same_count_different_items_percent=(
            "same_count_different_items",
            lambda x: 100.0 * x.mean(),
        ),
        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),
        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),
    )
)

print("DIRECT THRESHOLD SUMMARIES")
display(threshold_direct)


# =========================================================
# 3. Reconstruct all adjacent-threshold point estimates
# =========================================================

metric_names = [
    "mean_optimality_gap_percent",
    "mean_absolute_disclosure_cost_gap",
    "mean_exact_disclosure_cost",
    "exact_cost_recovery_percent",
    "large_gap_percent",
    "more_items_percent",
    "same_count_different_items_percent",
    "mean_jaccard_similarity",
    "mean_selected_count_difference",
]

direct_rows = []

for _, stored_row in contrasts.iterrows():

    higher_tau = float(stored_row["higher_tau"])
    lower_tau = float(stored_row["lower_tau"])
    metric = stored_row["metric"]

    higher_value = threshold_direct.loc[
        np.isclose(threshold_direct["tau"], higher_tau),
        metric,
    ].iloc[0]

    lower_value = threshold_direct.loc[
        np.isclose(threshold_direct["tau"], lower_tau),
        metric,
    ].iloc[0]

    reconstructed_difference = (
        higher_value - lower_value
    )

    direct_rows.append(
        {
            "higher_tau": higher_tau,
            "lower_tau": lower_tau,
            "contrast": stored_row["contrast"],
            "metric": metric,
            "higher_threshold_value": higher_value,
            "lower_threshold_value": lower_value,
            "reconstructed_difference":
                reconstructed_difference,
        }
    )

direct_differences = pd.DataFrame(direct_rows)


# =========================================================
# 4. Reconstruct bootstrap summaries from 15,000 rows
# =========================================================

bootstrap_long = bootstrap.melt(
    id_vars=[
        "bootstrap_replicate",
        "higher_tau",
        "lower_tau",
        "contrast",
    ],
    value_vars=metric_names,
    var_name="metric",
    value_name="bootstrap_difference",
)

bootstrap_summary = (
    bootstrap_long
    .groupby(
        [
            "higher_tau",
            "lower_tau",
            "contrast",
            "metric",
        ],
        as_index=False,
    )
    .agg(
        reconstructed_bootstrap_replicates=(
            "bootstrap_difference",
            "size",
        ),
        reconstructed_bootstrap_mean=(
            "bootstrap_difference",
            "mean",
        ),
        reconstructed_bootstrap_standard_error=(
            "bootstrap_difference",
            lambda x: x.std(ddof=1),
        ),
        reconstructed_ci_95_lower=(
            "bootstrap_difference",
            lambda x: x.quantile(
                0.025,
                interpolation="linear",
            ),
        ),
        reconstructed_ci_95_upper=(
            "bootstrap_difference",
            lambda x: x.quantile(
                0.975,
                interpolation="linear",
            ),
        ),
    )
)

bootstrap_summary["reconstructed_ci_excludes_zero"] = (
    (bootstrap_summary["reconstructed_ci_95_lower"] > 0.0)
    | (bootstrap_summary["reconstructed_ci_95_upper"] < 0.0)
)


# =========================================================
# 5. Merge stored and reconstructed values
# =========================================================

verification = (
    contrasts
    .merge(
        direct_differences,
        on=[
            "higher_tau",
            "lower_tau",
            "contrast",
            "metric",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        bootstrap_summary,
        on=[
            "higher_tau",
            "lower_tau",
            "contrast",
            "metric",
        ],
        how="left",
        validate="one_to_one",
    )
)

verification["estimate_residual"] = (
    verification["estimated_difference"]
    - verification["reconstructed_difference"]
).abs()

verification["standard_error_residual"] = (
    verification["bootstrap_standard_error"]
    - verification[
        "reconstructed_bootstrap_standard_error"
    ]
).abs()

verification["ci_lower_residual"] = (
    verification["ci_95_lower"]
    - verification["reconstructed_ci_95_lower"]
).abs()

verification["ci_upper_residual"] = (
    verification["ci_95_upper"]
    - verification["reconstructed_ci_95_upper"]
).abs()

verification["ci_flag_matches"] = (
    verification["ci_excludes_zero"]
    == verification["reconstructed_ci_excludes_zero"]
)

verification["replicate_count_matches"] = (
    verification["bootstrap_replicates"]
    == verification[
        "reconstructed_bootstrap_replicates"
    ]
)


# =========================================================
# 6. Design and denominator checks
# =========================================================

group_size_map = (
    primary[
        [
            "matched_group_id",
            "n_items",
        ]
    ]
    .drop_duplicates()
)

groups_per_size = (
    group_size_map
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        )
    )
)

duplicate_bootstrap_rows = int(
    bootstrap.duplicated(
        subset=[
            "bootstrap_replicate",
            "higher_tau",
            "lower_tau",
            "contrast",
        ]
    ).sum()
)

replicate_id_audit = (
    bootstrap
    .groupby(
        [
            "higher_tau",
            "lower_tau",
            "contrast",
        ],
        as_index=False,
    )
    .agg(
        rows=("bootstrap_replicate", "size"),
        unique_replicates=(
            "bootstrap_replicate",
            "nunique",
        ),
        minimum_replicate=(
            "bootstrap_replicate",
            "min",
        ),
        maximum_replicate=(
            "bootstrap_replicate",
            "max",
        ),
    )
)

design_audit = pd.DataFrame(
    [
        {
            "check": "contrast_rows_equal_27",
            "observed": len(contrasts),
            "expected": 27,
            "pass": len(contrasts) == 27,
        },
        {
            "check": "bootstrap_rows_equal_15000",
            "observed": len(bootstrap),
            "expected": 15000,
            "pass": len(bootstrap) == 15000,
        },
        {
            "check": "unique_matched_groups_equal_500",
            "observed": primary[
                "matched_group_id"
            ].nunique(),
            "expected": 500,
            "pass": (
                primary["matched_group_id"].nunique()
                == 500
            ),
        },
        {
            "check": "groups_per_size_equal_100",
            "observed": (
                groups_per_size[
                    "matched_groups"
                ].min()
            ),
            "expected": 100,
            "pass": (
                groups_per_size[
                    "matched_groups"
                ].eq(100).all()
            ),
        },
        {
            "check": "duplicate_bootstrap_rows",
            "observed": duplicate_bootstrap_rows,
            "expected": 0,
            "pass": duplicate_bootstrap_rows == 0,
        },
        {
            "check": "all_replicate_groups_have_5000_rows",
            "observed": (
                replicate_id_audit["rows"].min()
            ),
            "expected": 5000,
            "pass": (
                replicate_id_audit[
                    "rows"
                ].eq(5000).all()
            ),
        },
        {
            "check": "replicate_ids_span_0_to_4999",
            "observed": (
                f"{replicate_id_audit['minimum_replicate'].min()}"
                f"--"
                f"{replicate_id_audit['maximum_replicate'].max()}"
            ),
            "expected": "0--4999",
            "pass": (
                replicate_id_audit[
                    "minimum_replicate"
                ].eq(0).all()
                and replicate_id_audit[
                    "maximum_replicate"
                ].eq(4999).all()
            ),
        },
        {
            "check": "stored_resampling_design",
            "observed": contrasts[
                "resampling_design"
            ].nunique(),
            "expected": 1,
            "pass": (
                contrasts[
                    "resampling_design"
                ].eq(
                    "paired_across_thresholds_"
                    "stratified_by_candidate_set_size"
                ).all()
            ),
        },
        {
            "check": "multiple_comparison_adjustment_none",
            "observed": contrasts[
                "multiple_comparison_adjustment"
            ].unique()[0],
            "expected": "none",
            "pass": (
                contrasts[
                    "multiple_comparison_adjustment"
                ].eq("none").all()
            ),
        },
    ]
)


# =========================================================
# 7. Numerical verification summary
# =========================================================

TOLERANCE = 1e-12

numerical_summary = pd.DataFrame(
    [
        {
            "check": "point_estimates",
            "maximum_residual":
                verification[
                    "estimate_residual"
                ].max(),
            "failed_rows": int(
                (
                    verification[
                        "estimate_residual"
                    ] > TOLERANCE
                ).sum()
            ),
        },
        {
            "check": "bootstrap_standard_errors",
            "maximum_residual":
                verification[
                    "standard_error_residual"
                ].max(),
            "failed_rows": int(
                (
                    verification[
                        "standard_error_residual"
                    ] > TOLERANCE
                ).sum()
            ),
        },
        {
            "check": "percentile_ci_lower",
            "maximum_residual":
                verification[
                    "ci_lower_residual"
                ].max(),
            "failed_rows": int(
                (
                    verification[
                        "ci_lower_residual"
                    ] > TOLERANCE
                ).sum()
            ),
        },
        {
            "check": "percentile_ci_upper",
            "maximum_residual":
                verification[
                    "ci_upper_residual"
                ].max(),
            "failed_rows": int(
                (
                    verification[
                        "ci_upper_residual"
                    ] > TOLERANCE
                ).sum()
            ),
        },
        {
            "check": "ci_excludes_zero_flags",
            "maximum_residual": np.nan,
            "failed_rows": int(
                (~verification[
                    "ci_flag_matches"
                ]).sum()
            ),
        },
        {
            "check": "bootstrap_replicate_counts",
            "maximum_residual": np.nan,
            "failed_rows": int(
                (~verification[
                    "replicate_count_matches"
                ]).sum()
            ),
        },
    ]
)


# =========================================================
# 8. Manuscript rows currently reported
# =========================================================

selected_pairs = {
    ("0.80 minus 0.70", "mean_optimality_gap_percent"),
    (
        "0.80 minus 0.70",
        "mean_absolute_disclosure_cost_gap",
    ),
    (
        "0.80 minus 0.70",
        "mean_selected_count_difference",
    ),
    ("0.90 minus 0.80", "mean_optimality_gap_percent"),
    (
        "0.90 minus 0.80",
        "mean_absolute_disclosure_cost_gap",
    ),
    ("0.90 minus 0.80", "more_items_percent"),
    (
        "0.90 minus 0.80",
        "mean_selected_count_difference",
    ),
    ("0.95 minus 0.90", "mean_optimality_gap_percent"),
    (
        "0.95 minus 0.90",
        "mean_absolute_disclosure_cost_gap",
    ),
    ("0.95 minus 0.90", "more_items_percent"),
    (
        "0.95 minus 0.90",
        "mean_selected_count_difference",
    ),
}

manuscript_rows = verification[
    verification.apply(
        lambda row: (
            row["contrast"],
            row["metric"],
        ) in selected_pairs,
        axis=1,
    )
].copy()

manuscript_rows = manuscript_rows[
    [
        "contrast",
        "metric",
        "estimated_difference",
        "ci_95_lower",
        "ci_95_upper",
        "ci_excludes_zero",
        "matched_groups",
        "bootstrap_replicates",
        "estimate_residual",
        "standard_error_residual",
        "ci_lower_residual",
        "ci_upper_residual",
    ]
]


# =========================================================
# 9. Display outputs
# =========================================================

print("GROUPS PER CANDIDATE-SET SIZE")
display(groups_per_size)

print("\nBOOTSTRAP REPLICATE-ID AUDIT")
display(replicate_id_audit)

print("\nDESIGN AUDIT")
display(design_audit)

print("\nNUMERICAL VERIFICATION SUMMARY")
display(numerical_summary)

print("\nSELECTED MANUSCRIPT CONTRASTS")
display(manuscript_rows)

failed_verification_rows = verification[
    (verification["estimate_residual"] > TOLERANCE)
    | (
        verification["standard_error_residual"]
        > TOLERANCE
    )
    | (
        verification["ci_lower_residual"]
        > TOLERANCE
    )
    | (
        verification["ci_upper_residual"]
        > TOLERANCE
    )
    | (~verification["ci_flag_matches"])
    | (~verification["replicate_count_matches"])
]

print("\nFAILED NUMERICAL VERIFICATION ROWS")

if failed_verification_rows.empty:
    print(
        "PASS: All 27 contrast rows match direct and "
        "bootstrap reconstruction."
    )
else:
    display(failed_verification_rows)


# =========================================================
# 10. Save the verification evidence
# =========================================================

verification_output = (
    RESULTS
    / "results_threshold_contrasts_full_verification.csv"
)

design_output = (
    RESULTS
    / "results_threshold_contrasts_design_audit.csv"
)

numerical_output = (
    RESULTS
    / "results_threshold_contrasts_numerical_audit.csv"
)

manuscript_output = (
    TABLES
    / "results_threshold_contrasts_manuscript_verified.csv"
)

verification.to_csv(
    verification_output,
    index=False,
)

design_audit.to_csv(
    design_output,
    index=False,
)

numerical_summary.to_csv(
    numerical_output,
    index=False,
)

manuscript_rows.to_csv(
    manuscript_output,
    index=False,
)

print("\nSaved:")
print(verification_output)
print(design_output)
print(numerical_output)
print(manuscript_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

LONG_FILE = (
    RESULTS
    / "corrected_v2_algorithm_comparison_long.csv"
)

DESCRIPTIVE_FILE = (
    TABLES
    / "corrected_v2_dynamic_static_ratio_regime_threshold_descriptive.csv"
)

BOOTSTRAP_FILE = (
    RESULTS
    / "corrected_v2_dynamic_vs_static_ratio_regime_threshold_bootstrap_replicates.csv"
)

for path in (
    LONG_FILE,
    DESCRIPTIVE_FILE,
    BOOTSTRAP_FILE,
):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


long_df = pd.read_csv(LONG_FILE, low_memory=False)
descriptive = pd.read_csv(DESCRIPTIVE_FILE, low_memory=False)
bootstrap = pd.read_csv(BOOTSTRAP_FILE, low_memory=False)


# =========================================================
# 1. File shapes and complete schemas
# =========================================================

def show_schema(
    name: str,
    path: Path,
    data: pd.DataFrame,
) -> None:
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    print("\nFile:")
    print(path)

    print("\nShape:")
    print(data.shape)

    print("\nColumns:")
    for index, column in enumerate(data.columns, start=1):
        print(f"{index:02d}. {column}")


show_schema(
    "LONG-FORM ALGORITHM COMPARISON",
    LONG_FILE,
    long_df,
)

show_schema(
    "DYNAMIC-STATIC DESCRIPTIVE TABLE",
    DESCRIPTIVE_FILE,
    descriptive,
)

show_schema(
    "DYNAMIC-STATIC BOOTSTRAP REPLICATES",
    BOOTSTRAP_FILE,
    bootstrap,
)


# =========================================================
# 2. Important unique values in the long-form data
# =========================================================

important_columns = [
    column
    for column in (
        "benchmark_version",
        "generator_version",
        "algorithm",
        "algorithm_name",
        "regime",
        "tau",
        "weight_profile",
        "n_items",
        "matched_group_id",
        "exact_status",
        "algorithm_status",
        "status",
    )
    if column in long_df.columns
]

print("\nIMPORTANT LONG-FORM VALUES")

for column in important_columns:
    values = long_df[column].dropna()

    print(f"\n{column}:")
    print(f"  non-null rows: {len(values)}")
    print(f"  unique values: {values.nunique()}")

    if values.nunique() <= 30:
        print(
            "  values:",
            sorted(values.astype(str).unique()),
        )


# =========================================================
# 3. Detect the algorithm column
# =========================================================

algorithm_candidates = [
    column
    for column in long_df.columns
    if "algorithm" in column.lower()
]

print("\nALGORITHM-LIKE COLUMNS")
print(algorithm_candidates)

algorithm_column = None

for candidate in (
    "algorithm",
    "algorithm_name",
    "algorithm_variant",
):
    if candidate in long_df.columns:
        algorithm_column = candidate
        break

if algorithm_column is None:
    raise KeyError(
        "Could not identify the algorithm column. "
        f"Candidates found: {algorithm_candidates}"
    )

print("\nDetected algorithm column:")
print(algorithm_column)

print("\nRows per algorithm:")
display(
    long_df
    .groupby(algorithm_column, dropna=False)
    .size()
    .rename("rows")
    .reset_index()
)


# =========================================================
# 4. Select dynamic and static procedures
# =========================================================

dynamic_label = "dynamic_greedy"
static_label = "static_utility_cost_ratio"

available_algorithms = set(
    long_df[algorithm_column]
    .dropna()
    .astype(str)
    .unique()
)

for required_label in (
    dynamic_label,
    static_label,
):
    if required_label not in available_algorithms:
        raise ValueError(
            f"Required algorithm not found: {required_label}. "
            f"Available algorithms: {sorted(available_algorithms)}"
        )

comparison = long_df[
    long_df[algorithm_column].isin(
        [
            dynamic_label,
            static_label,
        ]
    )
].copy()

print("\nDYNAMIC-STATIC COMPARISON SHAPE")
print(comparison.shape)


# =========================================================
# 5. Find eligibility and result columns
# =========================================================

keywords = (
    "status",
    "gap",
    "selected",
    "count",
    "jaccard",
    "objective",
    "disclosure",
    "runtime",
    "recovery",
    "more_items",
    "feasible",
    "matched",
)

relevant_columns = [
    column
    for column in comparison.columns
    if any(
        keyword in column.lower()
        for keyword in keywords
    )
]

print("\nRELEVANT COMPARISON COLUMNS")

for index, column in enumerate(
    relevant_columns,
    start=1,
):
    print(f"{index:02d}. {column}")


# =========================================================
# 6. Counts by regime, threshold, and algorithm
# =========================================================

required_group_columns = [
    column
    for column in (
        "regime",
        "tau",
        algorithm_column,
    )
    if column in comparison.columns
]

group_counts = (
    comparison
    .groupby(
        required_group_columns,
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)

print("\nROWS BY REGIME, THRESHOLD, AND ALGORITHM")
display(group_counts)


# =========================================================
# 7. Matched-group counts and missing-pair audit
# =========================================================

if "matched_group_id" not in comparison.columns:
    raise KeyError(
        "matched_group_id is required for the paired audit."
    )

pair_index_columns = [
    column
    for column in (
        "matched_group_id",
        "regime",
        "tau",
        "weight_profile",
        "n_items",
    )
    if column in comparison.columns
]

pair_counts = (
    comparison
    .groupby(
        pair_index_columns,
        dropna=False,
    )[algorithm_column]
    .nunique()
    .rename("algorithm_count")
    .reset_index()
)

pair_audit = pd.DataFrame(
    [
        {
            "measure": "dynamic_static_rows",
            "value": len(comparison),
        },
        {
            "measure": "unique_pairing_conditions",
            "value": len(pair_counts),
        },
        {
            "measure": "conditions_with_both_algorithms",
            "value": int(
                (pair_counts["algorithm_count"] == 2).sum()
            ),
        },
        {
            "measure": "conditions_missing_one_algorithm",
            "value": int(
                (pair_counts["algorithm_count"] != 2).sum()
            ),
        },
        {
            "measure": "unique_matched_groups",
            "value": comparison[
                "matched_group_id"
            ].nunique(),
        },
    ]
)

print("\nPAIRING AUDIT")
display(pair_audit)


# =========================================================
# 8. Complete stored descriptive table
# =========================================================

print("\nCOMPLETE STORED DESCRIPTIVE TABLE")
display(descriptive)


# =========================================================
# 9. Bootstrap replicate structure
# =========================================================

print("\nFIRST 20 BOOTSTRAP ROWS")
display(bootstrap.head(20))

identifier_keywords = (
    "regime",
    "tau",
    "threshold",
    "contrast",
    "metric",
    "algorithm",
)

replicate_keywords = (
    "replicate",
    "bootstrap_id",
    "iteration",
)

bootstrap_identifiers = [
    column
    for column in bootstrap.columns
    if any(
        keyword in column.lower()
        for keyword in identifier_keywords
    )
]

bootstrap_replicate_columns = [
    column
    for column in bootstrap.columns
    if any(
        keyword in column.lower()
        for keyword in replicate_keywords
    )
]

print("\nDetected bootstrap identifier columns:")
print(bootstrap_identifiers)

print("\nDetected bootstrap replicate columns:")
print(bootstrap_replicate_columns)

grouping_columns = [
    column
    for column in bootstrap_identifiers
    if column not in bootstrap_replicate_columns
]

if grouping_columns:
    bootstrap_group_counts = (
        bootstrap
        .groupby(
            grouping_columns,
            dropna=False,
        )
        .size()
        .rename("bootstrap_rows")
        .reset_index()
    )

    print("\nBOOTSTRAP ROWS PER GROUP")
    display(bootstrap_group_counts)

else:
    bootstrap_group_counts = pd.DataFrame()

    print(
        "\nNo bootstrap grouping columns were "
        "detected automatically."
    )


# =========================================================
# 10. Missing-value and duplicate audit
# =========================================================

file_integrity = pd.DataFrame(
    [
        {
            "file": "long_form",
            "rows": len(long_df),
            "duplicate_complete_rows":
                int(long_df.duplicated().sum()),
            "columns_with_missing_values":
                int(long_df.isna().any().sum()),
        },
        {
            "file": "descriptive",
            "rows": len(descriptive),
            "duplicate_complete_rows":
                int(descriptive.duplicated().sum()),
            "columns_with_missing_values":
                int(descriptive.isna().any().sum()),
        },
        {
            "file": "bootstrap",
            "rows": len(bootstrap),
            "duplicate_complete_rows":
                int(bootstrap.duplicated().sum()),
            "columns_with_missing_values":
                int(bootstrap.isna().any().sum()),
        },
    ]
)

print("\nFILE-INTEGRITY AUDIT")
display(file_integrity)


# =========================================================
# 11. Save audit outputs
# =========================================================

group_output = (
    RESULTS
    / "results_dynamic_static_group_counts.csv"
)

pair_output = (
    RESULTS
    / "results_dynamic_static_pairing_audit.csv"
)

bootstrap_output = (
    RESULTS
    / "results_dynamic_static_bootstrap_group_counts.csv"
)

integrity_output = (
    RESULTS
    / "results_dynamic_static_file_integrity_audit.csv"
)

group_counts.to_csv(group_output, index=False)
pair_audit.to_csv(pair_output, index=False)
bootstrap_group_counts.to_csv(
    bootstrap_output,
    index=False,
)
file_integrity.to_csv(
    integrity_output,
    index=False,
)

print("\nSaved:")
print(group_output)
print(pair_output)
print(bootstrap_output)
print(integrity_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

LONG_FILE = (
    RESULTS
    / "corrected_v2_algorithm_comparison_long.csv"
)

DESCRIPTIVE_FILE = (
    TABLES
    / "corrected_v2_dynamic_static_ratio_regime_threshold_descriptive.csv"
)

BOOTSTRAP_FILE = (
    RESULTS
    / "corrected_v2_dynamic_vs_static_ratio_regime_threshold_bootstrap_replicates.csv"
)

long_df = pd.read_csv(LONG_FILE, low_memory=False)
stored_descriptive = pd.read_csv(
    DESCRIPTIVE_FILE,
    low_memory=False,
)
bootstrap = pd.read_csv(
    BOOTSTRAP_FILE,
    low_memory=False,
)


# =========================================================
# 1. Retain only the dynamic and static procedures
# =========================================================

ALGORITHMS = [
    "dynamic_greedy",
    "static_utility_cost_ratio",
]

comparison = long_df[
    long_df["algorithm"].isin(ALGORITHMS)
].copy()


def normalize_boolean(series: pd.Series) -> pd.Series:
    """
    Convert bool, numeric, or text Boolean fields into bool.
    Missing values become False.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().any():
        return numeric.fillna(0).ne(0)

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


comparison["is_comparable_bool"] = normalize_boolean(
    comparison["is_comparable"]
)


# =========================================================
# 2. Missing-value audit
# =========================================================

missing_rows = []

for column in comparison.columns:
    missing_mask = comparison[column].isna()

    if missing_mask.any():
        missing_rows.append(
            {
                "column": column,
                "total_missing_rows": int(
                    missing_mask.sum()
                ),
                "missing_in_comparable_rows": int(
                    (
                        missing_mask
                        & comparison["is_comparable_bool"]
                    ).sum()
                ),
                "missing_in_noncomparable_rows": int(
                    (
                        missing_mask
                        & ~comparison["is_comparable_bool"]
                    ).sum()
                ),
            }
        )

missingness_audit = pd.DataFrame(missing_rows)

print("MISSING-VALUE LOCATION AUDIT")
display(missingness_audit)


# =========================================================
# 3. Comparable and infeasible status audit
# =========================================================

status_audit = (
    comparison
    .groupby(
        [
            "is_comparable_bool",
            "exact_status",
            "algorithm_status",
        ],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)

print("\nSTATUS AND COMPARABILITY AUDIT")
display(status_audit)


comparable = comparison[
    comparison["is_comparable_bool"]
].copy()

noncomparable = comparison[
    ~comparison["is_comparable_bool"]
].copy()


# =========================================================
# 4. Reconstruct stored indicator columns
# =========================================================

comparable["reconstructed_exact_recovery"] = (
    comparable["optimality_gap_percent"] == 0.0
).astype(int)

comparable["reconstructed_large_gap"] = (
    comparable["optimality_gap_percent"] > 10.0
).astype(int)

comparable["reconstructed_more_items"] = (
    comparable["selected_count_difference"] > 0
).astype(int)

comparable["reconstructed_within_1"] = (
    comparable["optimality_gap_percent"] <= 1.0
).astype(int)

comparable["reconstructed_within_5"] = (
    comparable["optimality_gap_percent"] <= 5.0
).astype(int)

comparable["reconstructed_within_10"] = (
    comparable["optimality_gap_percent"] <= 10.0
).astype(int)


indicator_checks = [
    (
        "algorithm_optimal_cost_numeric",
        "reconstructed_exact_recovery",
    ),
    (
        "large_gap_numeric",
        "reconstructed_large_gap",
    ),
    (
        "algorithm_more_items_numeric",
        "reconstructed_more_items",
    ),
    (
        "within_1_percent_numeric",
        "reconstructed_within_1",
    ),
    (
        "within_5_percent_numeric",
        "reconstructed_within_5",
    ),
    (
        "within_10_percent_numeric",
        "reconstructed_within_10",
    ),
]

indicator_audit_rows = []

for stored_column, reconstructed_column in indicator_checks:
    stored_values = pd.to_numeric(
        comparable[stored_column],
        errors="coerce",
    )

    reconstructed_values = comparable[
        reconstructed_column
    ].astype(float)

    residual = (
        stored_values
        - reconstructed_values
    ).abs()

    indicator_audit_rows.append(
        {
            "stored_column": stored_column,
            "reconstructed_definition":
                reconstructed_column,
            "rows_checked": len(comparable),
            "missing_stored_rows": int(
                stored_values.isna().sum()
            ),
            "failed_rows": int(
                (residual.fillna(np.inf) > 1e-12).sum()
            ),
            "maximum_residual": (
                float(residual.max())
                if residual.notna().any()
                else np.nan
            ),
        }
    )

indicator_audit = pd.DataFrame(
    indicator_audit_rows
)

print("\nINDICATOR-COLUMN CONSISTENCY AUDIT")
display(indicator_audit)


# =========================================================
# 5. Reconstruct the complete descriptive table
# =========================================================

reconstructed_descriptive = (
    comparable
    .groupby(
        [
            "regime",
            "tau",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),
        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),
        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),
        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),
        p95_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.95),
        ),
        exact_cost_recovery_percent=(
            "reconstructed_exact_recovery",
            lambda values:
                100.0 * values.mean(),
        ),
        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),
        large_gap_percent=(
            "reconstructed_large_gap",
            lambda values:
                100.0 * values.mean(),
        ),
        more_items_percent=(
            "reconstructed_more_items",
            lambda values:
                100.0 * values.mean(),
        ),
        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),
        within_1_percent=(
            "reconstructed_within_1",
            lambda values:
                100.0 * values.mean(),
        ),
        within_5_percent=(
            "reconstructed_within_5",
            lambda values:
                100.0 * values.mean(),
        ),
        within_10_percent=(
            "reconstructed_within_10",
            lambda values:
                100.0 * values.mean(),
        ),
    )
)


descriptive_verification = stored_descriptive.merge(
    reconstructed_descriptive,
    on=[
        "regime",
        "tau",
        "algorithm",
    ],
    suffixes=("_stored", "_reconstructed"),
    validate="one_to_one",
)

descriptive_metrics = [
    "matched_groups",
    "comparable_rows",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "p95_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]

descriptive_summary_rows = []

for metric in descriptive_metrics:
    residual = (
        descriptive_verification[
            f"{metric}_stored"
        ]
        -
        descriptive_verification[
            f"{metric}_reconstructed"
        ]
    ).abs()

    descriptive_summary_rows.append(
        {
            "metric": metric,
            "maximum_residual": float(
                residual.max()
            ),
            "failed_rows": int(
                (residual > 1e-12).sum()
            ),
        }
    )

descriptive_verification_summary = pd.DataFrame(
    descriptive_summary_rows
)

print("\nDESCRIPTIVE-TABLE NUMERICAL VERIFICATION")
display(descriptive_verification_summary)


# =========================================================
# 6. Pairing and feasible-denominator audit
# =========================================================

PAIR_KEYS = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "replicate",
    "regime",
    "matched_seed",
    "tau",
    "weight_profile",
    "lambda_privacy",
    "lambda_cognitive",
    "lambda_cardinality",
]

pair_counts = (
    comparable
    .groupby(
        PAIR_KEYS,
        dropna=False,
    )["algorithm"]
    .nunique()
    .rename("algorithm_count")
    .reset_index()
)

denominator_by_condition = (
    comparable
    .groupby(
        [
            "regime",
            "tau",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        configured_rows=(
            "algorithm",
            "size",
        ),
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
    )
)

pairing_audit = pd.DataFrame(
    [
        {
            "check":
                "all_comparable_conditions_have_both_algorithms",
            "observed": int(
                (pair_counts["algorithm_count"] == 2).sum()
            ),
            "expected": len(pair_counts),
            "pass": bool(
                pair_counts[
                    "algorithm_count"
                ].eq(2).all()
            ),
        },
        {
            "check":
                "comparable_exact_status_is_OPTIMAL",
            "observed": int(
                (
                    comparable["exact_status"]
                    == "OPTIMAL"
                ).sum()
            ),
            "expected": len(comparable),
            "pass": bool(
                comparable[
                    "exact_status"
                ].eq("OPTIMAL").all()
            ),
        },
        {
            "check":
                "comparable_algorithm_status_is_FEASIBLE",
            "observed": int(
                (
                    comparable["algorithm_status"]
                    == "FEASIBLE"
                ).sum()
            ),
            "expected": len(comparable),
            "pass": bool(
                comparable[
                    "algorithm_status"
                ].eq("FEASIBLE").all()
            ),
        },
        {
            "check":
                "noncomparable_rows",
            "observed": len(noncomparable),
            "expected": 432,
            "pass": len(noncomparable) == 432,
        },
    ]
)

print("\nDENOMINATORS BY REGIME, THRESHOLD, AND ALGORITHM")
display(denominator_by_condition)

print("\nPAIRING AND STATUS AUDIT")
display(pairing_audit)


# =========================================================
# 7. Dynamic-minus-static point estimates
# =========================================================

comparison_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]

point_difference_rows = []

for (
    regime,
    tau,
), group in reconstructed_descriptive.groupby(
    [
        "regime",
        "tau",
    ]
):
    indexed = group.set_index("algorithm")

    for metric in comparison_metrics:
        dynamic_value = indexed.loc[
            "dynamic_greedy",
            metric,
        ]

        static_value = indexed.loc[
            "static_utility_cost_ratio",
            metric,
        ]

        point_difference_rows.append(
            {
                "regime": regime,
                "tau": tau,
                "metric": metric,
                "dynamic_value": dynamic_value,
                "static_value": static_value,
                "dynamic_minus_static":
                    dynamic_value - static_value,
            }
        )

point_differences = pd.DataFrame(
    point_difference_rows
)


# =========================================================
# 8. Reconstruct bootstrap summaries
# =========================================================

bootstrap_long = bootstrap.melt(
    id_vars=[
        "regime",
        "tau",
        "bootstrap_replicate",
    ],
    value_vars=comparison_metrics,
    var_name="metric",
    value_name="bootstrap_difference",
)

bootstrap_summary = (
    bootstrap_long
    .groupby(
        [
            "regime",
            "tau",
            "metric",
        ],
        as_index=False,
    )
    .agg(
        bootstrap_replicates=(
            "bootstrap_difference",
            "size",
        ),
        bootstrap_mean=(
            "bootstrap_difference",
            "mean",
        ),
        bootstrap_standard_error=(
            "bootstrap_difference",
            lambda values:
                values.std(ddof=1),
        ),
        ci_95_lower=(
            "bootstrap_difference",
            lambda values:
                values.quantile(
                    0.025,
                    interpolation="linear",
                ),
        ),
        ci_95_upper=(
            "bootstrap_difference",
            lambda values:
                values.quantile(
                    0.975,
                    interpolation="linear",
                ),
        ),
    )
)

bootstrap_summary["ci_excludes_zero"] = (
    (bootstrap_summary["ci_95_lower"] > 0)
    |
    (bootstrap_summary["ci_95_upper"] < 0)
)

dynamic_static_contrasts = point_differences.merge(
    bootstrap_summary,
    on=[
        "regime",
        "tau",
        "metric",
    ],
    validate="one_to_one",
)


# =========================================================
# 9. Endpoint values used in the manuscript
# =========================================================

endpoint_table = reconstructed_descriptive[
    reconstructed_descriptive["tau"].isin(
        [0.70, 0.95]
    )
][
    [
        "regime",
        "tau",
        "algorithm",
        "matched_groups",
        "comparable_rows",
        "mean_optimality_gap_percent",
        "mean_jaccard_similarity",
        "more_items_percent",
        "mean_selected_count_difference",
    ]
].copy()

endpoint_table = endpoint_table.sort_values(
    [
        "regime",
        "tau",
        "algorithm",
    ]
)

print("\nVERIFIED LOW- AND HIGH-THRESHOLD ENDPOINTS")
display(endpoint_table)


# =========================================================
# 10. Conflict-regime tau=0.95 contrasts
# =========================================================

conflict_095 = dynamic_static_contrasts[
    (
        dynamic_static_contrasts["regime"]
        == "utility_burden_conflict"
    )
    &
    np.isclose(
        dynamic_static_contrasts["tau"],
        0.95,
    )
    &
    dynamic_static_contrasts["metric"].isin(
        [
            "mean_optimality_gap_percent",
            "mean_jaccard_similarity",
            "more_items_percent",
            "mean_selected_count_difference",
        ]
    )
].copy()

conflict_095 = conflict_095[
    [
        "regime",
        "tau",
        "metric",
        "dynamic_value",
        "static_value",
        "dynamic_minus_static",
        "bootstrap_replicates",
        "bootstrap_standard_error",
        "ci_95_lower",
        "ci_95_upper",
        "ci_excludes_zero",
    ]
]

print("\nCONFLICT REGIME AT TAU=0.95")
display(conflict_095)


# =========================================================
# 11. Complete numerical pass/fail summary
# =========================================================

numerical_pass = pd.DataFrame(
    [
        {
            "check":
                "all_indicator_columns_reconstructed",
            "failed_rows": int(
                indicator_audit[
                    "failed_rows"
                ].sum()
            ),
            "pass": bool(
                indicator_audit[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "stored_descriptive_table_reconstructed",
            "failed_rows": int(
                descriptive_verification_summary[
                    "failed_rows"
                ].sum()
            ),
            "pass": bool(
                descriptive_verification_summary[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "all_bootstrap_groups_have_5000_rows",
            "failed_rows": int(
                (
                    bootstrap_summary[
                        "bootstrap_replicates"
                    ] != 5000
                ).sum()
            ),
            "pass": bool(
                bootstrap_summary[
                    "bootstrap_replicates"
                ].eq(5000).all()
            ),
        },
        {
            "check":
                "all_pairing_and_status_checks",
            "failed_rows": int(
                (~pairing_audit["pass"]).sum()
            ),
            "pass": bool(
                pairing_audit["pass"].all()
            ),
        },
    ]
)

print("\nFINAL NUMERICAL PASS/FAIL SUMMARY")
display(numerical_pass)


# =========================================================
# 12. Save verified outputs
# =========================================================

missingness_output = (
    RESULTS
    / "results_dynamic_static_missingness_audit.csv"
)

indicator_output = (
    RESULTS
    / "results_dynamic_static_indicator_audit.csv"
)

descriptive_output = (
    RESULTS
    / "results_dynamic_static_descriptive_verification.csv"
)

denominator_output = (
    RESULTS
    / "results_dynamic_static_denominator_verified.csv"
)

contrast_output = (
    TABLES
    / "results_dynamic_static_contrasts_verified.csv"
)

endpoint_output = (
    TABLES
    / "results_dynamic_static_endpoints_verified.csv"
)

conflict_output = (
    TABLES
    / "results_dynamic_static_conflict_tau095_verified.csv"
)

pass_output = (
    RESULTS
    / "results_dynamic_static_final_audit.csv"
)

missingness_audit.to_csv(
    missingness_output,
    index=False,
)

indicator_audit.to_csv(
    indicator_output,
    index=False,
)

descriptive_verification.to_csv(
    descriptive_output,
    index=False,
)

denominator_by_condition.to_csv(
    denominator_output,
    index=False,
)

dynamic_static_contrasts.to_csv(
    contrast_output,
    index=False,
)

endpoint_table.to_csv(
    endpoint_output,
    index=False,
)

conflict_095.to_csv(
    conflict_output,
    index=False,
)

numerical_pass.to_csv(
    pass_output,
    index=False,
)

print("\nSaved:")
print(missingness_output)
print(indicator_output)
print(descriptive_output)
print(denominator_output)
print(contrast_output)
print(endpoint_output)
print(conflict_output)
print(pass_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import ast
import importlib.util
import inspect
import sys

import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

PYTHON_DIR = ROOT / "03_Python_Optimization"
RESULTS = ROOT / "05_Results"

CORE_MODULE_FILE = PYTHON_DIR / "mst_core.py"

SEQUENCE_FILE = (
    RESULTS
    / "progressive_mst_validation_sequence.csv"
)

PROPERTY_FILE = (
    RESULTS
    / "progressive_mst_property_validation.csv"
)

NORMALIZED_SEQUENCE_FILE = (
    RESULTS
    / "normalized_progressive_mst_sequence.csv"
)

NORMALIZED_PROPERTY_FILE = (
    RESULTS
    / "normalized_mst_property_validation.csv"
)

for path in (
    CORE_MODULE_FILE,
    SEQUENCE_FILE,
    PROPERTY_FILE,
    NORMALIZED_SEQUENCE_FILE,
    NORMALIZED_PROPERTY_FILE,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


# =========================================================
# 1. Load stored validation files
# =========================================================

sequence = pd.read_csv(
    SEQUENCE_FILE,
    low_memory=False,
)

properties = pd.read_csv(
    PROPERTY_FILE,
    low_memory=False,
)

normalized_sequence = pd.read_csv(
    NORMALIZED_SEQUENCE_FILE,
    low_memory=False,
)

normalized_properties = pd.read_csv(
    NORMALIZED_PROPERTY_FILE,
    low_memory=False,
)


# =========================================================
# 2. Correctly load mst_core.py
# =========================================================

MODULE_NAME = "mst_core_progressive_audit"

module_spec = importlib.util.spec_from_file_location(
    MODULE_NAME,
    CORE_MODULE_FILE,
)

if (
    module_spec is None
    or module_spec.loader is None
):
    raise ImportError(
        f"Could not create module specification for "
        f"{CORE_MODULE_FILE}"
    )

mst_core = importlib.util.module_from_spec(
    module_spec
)

# Required for dataclass processing during dynamic import.
sys.modules[MODULE_NAME] = mst_core

try:
    module_spec.loader.exec_module(mst_core)
except Exception:
    sys.modules.pop(MODULE_NAME, None)
    raise


print("MODULE IMPORT")
print("PASS:", CORE_MODULE_FILE)


# =========================================================
# 3. Inspect authoritative progressive functions
# =========================================================

required_functions = [
    "progressive_mst_step",
    "progressive_mst_sequence",
]

function_rows = []

for function_name in required_functions:

    if not hasattr(mst_core, function_name):
        function_rows.append(
            {
                "function": function_name,
                "exists": False,
                "signature": "",
            }
        )
        continue

    function_object = getattr(
        mst_core,
        function_name,
    )

    function_rows.append(
        {
            "function": function_name,
            "exists": True,
            "signature": str(
                inspect.signature(function_object)
            ),
        }
    )

function_audit = pd.DataFrame(
    function_rows
)

print("\nPROGRESSIVE FUNCTION SIGNATURES")
display(function_audit)


for function_name in required_functions:

    if not hasattr(mst_core, function_name):
        continue

    print("\n" + "=" * 90)
    print(function_name)
    print("=" * 90)

    print(
        inspect.getsource(
            getattr(mst_core, function_name)
        )
    )


# =========================================================
# 4. Parse stored subset identifiers
# =========================================================

def parse_item_tuple(value: object) -> tuple[str, ...]:
    """
    Parse a stored tuple such as:
    ('z1', 'z2')
    """
    parsed = ast.literal_eval(str(value))

    if isinstance(parsed, str):
        return (parsed,)

    return tuple(parsed)


def normalize_boolean(value: object) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


sequence_sets = [
    set(parse_item_tuple(value))
    for value in sequence["items"]
]

normalized_sequence_sets = [
    set(parse_item_tuple(value))
    for value in normalized_sequence["items"]
]


# =========================================================
# 5. Directly reconstruct stored sequence properties
# =========================================================

def sequence_property_audit(
    data: pd.DataFrame,
    cost_column: str,
    label: str,
) -> pd.DataFrame:

    parsed_sets = [
        set(parse_item_tuple(value))
        for value in data["items"]
    ]

    nested_flags = [
        parsed_sets[index - 1].issubset(
            parsed_sets[index]
        )
        for index in range(1, len(parsed_sets))
    ]

    utility_values = pd.to_numeric(
        data["utility"],
        errors="raise",
    ).to_numpy()

    cost_values = pd.to_numeric(
        data[cost_column],
        errors="raise",
    ).to_numpy()

    step_values = pd.to_numeric(
        data["step"],
        errors="raise",
    ).to_numpy()

    expected_steps = np.arange(len(data))

    one_item_addition_flags = []
    added_item_flags = []

    for index in range(1, len(parsed_sets)):

        previous_set = parsed_sets[index - 1]
        current_set = parsed_sets[index]

        new_items = current_set - previous_set

        one_item_addition_flags.append(
            len(new_items) == 1
        )

        stored_added_item = str(
            data.iloc[index]["added_item"]
        )

        added_item_flags.append(
            len(new_items) == 1
            and stored_added_item
            == next(iter(new_items))
        )

    return pd.DataFrame(
        [
            {
                "sequence": label,
                "property":
                    "steps_are_consecutive_from_zero",
                "passed": bool(
                    np.array_equal(
                        step_values,
                        expected_steps,
                    )
                ),
            },
            {
                "sequence": label,
                "property":
                    "nested_disclosure",
                "passed": bool(
                    all(nested_flags)
                ),
            },
            {
                "sequence": label,
                "property":
                    "exactly_one_item_added_per_transition",
                "passed": bool(
                    all(one_item_addition_flags)
                ),
            },
            {
                "sequence": label,
                "property":
                    "added_item_matches_set_difference",
                "passed": bool(
                    all(added_item_flags)
                ),
            },
            {
                "sequence": label,
                "property":
                    "monotonic_utility",
                "passed": bool(
                    np.all(
                        np.diff(utility_values)
                        >= -1e-12
                    )
                ),
            },
            {
                "sequence": label,
                "property":
                    f"monotonic_{cost_column}",
                "passed": bool(
                    np.all(
                        np.diff(cost_values)
                        >= -1e-12
                    )
                ),
            },
        ]
    )


legacy_direct_audit = sequence_property_audit(
    sequence,
    cost_column="disclosure_cost",
    label="legacy_progressive_sequence",
)

normalized_direct_audit = sequence_property_audit(
    normalized_sequence,
    cost_column="disclosure_cost",
    label="normalized_progressive_sequence",
)

direct_property_audit = pd.concat(
    [
        legacy_direct_audit,
        normalized_direct_audit,
    ],
    ignore_index=True,
)


print("\nDIRECT SEQUENCE-PROPERTY RECONSTRUCTION")
display(direct_property_audit)


# =========================================================
# 6. Verify normalized interpretation-cost values
# =========================================================

final_total_cognitive_cost = float(
    normalized_sequence[
        "cognitive_cost"
    ].iloc[-1]
)

if final_total_cognitive_cost <= 0:
    raise ValueError(
        "Final stored cognitive cost must be positive."
    )

normalized_sequence[
    "reconstructed_normalized_cognitive_cost"
] = (
    normalized_sequence["cognitive_cost"]
    / final_total_cognitive_cost
)

normalized_sequence[
    "normalized_cognitive_cost_residual"
] = (
    normalized_sequence[
        "normalized_cognitive_cost"
    ]
    -
    normalized_sequence[
        "reconstructed_normalized_cognitive_cost"
    ]
).abs()


normalization_audit = pd.DataFrame(
    [
        {
            "check":
                "normalized_cognitive_cost_reconstruction",
            "rows_checked":
                len(normalized_sequence),
            "maximum_residual":
                normalized_sequence[
                    "normalized_cognitive_cost_residual"
                ].max(),
            "failed_rows":
                int(
                    (
                        normalized_sequence[
                            "normalized_cognitive_cost_residual"
                        ]
                        > 1e-12
                    ).sum()
                ),
        }
    ]
)

print("\nNORMALIZED INTERPRETATION-COST AUDIT")
display(
    normalized_sequence[
        [
            "step",
            "items",
            "cognitive_cost",
            "normalized_cognitive_cost",
            "reconstructed_normalized_cognitive_cost",
            "normalized_cognitive_cost_residual",
        ]
    ]
)

display(normalization_audit)


# =========================================================
# 7. Compare direct checks with stored property files
# =========================================================

stored_property_rows = []

for file_label, data in (
    (
        "progressive_mst_property_validation",
        properties,
    ),
    (
        "normalized_mst_property_validation",
        normalized_properties,
    ),
):

    for _, row in data.iterrows():
        stored_property_rows.append(
            {
                "file": file_label,
                "property": row["property"],
                "stored_passed":
                    normalize_boolean(row["passed"]),
            }
        )

stored_property_audit = pd.DataFrame(
    stored_property_rows
)

print("\nSTORED PROPERTY FLAGS")
display(stored_property_audit)


# =========================================================
# 8. Implementation-source checks
# =========================================================

step_source = inspect.getsource(
    mst_core.progressive_mst_step
)

sequence_source = inspect.getsource(
    mst_core.progressive_mst_sequence
)

implementation_checks = pd.DataFrame(
    [
        {
            "check":
                "progressive_step_function_present",
            "passed":
                hasattr(
                    mst_core,
                    "progressive_mst_step",
                ),
        },
        {
            "check":
                "progressive_sequence_function_present",
            "passed":
                hasattr(
                    mst_core,
                    "progressive_mst_sequence",
                ),
        },
        {
            "check":
                "current_set_referenced_in_step",
            "passed":
                "current_set" in step_source,
        },
        {
            "check":
                "threshold_referenced_in_step",
            "passed":
                "threshold" in step_source.lower(),
        },
        {
            "check":
                "thresholds_referenced_in_sequence",
            "passed":
                "threshold" in sequence_source.lower(),
        },
        {
            "check":
                "epsilon_1e_minus_12_in_progressive_source",
            "passed":
                "1e-12" in (
                    step_source
                    + sequence_source
                ),
        },
    ]
)

print("\nIMPLEMENTATION-SOURCE CHECKS")
display(implementation_checks)


# =========================================================
# 9. Final summary
# =========================================================

final_summary = pd.DataFrame(
    [
        {
            "check":
                "authoritative_module_imported",
            "failed_rows": 0,
            "passed": True,
        },
        {
            "check":
                "required_progressive_functions_exist",
            "failed_rows": int(
                (~function_audit["exists"]).sum()
            ),
            "passed": bool(
                function_audit["exists"].all()
            ),
        },
        {
            "check":
                "direct_sequence_properties",
            "failed_rows": int(
                (
                    ~direct_property_audit["passed"]
                ).sum()
            ),
            "passed": bool(
                direct_property_audit[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "normalized_cognitive_cost",
            "failed_rows": int(
                normalization_audit[
                    "failed_rows"
                ].sum()
            ),
            "passed": bool(
                normalization_audit[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "stored_property_flags",
            "failed_rows": int(
                (
                    ~stored_property_audit[
                        "stored_passed"
                    ]
                ).sum()
            ),
            "passed": bool(
                stored_property_audit[
                    "stored_passed"
                ].all()
            ),
        },
        {
            "check":
                "implementation_source_checks",
            "failed_rows": int(
                (
                    ~implementation_checks[
                        "passed"
                    ]
                ).sum()
            ),
            "passed": bool(
                implementation_checks[
                    "passed"
                ].all()
            ),
        },
    ]
)

print("\nFINAL PROGRESSIVE AUDIT SUMMARY")
display(final_summary)


# =========================================================
# 10. Save evidence
# =========================================================

function_output = (
    RESULTS
    / "results_progressive_function_signatures.csv"
)

direct_output = (
    RESULTS
    / "results_progressive_direct_property_audit.csv"
)

normalization_output = (
    RESULTS
    / "results_progressive_normalization_audit.csv"
)

stored_output = (
    RESULTS
    / "results_progressive_stored_property_flags.csv"
)

implementation_output = (
    RESULTS
    / "results_progressive_implementation_source_audit.csv"
)

summary_output = (
    RESULTS
    / "results_progressive_final_audit.csv"
)

function_audit.to_csv(
    function_output,
    index=False,
)

direct_property_audit.to_csv(
    direct_output,
    index=False,
)

normalization_audit.to_csv(
    normalization_output,
    index=False,
)

stored_property_audit.to_csv(
    stored_output,
    index=False,
)

implementation_checks.to_csv(
    implementation_output,
    index=False,
)

final_summary.to_csv(
    summary_output,
    index=False,
)

print("\nSaved:")
print(function_output)
print(direct_output)
print(normalization_output)
print(stored_output)
print(implementation_output)
print(summary_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import importlib.util
import inspect
import sys

import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

PYTHON_DIR = ROOT / "03_Python_Optimization"
RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

PRIMARY_FILE = (
    RESULTS
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

CORE_MODULE_FILE = (
    PYTHON_DIR
    / "mst_core.py"
)

for path in (
    PRIMARY_FILE,
    CORE_MODULE_FILE,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


# =========================================================
# 1. Discover possible scalability and runtime files
# =========================================================

filename_keywords = (
    "scalab",
    "runtime",
    "timing",
    "large",
    "size",
    "performance",
)

all_csv_files = sorted(
    set(RESULTS.rglob("*.csv"))
    | set(TABLES.rglob("*.csv"))
)

candidate_records = []

for path in all_csv_files:

    filename_match = any(
        keyword in path.name.lower()
        for keyword in filename_keywords
    )

    try:
        header = pd.read_csv(
            path,
            nrows=0,
        ).columns.tolist()
    except Exception as error:
        candidate_records.append(
            {
                "path": str(path),
                "filename_match": filename_match,
                "schema_match": False,
                "columns": "",
                "read_error": str(error),
            }
        )
        continue

    schema_match = (
        any(
            "runtime" in column.lower()
            or "time_seconds" in column.lower()
            for column in header
        )
        and any(
            column.lower() in {
                "n_items",
                "candidate_count",
                "candidate_set_size",
                "size",
            }
            for column in header
        )
    )

    if filename_match or schema_match:
        candidate_records.append(
            {
                "path": str(path),
                "filename_match": filename_match,
                "schema_match": schema_match,
                "columns": " | ".join(header),
                "read_error": "",
            }
        )

candidate_files = pd.DataFrame(
    candidate_records
)

print("=" * 90)
print("POSSIBLE SCALABILITY OR RUNTIME FILES")
print("=" * 90)

if candidate_files.empty:
    print("No candidate files were detected.")
else:
    display(candidate_files)


# =========================================================
# 2. Inspect each candidate file
# =========================================================

candidate_file_summaries = []

for path_text in candidate_files.get(
    "path",
    pd.Series(dtype=str),
):

    path = Path(path_text)

    try:
        data = pd.read_csv(
            path,
            low_memory=False,
        )
    except Exception as error:
        candidate_file_summaries.append(
            {
                "file": path.name,
                "rows": np.nan,
                "columns": np.nan,
                "n_items_values": "",
                "algorithm_values": "",
                "status_values": "",
                "error": str(error),
            }
        )
        continue

    size_column = next(
        (
            column
            for column in (
                "n_items",
                "candidate_count",
                "candidate_set_size",
                "size",
            )
            if column in data.columns
        ),
        None,
    )

    algorithm_column = next(
        (
            column
            for column in (
                "algorithm",
                "method",
                "solver",
                "algorithm_name",
            )
            if column in data.columns
        ),
        None,
    )

    status_columns = [
        column
        for column in data.columns
        if "status" in column.lower()
    ]

    size_values = ""

    if size_column is not None:
        size_values = " | ".join(
            map(
                str,
                sorted(
                    pd.to_numeric(
                        data[size_column],
                        errors="coerce",
                    )
                    .dropna()
                    .unique()
                ),
            )
        )

    algorithm_values = ""

    if algorithm_column is not None:
        algorithm_values = " | ".join(
            sorted(
                data[algorithm_column]
                .dropna()
                .astype(str)
                .unique()
            )
        )

    status_values = ""

    if status_columns:
        values = []

        for column in status_columns:
            unique_values = sorted(
                data[column]
                .dropna()
                .astype(str)
                .unique()
            )

            values.append(
                f"{column}: {unique_values}"
            )

        status_values = " || ".join(values)

    candidate_file_summaries.append(
        {
            "file": path.name,
            "full_path": str(path),
            "rows": len(data),
            "columns": len(data.columns),
            "size_column": size_column,
            "n_items_values": size_values,
            "algorithm_column": algorithm_column,
            "algorithm_values": algorithm_values,
            "status_values": status_values,
            "missing_value_columns": int(
                data.isna().any().sum()
            ),
            "duplicate_complete_rows": int(
                data.duplicated().sum()
            ),
            "error": "",
        }
    )

candidate_summary = pd.DataFrame(
    candidate_file_summaries
)

print("\n" + "=" * 90)
print("CANDIDATE-FILE SUMMARY")
print("=" * 90)

display(candidate_summary)


# Print schemas and first rows only for reasonably sized candidates.
for _, row in candidate_summary.iterrows():

    if row.get("error", ""):
        continue

    path = Path(row["full_path"])
    data = pd.read_csv(
        path,
        low_memory=False,
    )

    print("\n" + "=" * 90)
    print(path.name)
    print("=" * 90)

    print("\nShape:")
    print(data.shape)

    print("\nColumns:")

    for index, column in enumerate(
        data.columns,
        start=1,
    ):
        print(f"{index:02d}. {column}")

    print("\nFirst 10 rows:")
    display(data.head(10))


# =========================================================
# 3. Audit primary exact and greedy runtime columns
# =========================================================

primary = pd.read_csv(
    PRIMARY_FILE,
    low_memory=False,
)

required_primary_columns = [
    "n_items",
    "exact_status",
    "greedy_status",
    "exact_runtime_seconds",
    "greedy_runtime_seconds",
]

missing_primary_columns = [
    column
    for column in required_primary_columns
    if column not in primary.columns
]

if missing_primary_columns:
    raise KeyError(
        "Primary benchmark is missing columns: "
        f"{missing_primary_columns}"
    )


primary_runtime_integrity = pd.DataFrame(
    [
        {
            "check": "primary_rows",
            "observed": len(primary),
            "expected": 24000,
            "pass": len(primary) == 24000,
        },
        {
            "check": "exact_status_OPTIMAL",
            "observed": int(
                primary["exact_status"]
                .eq("OPTIMAL")
                .sum()
            ),
            "expected": len(primary),
            "pass": bool(
                primary["exact_status"]
                .eq("OPTIMAL")
                .all()
            ),
        },
        {
            "check": "greedy_status_FEASIBLE",
            "observed": int(
                primary["greedy_status"]
                .eq("FEASIBLE")
                .sum()
            ),
            "expected": len(primary),
            "pass": bool(
                primary["greedy_status"]
                .eq("FEASIBLE")
                .all()
            ),
        },
        {
            "check": "finite_exact_runtime",
            "observed": int(
                np.isfinite(
                    primary[
                        "exact_runtime_seconds"
                    ]
                ).sum()
            ),
            "expected": len(primary),
            "pass": bool(
                np.isfinite(
                    primary[
                        "exact_runtime_seconds"
                    ]
                ).all()
            ),
        },
        {
            "check": "finite_greedy_runtime",
            "observed": int(
                np.isfinite(
                    primary[
                        "greedy_runtime_seconds"
                    ]
                ).sum()
            ),
            "expected": len(primary),
            "pass": bool(
                np.isfinite(
                    primary[
                        "greedy_runtime_seconds"
                    ]
                ).all()
            ),
        },
        {
            "check": "positive_exact_runtime",
            "observed": int(
                (
                    primary[
                        "exact_runtime_seconds"
                    ]
                    > 0
                ).sum()
            ),
            "expected": len(primary),
            "pass": bool(
                (
                    primary[
                        "exact_runtime_seconds"
                    ]
                    > 0
                ).all()
            ),
        },
        {
            "check": "positive_greedy_runtime",
            "observed": int(
                (
                    primary[
                        "greedy_runtime_seconds"
                    ]
                    > 0
                ).sum()
            ),
            "expected": len(primary),
            "pass": bool(
                (
                    primary[
                        "greedy_runtime_seconds"
                    ]
                    > 0
                ).all()
            ),
        },
    ]
)

print("\n" + "=" * 90)
print("PRIMARY RUNTIME INTEGRITY")
print("=" * 90)

display(primary_runtime_integrity)


# =========================================================
# 4. Runtime summary by candidate-set size
# =========================================================

def percentile(
    values: pd.Series,
    probability: float,
) -> float:
    return float(
        values.quantile(probability)
    )


primary["runtime_ratio_exact_over_greedy"] = (
    primary["exact_runtime_seconds"]
    / primary["greedy_runtime_seconds"]
)

primary_runtime_by_size = (
    primary
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        rows=(
            "n_items",
            "size",
        ),
        exact_runtime_mean=(
            "exact_runtime_seconds",
            "mean",
        ),
        exact_runtime_median=(
            "exact_runtime_seconds",
            "median",
        ),
        exact_runtime_p90=(
            "exact_runtime_seconds",
            lambda values:
                percentile(values, 0.90),
        ),
        exact_runtime_p95=(
            "exact_runtime_seconds",
            lambda values:
                percentile(values, 0.95),
        ),
        greedy_runtime_mean=(
            "greedy_runtime_seconds",
            "mean",
        ),
        greedy_runtime_median=(
            "greedy_runtime_seconds",
            "median",
        ),
        greedy_runtime_p90=(
            "greedy_runtime_seconds",
            lambda values:
                percentile(values, 0.90),
        ),
        greedy_runtime_p95=(
            "greedy_runtime_seconds",
            lambda values:
                percentile(values, 0.95),
        ),
        runtime_ratio_mean=(
            "runtime_ratio_exact_over_greedy",
            "mean",
        ),
        runtime_ratio_median=(
            "runtime_ratio_exact_over_greedy",
            "median",
        ),
    )
)

print("\n" + "=" * 90)
print("PRIMARY EXACT AND GREEDY RUNTIME BY SIZE")
print("=" * 90)

display(primary_runtime_by_size)


# =========================================================
# 5. Runtime counts by complete factorial cell
# =========================================================

primary_cell_counts = (
    primary
    .groupby(
        [
            "n_items",
            "regime",
            "tau",
            "weight_profile",
        ],
        as_index=False,
    )
    .agg(
        rows=("instance_id", "size"),
        unique_instances=(
            "instance_id",
            "nunique",
        ),
        finite_exact_runtime=(
            "exact_runtime_seconds",
            lambda values:
                int(np.isfinite(values).sum()),
        ),
        finite_greedy_runtime=(
            "greedy_runtime_seconds",
            lambda values:
                int(np.isfinite(values).sum()),
        ),
    )
)

print("\nPRIMARY FACTORIAL CELL COUNTS")
display(primary_cell_counts)


# =========================================================
# 6. Inspect exact and greedy implementations
# =========================================================

MODULE_NAME = "mst_core_scalability_audit"

module_spec = importlib.util.spec_from_file_location(
    MODULE_NAME,
    CORE_MODULE_FILE,
)

if (
    module_spec is None
    or module_spec.loader is None
):
    raise ImportError(
        f"Could not load: {CORE_MODULE_FILE}"
    )

mst_core = importlib.util.module_from_spec(
    module_spec
)

sys.modules[MODULE_NAME] = mst_core

try:
    module_spec.loader.exec_module(mst_core)
except Exception:
    sys.modules.pop(MODULE_NAME, None)
    raise


function_names = [
    "exact_mst",
    "greedy_mst",
]

function_rows = []

for function_name in function_names:

    exists = hasattr(
        mst_core,
        function_name,
    )

    function_rows.append(
        {
            "function": function_name,
            "exists": exists,
            "signature": (
                str(
                    inspect.signature(
                        getattr(
                            mst_core,
                            function_name,
                        )
                    )
                )
                if exists
                else ""
            ),
        }
    )

function_audit = pd.DataFrame(
    function_rows
)

print("\n" + "=" * 90)
print("EXACT AND GREEDY FUNCTION SIGNATURES")
print("=" * 90)

display(function_audit)


for function_name in function_names:

    if not hasattr(
        mst_core,
        function_name,
    ):
        continue

    print("\n" + "=" * 90)
    print(function_name)
    print("=" * 90)

    print(
        inspect.getsource(
            getattr(
                mst_core,
                function_name,
            )
        )
    )


# =========================================================
# 7. Save discovery and primary timing evidence
# =========================================================

candidate_output = (
    RESULTS
    / "results_scalability_candidate_files.csv"
)

runtime_integrity_output = (
    RESULTS
    / "results_primary_runtime_integrity_audit.csv"
)

runtime_size_output = (
    TABLES
    / "results_primary_runtime_by_size_verified.csv"
)

cell_count_output = (
    RESULTS
    / "results_primary_runtime_factorial_cell_counts.csv"
)

function_output = (
    RESULTS
    / "results_scalability_function_signatures.csv"
)

candidate_summary.to_csv(
    candidate_output,
    index=False,
)

primary_runtime_integrity.to_csv(
    runtime_integrity_output,
    index=False,
)

primary_runtime_by_size.to_csv(
    runtime_size_output,
    index=False,
)

primary_cell_counts.to_csv(
    cell_count_output,
    index=False,
)

function_audit.to_csv(
    function_output,
    index=False,
)

print("\nSaved:")
print(candidate_output)
print(runtime_integrity_output)
print(runtime_size_output)
print(cell_count_output)
print(function_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

RAW_FILE = RESULTS / "greedy_scalability_v2.csv"

MATCHED_GROUP_FILE = (
    RESULTS
    / "greedy_scalability_v2_matched_group_runtime.csv"
)

RUNTIME_SIZE_FILE = (
    TABLES
    / "greedy_scalability_v2_runtime_by_candidate_size.csv"
)

MATCHED_SIZE_FILE = (
    TABLES
    / "greedy_scalability_v2_matched_group_runtime_by_size.csv"
)

COMPACTNESS_FILE = (
    TABLES
    / "greedy_scalability_v2_solution_compactness_by_size.csv"
)

POWER_BOOTSTRAP_FILE = (
    RESULTS
    / "greedy_scalability_v2_power_law_bootstrap_replicates.csv"
)

POWER_MODEL_FILE = (
    TABLES
    / "greedy_scalability_v2_power_law_model.csv"
)

LOCAL_RATIO_FILE = (
    TABLES
    / "greedy_scalability_v2_local_doubling_ratios.csv"
)

required_files = [
    RAW_FILE,
    MATCHED_GROUP_FILE,
    RUNTIME_SIZE_FILE,
    MATCHED_SIZE_FILE,
    COMPACTNESS_FILE,
    POWER_BOOTSTRAP_FILE,
    POWER_MODEL_FILE,
    LOCAL_RATIO_FILE,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


raw = pd.read_csv(RAW_FILE, low_memory=False)
stored_groups = pd.read_csv(
    MATCHED_GROUP_FILE,
    low_memory=False,
)
stored_runtime_size = pd.read_csv(
    RUNTIME_SIZE_FILE,
    low_memory=False,
)
stored_matched_size = pd.read_csv(
    MATCHED_SIZE_FILE,
    low_memory=False,
)
stored_compactness = pd.read_csv(
    COMPACTNESS_FILE,
    low_memory=False,
)
power_bootstrap = pd.read_csv(
    POWER_BOOTSTRAP_FILE,
    low_memory=False,
)
stored_power_model = pd.read_csv(
    POWER_MODEL_FILE,
    low_memory=False,
)
stored_local_ratios = pd.read_csv(
    LOCAL_RATIO_FILE,
    low_memory=False,
)


# =========================================================
# 1. Raw-file structure and integrity
# =========================================================

EXPECTED_SIZES = [16, 32, 64, 128, 256]

raw_integrity = pd.DataFrame(
    [
        {
            "check": "raw_rows",
            "observed": len(raw),
            "expected": 24000,
            "passed": len(raw) == 24000,
        },
        {
            "check": "candidate_sizes",
            "observed": str(
                sorted(raw["n_items"].unique())
            ),
            "expected": str(EXPECTED_SIZES),
            "passed": (
                sorted(raw["n_items"].unique())
                == EXPECTED_SIZES
            ),
        },
        {
            "check": "all_status_FEASIBLE",
            "observed": int(
                raw["greedy_status"]
                .eq("FEASIBLE")
                .sum()
            ),
            "expected": len(raw),
            "passed": bool(
                raw["greedy_status"]
                .eq("FEASIBLE")
                .all()
            ),
        },
        {
            "check": "finite_positive_runtime",
            "observed": int(
                (
                    np.isfinite(
                        raw["greedy_runtime_seconds"]
                    )
                    &
                    (
                        raw["greedy_runtime_seconds"]
                        > 0
                    )
                ).sum()
            ),
            "expected": len(raw),
            "passed": bool(
                np.isfinite(
                    raw["greedy_runtime_seconds"]
                ).all()
                and (
                    raw["greedy_runtime_seconds"]
                    > 0
                ).all()
            ),
        },
        {
            "check": "no_complete_duplicate_rows",
            "observed": int(
                raw.duplicated().sum()
            ),
            "expected": 0,
            "passed": not raw.duplicated().any(),
        },
        {
            "check": "no_missing_values",
            "observed": int(
                raw.isna().sum().sum()
            ),
            "expected": 0,
            "passed": not raw.isna().any().any(),
        },
        {
            "check": "benchmark_version",
            "observed": raw[
                "benchmark_version"
            ].nunique(),
            "expected": 1,
            "passed": bool(
                raw["benchmark_version"].eq(
                    "greedy_scalability_"
                    "marginal_preserving_v2"
                ).all()
            ),
        },
        {
            "check": "generator_version",
            "observed": raw[
                "generator_version"
            ].nunique(),
            "expected": 1,
            "passed": bool(
                raw["generator_version"].eq(
                    "marginal_preserving_v2"
                ).all()
            ),
        },
        {
            "check": "regime_strength",
            "observed": raw[
                "regime_strength"
            ].nunique(),
            "expected": 1,
            "passed": bool(
                np.isclose(
                    raw["regime_strength"],
                    0.75,
                ).all()
            ),
        },
    ]
)

print("RAW GREEDY-SCALABILITY INTEGRITY")
display(raw_integrity)


# =========================================================
# 2. Factorial denominators
# =========================================================

size_denominators = (
    raw
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),
        condition_rows=(
            "instance_id",
            "size",
        ),
        feasible_rows=(
            "greedy_status",
            lambda values:
                int(values.eq("FEASIBLE").sum()),
        ),
        regimes=(
            "regime",
            "nunique",
        ),
        thresholds=(
            "tau",
            "nunique",
        ),
        weight_profiles=(
            "weight_profile",
            "nunique",
        ),
    )
)

size_denominators[
    "feasibility_percent"
] = (
    100.0
    * size_denominators["feasible_rows"]
    / size_denominators["condition_rows"]
)

print("\nDENOMINATORS BY CANDIDATE SIZE")
display(size_denominators)


# =========================================================
# 3. Reconstruct row-level runtime summary
# =========================================================

runtime_reconstructed = (
    raw
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),
        condition_rows=(
            "greedy_runtime_seconds",
            "size",
        ),
        feasible_rows=(
            "greedy_status",
            lambda values:
                int(values.eq("FEASIBLE").sum()),
        ),
        median_runtime_seconds=(
            "greedy_runtime_seconds",
            "median",
        ),
        q25_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(0.25),
        ),
        q75_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(0.75),
        ),
        p90_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(0.90),
        ),
        p95_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(0.95),
        ),
        mean_selected_count=(
            "greedy_selected_count",
            "mean",
        ),
    )
)

runtime_reconstructed[
    "median_runtime_milliseconds"
] = (
    1000.0
    * runtime_reconstructed[
        "median_runtime_seconds"
    ]
)

runtime_reconstructed[
    "q25_runtime_milliseconds"
] = (
    1000.0
    * runtime_reconstructed[
        "q25_runtime_seconds"
    ]
)

runtime_reconstructed[
    "q75_runtime_milliseconds"
] = (
    1000.0
    * runtime_reconstructed[
        "q75_runtime_seconds"
    ]
)

runtime_reconstructed[
    "p90_runtime_milliseconds"
] = (
    1000.0
    * runtime_reconstructed[
        "p90_runtime_seconds"
    ]
)

runtime_reconstructed[
    "p95_runtime_milliseconds"
] = (
    1000.0
    * runtime_reconstructed[
        "p95_runtime_seconds"
    ]
)

runtime_reconstructed[
    "feasibility_percent"
] = (
    100.0
    * runtime_reconstructed["feasible_rows"]
    / runtime_reconstructed["condition_rows"]
)

runtime_verification = stored_runtime_size.merge(
    runtime_reconstructed,
    on="n_items",
    suffixes=("_stored", "_reconstructed"),
    validate="one_to_one",
)

runtime_metrics = [
    column
    for column in stored_runtime_size.columns
    if column != "n_items"
]

runtime_check_rows = []

for metric in runtime_metrics:
    residual = (
        runtime_verification[
            f"{metric}_stored"
        ]
        -
        runtime_verification[
            f"{metric}_reconstructed"
        ]
    ).abs()

    runtime_check_rows.append(
        {
            "metric": metric,
            "maximum_residual": float(
                residual.max()
            ),
            "failed_rows": int(
                (residual > 1e-12).sum()
            ),
        }
    )

runtime_verification_summary = pd.DataFrame(
    runtime_check_rows
)

print("\nRECONSTRUCTED ROW-LEVEL RUNTIME TABLE")
display(runtime_reconstructed)

print("\nROW-LEVEL RUNTIME VERIFICATION")
display(runtime_verification_summary)


# =========================================================
# 4. Reconstruct matched-group runtime file
# =========================================================

group_reconstructed = (
    raw
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        condition_rows=(
            "greedy_runtime_seconds",
            "size",
        ),
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),
        median_runtime_seconds=(
            "greedy_runtime_seconds",
            "median",
        ),
        mean_runtime_seconds=(
            "greedy_runtime_seconds",
            "mean",
        ),
        p90_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(0.90),
        ),
        p95_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(0.95),
        ),
        mean_selected_count=(
            "greedy_selected_count",
            "mean",
        ),
        median_selected_count=(
            "greedy_selected_count",
            "median",
        ),
        maximum_selected_count=(
            "greedy_selected_count",
            "max",
        ),
    )
)

group_verification = stored_groups.merge(
    group_reconstructed,
    on=[
        "matched_group_id",
        "n_items",
    ],
    suffixes=("_stored", "_reconstructed"),
    validate="one_to_one",
)

group_metrics = [
    column
    for column in stored_groups.columns
    if column not in {
        "matched_group_id",
        "n_items",
    }
]

group_check_rows = []

for metric in group_metrics:
    residual = (
        group_verification[
            f"{metric}_stored"
        ]
        -
        group_verification[
            f"{metric}_reconstructed"
        ]
    ).abs()

    group_check_rows.append(
        {
            "metric": metric,
            "maximum_residual": float(
                residual.max()
            ),
            "failed_rows": int(
                (residual > 1e-12).sum()
            ),
        }
    )

group_verification_summary = pd.DataFrame(
    group_check_rows
)

print("\nMATCHED-GROUP FILE VERIFICATION")
display(group_verification_summary)


# =========================================================
# 5. Reconstruct matched-group summary by size
# =========================================================

matched_size_reconstructed = (
    group_reconstructed
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        median_group_median_runtime_milliseconds=(
            "median_runtime_seconds",
            lambda values:
                1000.0 * values.median(),
        ),
        q25_group_median_runtime_milliseconds=(
            "median_runtime_seconds",
            lambda values:
                1000.0 * values.quantile(0.25),
        ),
        q75_group_median_runtime_milliseconds=(
            "median_runtime_seconds",
            lambda values:
                1000.0 * values.quantile(0.75),
        ),
        median_group_mean_runtime_milliseconds=(
            "mean_runtime_seconds",
            lambda values:
                1000.0 * values.median(),
        ),
        median_group_p95_runtime_milliseconds=(
            "p95_runtime_seconds",
            lambda values:
                1000.0 * values.median(),
        ),
        mean_selected_count=(
            "mean_selected_count",
            "mean",
        ),
        q25_group_mean_selected_count=(
            "mean_selected_count",
            lambda values:
                values.quantile(0.25),
        ),
        q75_group_mean_selected_count=(
            "mean_selected_count",
            lambda values:
                values.quantile(0.75),
        ),
        median_maximum_selected_count=(
            "maximum_selected_count",
            "median",
        ),
    )
)

matched_size_verification = (
    stored_matched_size.merge(
        matched_size_reconstructed,
        on="n_items",
        suffixes=(
            "_stored",
            "_reconstructed",
        ),
        validate="one_to_one",
    )
)

matched_size_check_rows = []

for metric in [
    column
    for column in stored_matched_size.columns
    if column != "n_items"
]:
    residual = (
        matched_size_verification[
            f"{metric}_stored"
        ]
        -
        matched_size_verification[
            f"{metric}_reconstructed"
        ]
    ).abs()

    matched_size_check_rows.append(
        {
            "metric": metric,
            "maximum_residual": float(
                residual.max()
            ),
            "failed_rows": int(
                (residual > 1e-12).sum()
            ),
        }
    )

matched_size_verification_summary = pd.DataFrame(
    matched_size_check_rows
)

print("\nRECONSTRUCTED MATCHED-GROUP SUMMARY")
display(matched_size_reconstructed)

print("\nMATCHED-GROUP SIZE-SUMMARY VERIFICATION")
display(matched_size_verification_summary)


# =========================================================
# 6. Reconstruct solution compactness table
# =========================================================

raw["selected_fraction_percent"] = (
    100.0
    * raw["greedy_selected_count"]
    / raw["n_items"]
)

raw["candidate_pool_reduction_percent"] = (
    100.0
    - raw["selected_fraction_percent"]
)

raw["utility_surplus"] = (
    raw["greedy_utility"]
    - raw["tau"]
)

raw["selected_at_most_5"] = (
    raw["greedy_selected_count"] <= 5
)

raw["selected_at_most_10"] = (
    raw["greedy_selected_count"] <= 10
)

compactness_reconstructed = (
    raw
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),
        condition_rows=(
            "greedy_selected_count",
            "size",
        ),
        mean_selected_count=(
            "greedy_selected_count",
            "mean",
        ),
        median_selected_count=(
            "greedy_selected_count",
            "median",
        ),
        p90_selected_count=(
            "greedy_selected_count",
            lambda values:
                values.quantile(0.90),
        ),
        p95_selected_count=(
            "greedy_selected_count",
            lambda values:
                values.quantile(0.95),
        ),
        maximum_selected_count=(
            "greedy_selected_count",
            "max",
        ),
        selected_at_most_5_percent=(
            "selected_at_most_5",
            lambda values:
                100.0 * values.mean(),
        ),
        selected_at_most_10_percent=(
            "selected_at_most_10",
            lambda values:
                100.0 * values.mean(),
        ),
        mean_selected_fraction_percent=(
            "selected_fraction_percent",
            "mean",
        ),
        mean_candidate_pool_reduction_percent=(
            "candidate_pool_reduction_percent",
            "mean",
        ),
        mean_utility_surplus=(
            "utility_surplus",
            "mean",
        ),
        median_utility_surplus=(
            "utility_surplus",
            "median",
        ),
        mean_privacy_exposure=(
            "greedy_privacy",
            "mean",
        ),
        mean_normalized_cognitive_cost=(
            "greedy_normalized_cognitive_cost",
            "mean",
        ),
        mean_disclosure_cost=(
            "greedy_disclosure_cost",
            "mean",
        ),
    )
)

compactness_verification = (
    stored_compactness.merge(
        compactness_reconstructed,
        on="n_items",
        suffixes=(
            "_stored",
            "_reconstructed",
        ),
        validate="one_to_one",
    )
)

compactness_check_rows = []

for metric in [
    column
    for column in stored_compactness.columns
    if column != "n_items"
]:
    residual = (
        compactness_verification[
            f"{metric}_stored"
        ]
        -
        compactness_verification[
            f"{metric}_reconstructed"
        ]
    ).abs()

    compactness_check_rows.append(
        {
            "metric": metric,
            "maximum_residual": float(
                residual.max()
            ),
            "failed_rows": int(
                (residual > 1e-12).sum()
            ),
        }
    )

compactness_verification_summary = pd.DataFrame(
    compactness_check_rows
)

print("\nRECONSTRUCTED COMPACTNESS TABLE")
display(compactness_reconstructed)

print("\nCOMPACTNESS-TABLE VERIFICATION")
display(compactness_verification_summary)


# =========================================================
# 7. Reconstruct power-law point estimates
# =========================================================

def linear_fit(
    x: np.ndarray,
    y: np.ndarray,
) -> tuple[float, float, float]:
    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
    )

    predicted = intercept + slope * x

    ss_residual = np.sum(
        (y - predicted) ** 2
    )

    ss_total = np.sum(
        (y - np.mean(y)) ** 2
    )

    r_squared = (
        1.0 - ss_residual / ss_total
        if ss_total > 0
        else np.nan
    )

    return (
        float(slope),
        float(intercept),
        float(r_squared),
    )


runtime_exponent, _, runtime_r2 = (
    linear_fit(
        np.log2(
            group_reconstructed[
                "n_items"
            ].to_numpy(dtype=float)
        ),
        np.log2(
            group_reconstructed[
                "median_runtime_seconds"
            ].to_numpy(dtype=float)
        ),
    )
)

selected_count_slope, _, selected_count_r2 = (
    linear_fit(
        np.log2(
            group_reconstructed[
                "n_items"
            ].to_numpy(dtype=float)
        ),
        group_reconstructed[
            "mean_selected_count"
        ].to_numpy(dtype=float),
    )
)

runtime_factor_per_doubling = (
    2.0 ** runtime_exponent
)


# =========================================================
# 8. Reconstruct bootstrap intervals and model table
# =========================================================

bootstrap_quantities = {
    "runtime_power_law_exponent":
        "runtime_power_law_exponent",

    "runtime_factor_per_doubling":
        "runtime_factor_per_doubling",

    "runtime_model_r_squared":
        "runtime_model_r_squared",

    "selected_count_increase_per_doubling":
        "selected_count_increase_per_doubling",

    "selected_count_model_r_squared":
        "selected_count_model_r_squared",
}

bootstrap_summary_rows = []

for label, column in bootstrap_quantities.items():
    values = power_bootstrap[column]

    bootstrap_summary_rows.append(
        {
            "quantity": label,
            "bootstrap_rows": len(values),
            "bootstrap_standard_error":
                values.std(ddof=1),
            "ci_95_lower":
                values.quantile(0.025),
            "ci_95_upper":
                values.quantile(0.975),
        }
    )

bootstrap_summary = pd.DataFrame(
    bootstrap_summary_rows
)


runtime_model_stored = (
    stored_power_model.loc[
        stored_power_model["parameter"]
        == "runtime_power_law_exponent"
    ].iloc[0]
)

selected_model_stored = (
    stored_power_model.loc[
        stored_power_model["parameter"]
        == "selected_count_increase_per_doubling"
    ].iloc[0]
)

model_verification = pd.DataFrame(
    [
        {
            "quantity":
                "runtime_power_law_exponent",
            "stored":
                runtime_model_stored["estimate"],
            "reconstructed":
                runtime_exponent,
            "residual": abs(
                runtime_model_stored["estimate"]
                - runtime_exponent
            ),
        },
        {
            "quantity":
                "runtime_factor_per_doubling",
            "stored":
                runtime_model_stored[
                    "secondary_estimate"
                ],
            "reconstructed":
                runtime_factor_per_doubling,
            "residual": abs(
                runtime_model_stored[
                    "secondary_estimate"
                ]
                - runtime_factor_per_doubling
            ),
        },
        {
            "quantity":
                "runtime_model_r_squared",
            "stored":
                runtime_model_stored[
                    "observed_r_squared"
                ],
            "reconstructed":
                runtime_r2,
            "residual": abs(
                runtime_model_stored[
                    "observed_r_squared"
                ]
                - runtime_r2
            ),
        },
        {
            "quantity":
                "selected_count_increase_per_doubling",
            "stored":
                selected_model_stored[
                    "estimate"
                ],
            "reconstructed":
                selected_count_slope,
            "residual": abs(
                selected_model_stored[
                    "estimate"
                ]
                - selected_count_slope
            ),
        },
        {
            "quantity":
                "selected_count_model_r_squared",
            "stored":
                selected_model_stored[
                    "observed_r_squared"
                ],
            "reconstructed":
                selected_count_r2,
            "residual": abs(
                selected_model_stored[
                    "observed_r_squared"
                ]
                - selected_count_r2
            ),
        },
    ]
)

print("\nPOWER-LAW POINT-ESTIMATE VERIFICATION")
display(model_verification)

print("\nPOWER-LAW BOOTSTRAP SUMMARY")
display(bootstrap_summary)


# =========================================================
# 9. Verify stored model confidence intervals
# =========================================================

model_interval_verification = pd.DataFrame(
    [
        {
            "quantity":
                "runtime_power_law_exponent",
            "stored_lower":
                runtime_model_stored[
                    "ci_95_lower"
                ],
            "reconstructed_lower":
                power_bootstrap[
                    "runtime_power_law_exponent"
                ].quantile(0.025),
            "stored_upper":
                runtime_model_stored[
                    "ci_95_upper"
                ],
            "reconstructed_upper":
                power_bootstrap[
                    "runtime_power_law_exponent"
                ].quantile(0.975),
        },
        {
            "quantity":
                "runtime_factor_per_doubling",
            "stored_lower":
                runtime_model_stored[
                    "secondary_ci_95_lower"
                ],
            "reconstructed_lower":
                power_bootstrap[
                    "runtime_factor_per_doubling"
                ].quantile(0.025),
            "stored_upper":
                runtime_model_stored[
                    "secondary_ci_95_upper"
                ],
            "reconstructed_upper":
                power_bootstrap[
                    "runtime_factor_per_doubling"
                ].quantile(0.975),
        },
        {
            "quantity":
                "runtime_model_r_squared",
            "stored_lower":
                runtime_model_stored[
                    "r_squared_ci_95_lower"
                ],
            "reconstructed_lower":
                power_bootstrap[
                    "runtime_model_r_squared"
                ].quantile(0.025),
            "stored_upper":
                runtime_model_stored[
                    "r_squared_ci_95_upper"
                ],
            "reconstructed_upper":
                power_bootstrap[
                    "runtime_model_r_squared"
                ].quantile(0.975),
        },
        {
            "quantity":
                "selected_count_increase_per_doubling",
            "stored_lower":
                selected_model_stored[
                    "ci_95_lower"
                ],
            "reconstructed_lower":
                power_bootstrap[
                    "selected_count_increase_per_doubling"
                ].quantile(0.025),
            "stored_upper":
                selected_model_stored[
                    "ci_95_upper"
                ],
            "reconstructed_upper":
                power_bootstrap[
                    "selected_count_increase_per_doubling"
                ].quantile(0.975),
        },
        {
            "quantity":
                "selected_count_model_r_squared",
            "stored_lower":
                selected_model_stored[
                    "r_squared_ci_95_lower"
                ],
            "reconstructed_lower":
                power_bootstrap[
                    "selected_count_model_r_squared"
                ].quantile(0.025),
            "stored_upper":
                selected_model_stored[
                    "r_squared_ci_95_upper"
                ],
            "reconstructed_upper":
                power_bootstrap[
                    "selected_count_model_r_squared"
                ].quantile(0.975),
        },
    ]
)

model_interval_verification[
    "lower_residual"
] = (
    model_interval_verification[
        "stored_lower"
    ]
    -
    model_interval_verification[
        "reconstructed_lower"
    ]
).abs()

model_interval_verification[
    "upper_residual"
] = (
    model_interval_verification[
        "stored_upper"
    ]
    -
    model_interval_verification[
        "reconstructed_upper"
    ]
).abs()

print("\nPOWER-LAW INTERVAL VERIFICATION")
display(model_interval_verification)


# =========================================================
# 10. Verify local doubling ratios
# =========================================================

runtime_by_size_map = (
    matched_size_reconstructed
    .set_index("n_items")[
        "median_group_median_runtime_milliseconds"
    ]
)

ratio_column_map = {
    (16, 32): "runtime_ratio_M32_to_M16",
    (32, 64): "runtime_ratio_M64_to_M32",
    (64, 128): "runtime_ratio_M128_to_M64",
    (128, 256): "runtime_ratio_M256_to_M128",
}

ratio_rows = []

for (
    smaller_size,
    larger_size,
), bootstrap_column in ratio_column_map.items():

    observed_ratio = (
        runtime_by_size_map.loc[larger_size]
        / runtime_by_size_map.loc[smaller_size]
    )

    stored_row = stored_local_ratios.loc[
        (
            stored_local_ratios[
                "smaller_size"
            ]
            == smaller_size
        )
        &
        (
            stored_local_ratios[
                "larger_size"
            ]
            == larger_size
        )
    ].iloc[0]

    ratio_rows.append(
        {
            "contrast":
                f"M={larger_size} / M={smaller_size}",
            "reconstructed_ratio":
                observed_ratio,
            "stored_ratio":
                stored_row[
                    "observed_runtime_ratio"
                ],
            "ratio_residual":
                abs(
                    observed_ratio
                    - stored_row[
                        "observed_runtime_ratio"
                    ]
                ),
            "reconstructed_ci_lower":
                power_bootstrap[
                    bootstrap_column
                ].quantile(0.025),
            "stored_ci_lower":
                stored_row["ci_95_lower"],
            "reconstructed_ci_upper":
                power_bootstrap[
                    bootstrap_column
                ].quantile(0.975),
            "stored_ci_upper":
                stored_row["ci_95_upper"],
            "bootstrap_replicates":
                len(power_bootstrap),
        }
    )

ratio_verification = pd.DataFrame(
    ratio_rows
)

ratio_verification[
    "ci_lower_residual"
] = (
    ratio_verification[
        "reconstructed_ci_lower"
    ]
    -
    ratio_verification[
        "stored_ci_lower"
    ]
).abs()

ratio_verification[
    "ci_upper_residual"
] = (
    ratio_verification[
        "reconstructed_ci_upper"
    ]
    -
    ratio_verification[
        "stored_ci_upper"
    ]
).abs()

print("\nLOCAL DOUBLING-RATIO VERIFICATION")
display(ratio_verification)


# =========================================================
# 11. Final audit summary
# =========================================================

TOLERANCE = 1e-12

final_audit = pd.DataFrame(
    [
        {
            "check": "raw_integrity",
            "failed_rows": int(
                (~raw_integrity["passed"]).sum()
            ),
            "passed": bool(
                raw_integrity["passed"].all()
            ),
        },
        {
            "check":
                "row_runtime_table_reconstructed",
            "failed_rows": int(
                runtime_verification_summary[
                    "failed_rows"
                ].sum()
            ),
            "passed": bool(
                runtime_verification_summary[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "matched_group_file_reconstructed",
            "failed_rows": int(
                group_verification_summary[
                    "failed_rows"
                ].sum()
            ),
            "passed": bool(
                group_verification_summary[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "matched_group_size_table_reconstructed",
            "failed_rows": int(
                matched_size_verification_summary[
                    "failed_rows"
                ].sum()
            ),
            "passed": bool(
                matched_size_verification_summary[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "compactness_table_reconstructed",
            "failed_rows": int(
                compactness_verification_summary[
                    "failed_rows"
                ].sum()
            ),
            "passed": bool(
                compactness_verification_summary[
                    "failed_rows"
                ].eq(0).all()
            ),
        },
        {
            "check":
                "power_law_point_estimates",
            "failed_rows": int(
                (
                    model_verification[
                        "residual"
                    ]
                    > TOLERANCE
                ).sum()
            ),
            "passed": bool(
                (
                    model_verification[
                        "residual"
                    ]
                    <= TOLERANCE
                ).all()
            ),
        },
        {
            "check":
                "power_law_intervals",
            "failed_rows": int(
                (
                    (
                        model_interval_verification[
                            "lower_residual"
                        ]
                        > TOLERANCE
                    )
                    |
                    (
                        model_interval_verification[
                            "upper_residual"
                        ]
                        > TOLERANCE
                    )
                ).sum()
            ),
            "passed": bool(
                (
                    model_interval_verification[
                        [
                            "lower_residual",
                            "upper_residual",
                        ]
                    ]
                    <= TOLERANCE
                ).all().all()
            ),
        },
        {
            "check":
                "local_doubling_ratios",
            "failed_rows": int(
                (
                    (
                        ratio_verification[
                            "ratio_residual"
                        ]
                        > TOLERANCE
                    )
                    |
                    (
                        ratio_verification[
                            "ci_lower_residual"
                        ]
                        > TOLERANCE
                    )
                    |
                    (
                        ratio_verification[
                            "ci_upper_residual"
                        ]
                        > TOLERANCE
                    )
                ).sum()
            ),
            "passed": bool(
                (
                    ratio_verification[
                        [
                            "ratio_residual",
                            "ci_lower_residual",
                            "ci_upper_residual",
                        ]
                    ]
                    <= TOLERANCE
                ).all().all()
            ),
        },
    ]
)

print("\nFINAL LARGE-SIZE GREEDY AUDIT")
display(final_audit)


# =========================================================
# 12. Save verified evidence
# =========================================================

outputs = {
    RESULTS
    / "results_greedy_scalability_raw_integrity.csv":
        raw_integrity,

    RESULTS
    / "results_greedy_scalability_denominators.csv":
        size_denominators,

    TABLES
    / "results_greedy_scalability_runtime_verified.csv":
        runtime_reconstructed,

    RESULTS
    / "results_greedy_scalability_runtime_audit.csv":
        runtime_verification_summary,

    RESULTS
    / "results_greedy_scalability_group_audit.csv":
        group_verification_summary,

    TABLES
    / "results_greedy_scalability_group_summary_verified.csv":
        matched_size_reconstructed,

    TABLES
    / "results_greedy_scalability_compactness_verified.csv":
        compactness_reconstructed,

    RESULTS
    / "results_greedy_scalability_compactness_audit.csv":
        compactness_verification_summary,

    TABLES
    / "results_greedy_scalability_model_verified.csv":
        model_verification,

    RESULTS
    / "results_greedy_scalability_model_interval_audit.csv":
        model_interval_verification,

    TABLES
    / "results_greedy_scalability_doubling_ratios_verified.csv":
        ratio_verification,

    RESULTS
    / "results_greedy_scalability_final_audit.csv":
        final_audit,
}

for path, data in outputs.items():
    data.to_csv(path, index=False)
    print("Saved:")
    print(path)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

SIZE_SUMMARY_FILE = (
    TABLES
    / "results_greedy_scalability_group_summary_verified.csv"
)

POWER_MODEL_FILE = (
    TABLES
    / "greedy_scalability_v2_power_law_model.csv"
)

POWER_BOOTSTRAP_FILE = (
    RESULTS
    / "greedy_scalability_v2_power_law_bootstrap_replicates.csv"
)

for path in (
    SIZE_SUMMARY_FILE,
    POWER_MODEL_FILE,
    POWER_BOOTSTRAP_FILE,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


size_summary = pd.read_csv(
    SIZE_SUMMARY_FILE,
    low_memory=False,
)

stored_model = pd.read_csv(
    POWER_MODEL_FILE,
    low_memory=False,
)

bootstrap = pd.read_csv(
    POWER_BOOTSTRAP_FILE,
    low_memory=False,
)


def linear_fit(
    x: np.ndarray,
    y: np.ndarray,
) -> tuple[float, float, float]:

    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
    )

    predicted = intercept + slope * x

    ss_residual = np.sum(
        (y - predicted) ** 2
    )

    ss_total = np.sum(
        (y - np.mean(y)) ** 2
    )

    r_squared = (
        1.0 - ss_residual / ss_total
        if ss_total > 0
        else np.nan
    )

    return (
        float(slope),
        float(intercept),
        float(r_squared),
    )


# =========================================================
# 1. Correct size-level runtime fit
# =========================================================

x = np.log2(
    size_summary["n_items"].to_numpy(
        dtype=float
    )
)

runtime_y = np.log2(
    size_summary[
        "median_group_median_runtime_milliseconds"
    ].to_numpy(dtype=float)
)

(
    runtime_exponent,
    runtime_intercept,
    runtime_r_squared,
) = linear_fit(
    x,
    runtime_y,
)

runtime_factor_per_doubling = (
    2.0 ** runtime_exponent
)


# =========================================================
# 2. Correct size-level selected-count fit
# =========================================================

selected_count_y = size_summary[
    "mean_selected_count"
].to_numpy(dtype=float)

(
    selected_count_slope,
    selected_count_intercept,
    selected_count_r_squared,
) = linear_fit(
    x,
    selected_count_y,
)


# =========================================================
# 3. Read stored model rows
# =========================================================

runtime_stored = stored_model.loc[
    stored_model["parameter"]
    == "runtime_power_law_exponent"
].iloc[0]

count_stored = stored_model.loc[
    stored_model["parameter"]
    == "selected_count_increase_per_doubling"
].iloc[0]


# =========================================================
# 4. Reconstruct point estimates
# =========================================================

point_verification = pd.DataFrame(
    [
        {
            "quantity":
                "runtime_power_law_exponent",
            "stored":
                runtime_stored["estimate"],
            "reconstructed":
                runtime_exponent,
        },
        {
            "quantity":
                "runtime_factor_per_doubling",
            "stored":
                runtime_stored[
                    "secondary_estimate"
                ],
            "reconstructed":
                runtime_factor_per_doubling,
        },
        {
            "quantity":
                "runtime_model_r_squared",
            "stored":
                runtime_stored[
                    "observed_r_squared"
                ],
            "reconstructed":
                runtime_r_squared,
        },
        {
            "quantity":
                "selected_count_increase_per_doubling",
            "stored":
                count_stored["estimate"],
            "reconstructed":
                selected_count_slope,
        },
        {
            "quantity":
                "selected_count_model_r_squared",
            "stored":
                count_stored[
                    "observed_r_squared"
                ],
            "reconstructed":
                selected_count_r_squared,
        },
    ]
)

point_verification["residual"] = (
    point_verification["stored"]
    -
    point_verification["reconstructed"]
).abs()

point_verification["passed"] = (
    point_verification["residual"]
    <= 1e-12
)


# =========================================================
# 5. Confirm bootstrap support
# =========================================================

bootstrap_summary = pd.DataFrame(
    [
        {
            "quantity":
                "runtime_power_law_exponent",
            "replicates": len(bootstrap),
            "ci_95_lower":
                bootstrap[
                    "runtime_power_law_exponent"
                ].quantile(0.025),
            "ci_95_upper":
                bootstrap[
                    "runtime_power_law_exponent"
                ].quantile(0.975),
        },
        {
            "quantity":
                "runtime_factor_per_doubling",
            "replicates": len(bootstrap),
            "ci_95_lower":
                bootstrap[
                    "runtime_factor_per_doubling"
                ].quantile(0.025),
            "ci_95_upper":
                bootstrap[
                    "runtime_factor_per_doubling"
                ].quantile(0.975),
        },
        {
            "quantity":
                "selected_count_increase_per_doubling",
            "replicates": len(bootstrap),
            "ci_95_lower":
                bootstrap[
                    "selected_count_increase_per_doubling"
                ].quantile(0.025),
            "ci_95_upper":
                bootstrap[
                    "selected_count_increase_per_doubling"
                ].quantile(0.975),
        },
    ]
)


# =========================================================
# 6. Corrected final audit
# =========================================================

corrected_final_audit = pd.DataFrame(
    [
        {
            "check":
                "size_level_power_law_point_estimates",
            "failed_rows": int(
                (~point_verification["passed"]).sum()
            ),
            "passed": bool(
                point_verification["passed"].all()
            ),
        },
        {
            "check":
                "power_law_bootstrap_replicates",
            "failed_rows": int(
                len(bootstrap) != 5000
            ),
            "passed": len(bootstrap) == 5000,
        },
    ]
)


print("CORRECTED SIZE-LEVEL MODEL INPUTS")
display(
    size_summary[
        [
            "n_items",
            "median_group_median_runtime_milliseconds",
            "mean_selected_count",
        ]
    ]
)

print("\nCORRECTED POWER-LAW POINT-ESTIMATE VERIFICATION")
display(point_verification)

print("\nBOOTSTRAP SUPPORT")
display(bootstrap_summary)

print("\nCORRECTED FINAL POWER-LAW AUDIT")
display(corrected_final_audit)


# =========================================================
# 7. Save corrected evidence
# =========================================================

point_output = (
    TABLES
    / "results_greedy_scalability_model_verified_corrected.csv"
)

bootstrap_output = (
    TABLES
    / "results_greedy_scalability_model_bootstrap_verified.csv"
)

audit_output = (
    RESULTS
    / "results_greedy_scalability_power_law_final_audit_corrected.csv"
)

point_verification.to_csv(
    point_output,
    index=False,
)

bootstrap_summary.to_csv(
    bootstrap_output,
    index=False,
)

corrected_final_audit.to_csv(
    audit_output,
    index=False,
)

print("\nSaved:")
print(point_output)
print(bootstrap_output)
print(audit_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

RESULTS = ROOT / "05_Results"
TABLES = ROOT / "07_Tables"

INSTANCE_FILE = (
    RESULTS
    / "corrected_v2_runtime_instance_level.csv"
)

GROUP_FILE = (
    RESULTS
    / "corrected_v2_runtime_matched_group_level.csv"
)

SIZE_FILE = (
    TABLES
    / "corrected_v2_runtime_scaling_by_candidate_size.csv"
)

MODEL_FILE = (
    TABLES
    / "corrected_v2_runtime_scaling_model_fits.csv"
)

BOOTSTRAP_FILE = (
    RESULTS
    / "corrected_v2_runtime_scaling_bootstrap_replicates.csv"
)

for path in (
    INSTANCE_FILE,
    GROUP_FILE,
    SIZE_FILE,
    MODEL_FILE,
    BOOTSTRAP_FILE,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


instance = pd.read_csv(
    INSTANCE_FILE,
    low_memory=False,
)

grouped = pd.read_csv(
    GROUP_FILE,
    low_memory=False,
)

size_summary = pd.read_csv(
    SIZE_FILE,
    low_memory=False,
)

model_fits = pd.read_csv(
    MODEL_FILE,
    low_memory=False,
)

bootstrap = pd.read_csv(
    BOOTSTRAP_FILE,
    low_memory=False,
)


# =========================================================
# 1. Focused file schemas
# =========================================================

def show_file(
    title: str,
    path: Path,
    data: pd.DataFrame,
    complete: bool = False,
    preview_rows: int = 10,
) -> None:

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    print("\nFile:")
    print(path)

    print("\nShape:")
    print(data.shape)

    print("\nColumns:")

    for index, column in enumerate(
        data.columns,
        start=1,
    ):
        print(f"{index:02d}. {column}")

    if complete:
        print("\nComplete contents:")
        display(data)
    else:
        print(f"\nFirst {preview_rows} rows:")
        display(data.head(preview_rows))


show_file(
    "INSTANCE-LEVEL RUNTIME DATA",
    INSTANCE_FILE,
    instance,
)

show_file(
    "MATCHED-GROUP RUNTIME DATA",
    GROUP_FILE,
    grouped,
)

show_file(
    "RUNTIME SCALING BY CANDIDATE SIZE",
    SIZE_FILE,
    size_summary,
    complete=True,
)

show_file(
    "RUNTIME SCALING MODEL FITS",
    MODEL_FILE,
    model_fits,
    complete=True,
)

show_file(
    "RUNTIME SCALING BOOTSTRAP REPLICATES",
    BOOTSTRAP_FILE,
    bootstrap,
    preview_rows=12,
)


# =========================================================
# 2. Basic structural integrity
# =========================================================

EXPECTED_SIZES = [8, 10, 12, 14, 16]

instance_integrity = pd.DataFrame(
    [
        {
            "check": "instance_rows",
            "observed": len(instance),
            "expected": 1500,
            "passed": len(instance) == 1500,
        },
        {
            "check": "matched_group_rows",
            "observed": len(grouped),
            "expected": 500,
            "passed": len(grouped) == 500,
        },
        {
            "check": "candidate_sizes",
            "observed": str(
                sorted(instance["n_items"].unique())
            ),
            "expected": str(EXPECTED_SIZES),
            "passed": (
                sorted(instance["n_items"].unique())
                == EXPECTED_SIZES
            ),
        },
        {
            "check": "instance_complete_duplicates",
            "observed": int(
                instance.duplicated().sum()
            ),
            "expected": 0,
            "passed": not instance.duplicated().any(),
        },
        {
            "check": "group_complete_duplicates",
            "observed": int(
                grouped.duplicated().sum()
            ),
            "expected": 0,
            "passed": not grouped.duplicated().any(),
        },
        {
            "check": "size_summary_rows",
            "observed": len(size_summary),
            "expected": 5,
            "passed": len(size_summary) == 5,
        },
        {
            "check": "model_fit_rows",
            "observed": len(model_fits),
            "expected": 4,
            "passed": len(model_fits) == 4,
        },
        {
            "check": "bootstrap_rows",
            "observed": len(bootstrap),
            "expected": 20000,
            "passed": len(bootstrap) == 20000,
        },
    ]
)

print("\nSTRUCTURAL INTEGRITY")
display(instance_integrity)


# =========================================================
# 3. Instance and group denominators
# =========================================================

instance_denominators = (
    instance
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),
        regimes=(
            "regime",
            "nunique",
        ),
        minimum_conditions=(
            "conditions",
            "min",
        ),
        maximum_conditions=(
            "conditions",
            "max",
        ),
        total_conditions=(
            "conditions",
            "sum",
        ),
    )
)

group_denominators = (
    grouped
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        minimum_regime_specific_instances=(
            "regime_specific_instances",
            "min",
        ),
        maximum_regime_specific_instances=(
            "regime_specific_instances",
            "max",
        ),
    )
)

print("\nINSTANCE-LEVEL DENOMINATORS")
display(instance_denominators)

print("\nMATCHED-GROUP DENOMINATORS")
display(group_denominators)


# =========================================================
# 4. Missing, finite, and positive-value audit
# =========================================================

runtime_columns_instance = [
    column
    for column in instance.columns
    if (
        "seconds" in column.lower()
        or "runtime" in column.lower()
    )
]

runtime_columns_group = [
    column
    for column in grouped.columns
    if (
        "seconds" in column.lower()
        or "runtime" in column.lower()
    )
]

runtime_integrity_rows = []

for file_label, data, columns in (
    (
        "instance_level",
        instance,
        runtime_columns_instance,
    ),
    (
        "matched_group_level",
        grouped,
        runtime_columns_group,
    ),
):

    for column in columns:

        numeric = pd.to_numeric(
            data[column],
            errors="coerce",
        )

        runtime_integrity_rows.append(
            {
                "file": file_label,
                "column": column,
                "rows": len(data),
                "missing_rows": int(
                    numeric.isna().sum()
                ),
                "nonfinite_rows": int(
                    (
                        numeric.notna()
                        & ~np.isfinite(numeric)
                    ).sum()
                ),
                "nonpositive_rows": int(
                    (
                        numeric.notna()
                        & (numeric <= 0)
                    ).sum()
                ),
                "minimum": (
                    float(numeric.min())
                    if numeric.notna().any()
                    else np.nan
                ),
                "maximum": (
                    float(numeric.max())
                    if numeric.notna().any()
                    else np.nan
                ),
            }
        )

runtime_integrity = pd.DataFrame(
    runtime_integrity_rows
)

print("\nRUNTIME-COLUMN INTEGRITY")
display(runtime_integrity)


# =========================================================
# 5. Direct instance-level timing identities
# =========================================================

identity_rows = []


def record_identity(
    name: str,
    stored: pd.Series,
    reconstructed: pd.Series,
) -> None:

    stored_numeric = pd.to_numeric(
        stored,
        errors="coerce",
    )

    reconstructed_numeric = pd.to_numeric(
        reconstructed,
        errors="coerce",
    )

    residual = (
        stored_numeric
        - reconstructed_numeric
    ).abs()

    identity_rows.append(
        {
            "identity": name,
            "rows_checked": len(residual),
            "maximum_residual": float(
                residual.max()
            ),
            "failed_rows_at_1e_minus_12": int(
                (residual > 1e-12).sum()
            ),
        }
    )


required_instance_columns = {
    "conditions",
    "subset_precomputation_seconds",
    "cached_exact_total_seconds",
    "greedy_total_seconds",
}

missing_required = (
    required_instance_columns
    - set(instance.columns)
)

if missing_required:
    raise KeyError(
        "Instance-level file is missing required "
        f"columns: {sorted(missing_required)}"
    )


# Complete exact time for one regime-specific instance.
reconstructed_total_exact = (
    instance["subset_precomputation_seconds"]
    + instance["cached_exact_total_seconds"]
)

if "total_exact_seconds" in instance.columns:
    record_identity(
        "instance total exact = precomputation + cached exact",
        instance["total_exact_seconds"],
        reconstructed_total_exact,
    )


# Cached exact solve time per configured condition.
reconstructed_cached_exact_mean = (
    instance["cached_exact_total_seconds"]
    / instance["conditions"]
)

if (
    "cached_exact_mean_seconds_per_condition"
    in instance.columns
):
    record_identity(
        "cached exact mean per condition",
        instance[
            "cached_exact_mean_seconds_per_condition"
        ],
        reconstructed_cached_exact_mean,
    )


# Greedy time per configured condition.
reconstructed_greedy_mean = (
    instance["greedy_total_seconds"]
    / instance["conditions"]
)

if "greedy_mean_seconds_per_condition" in instance.columns:
    record_identity(
        "greedy mean per condition",
        instance[
            "greedy_mean_seconds_per_condition"
        ],
        reconstructed_greedy_mean,
    )


# End-to-end exact time amortized over the configured conditions.
reconstructed_total_exact_mean = (
    reconstructed_total_exact
    / instance["conditions"]
)

possible_total_exact_mean_columns = [
    column
    for column in (
        "total_exact_mean_seconds_per_condition",
        "amortized_total_exact_seconds_per_condition",
        "total_exact_seconds_per_condition",
    )
    if column in instance.columns
]

for column in possible_total_exact_mean_columns:
    record_identity(
        f"{column} reconstruction",
        instance[column],
        reconstructed_total_exact_mean,
    )


identity_audit = pd.DataFrame(identity_rows)

print("\nINSTANCE-LEVEL TIMING IDENTITIES")

if identity_audit.empty:
    print(
        "No stored identity columns were detected "
        "beyond the source components."
    )
else:
    display(identity_audit)


# =========================================================
# 6. Derived end-to-end exact-versus-greedy summary
# =========================================================

instance_derived = instance[
    [
        "instance_id",
        "matched_group_id",
        "n_items",
        "regime",
        "conditions",
    ]
].copy()

instance_derived[
    "subset_precomputation_seconds"
] = instance[
    "subset_precomputation_seconds"
]

instance_derived[
    "cached_exact_total_seconds"
] = instance[
    "cached_exact_total_seconds"
]

instance_derived[
    "total_exact_seconds"
] = reconstructed_total_exact

instance_derived[
    "cached_exact_mean_seconds_per_condition"
] = reconstructed_cached_exact_mean

instance_derived[
    "total_exact_mean_seconds_per_condition"
] = reconstructed_total_exact_mean

instance_derived[
    "greedy_mean_seconds_per_condition"
] = reconstructed_greedy_mean

instance_derived[
    "cached_exact_over_greedy_ratio"
] = (
    reconstructed_cached_exact_mean
    / reconstructed_greedy_mean
)

instance_derived[
    "end_to_end_exact_over_greedy_ratio"
] = (
    reconstructed_total_exact_mean
    / reconstructed_greedy_mean
)


derived_summary = (
    instance_derived
    .groupby("n_items", as_index=False)
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),
        median_subset_precomputation_seconds=(
            "subset_precomputation_seconds",
            "median",
        ),
        median_cached_exact_seconds_per_condition=(
            "cached_exact_mean_seconds_per_condition",
            "median",
        ),
        median_end_to_end_exact_seconds_per_condition=(
            "total_exact_mean_seconds_per_condition",
            "median",
        ),
        median_greedy_seconds_per_condition=(
            "greedy_mean_seconds_per_condition",
            "median",
        ),
        median_cached_exact_over_greedy_ratio=(
            "cached_exact_over_greedy_ratio",
            "median",
        ),
        median_end_to_end_exact_over_greedy_ratio=(
            "end_to_end_exact_over_greedy_ratio",
            "median",
        ),
    )
)

for column in [
    "median_subset_precomputation_seconds",
    "median_cached_exact_seconds_per_condition",
    "median_end_to_end_exact_seconds_per_condition",
    "median_greedy_seconds_per_condition",
]:
    derived_summary[
        column.replace(
            "_seconds",
            "_milliseconds",
        )
    ] = (
        1000.0
        * derived_summary[column]
    )


print("\nDERIVED END-TO-END EXACT RUNTIME SUMMARY")
display(derived_summary)


# =========================================================
# 7. Stored ratio/speedup column discovery
# =========================================================

ratio_columns_instance = [
    column
    for column in instance.columns
    if (
        "ratio" in column.lower()
        or "speedup" in column.lower()
    )
]

ratio_columns_group = [
    column
    for column in grouped.columns
    if (
        "ratio" in column.lower()
        or "speedup" in column.lower()
    )
]

ratio_columns_size = [
    column
    for column in size_summary.columns
    if (
        "ratio" in column.lower()
        or "speedup" in column.lower()
    )
]

print("\nRATIO OR SPEEDUP COLUMNS")
print("Instance level:")
print(ratio_columns_instance)

print("\nMatched-group level:")
print(ratio_columns_group)

print("\nCandidate-size summary:")
print(ratio_columns_size)


# =========================================================
# 8. Bootstrap design
# =========================================================

bootstrap_group_counts = (
    bootstrap
    .groupby(
        [
            "runtime_component",
            "model_type",
        ],
        dropna=False,
    )
    .agg(
        rows=(
            "bootstrap_replicate",
            "size",
        ),
        unique_replicates=(
            "bootstrap_replicate",
            "nunique",
        ),
        minimum_replicate=(
            "bootstrap_replicate",
            "min",
        ),
        maximum_replicate=(
            "bootstrap_replicate",
            "max",
        ),
    )
    .reset_index()
)

print("\nBOOTSTRAP GROUP COUNTS")
display(bootstrap_group_counts)


# =========================================================
# 9. Model-fit overview
# =========================================================

model_overview_columns = [
    column
    for column in (
        "runtime_component",
        "runtime_column",
        "model_type",
        "observed_slope",
        "observed_intercept",
        "observed_r_squared",
        "reported_parameter",
        "parameter_estimate",
        "parameter_ci_95_lower",
        "parameter_ci_95_upper",
        "secondary_parameter",
        "secondary_estimate",
        "secondary_ci_95_lower",
        "secondary_ci_95_upper",
    )
    if column in model_fits.columns
]

print("\nMODEL-FIT OVERVIEW")
display(
    model_fits[model_overview_columns]
)


# =========================================================
# 10. Final focused audit
# =========================================================

focused_audit = pd.DataFrame(
    [
        {
            "check": "structural_integrity",
            "failed_rows": int(
                (~instance_integrity["passed"]).sum()
            ),
            "passed": bool(
                instance_integrity["passed"].all()
            ),
        },
        {
            "check":
                "all_instance_runtime_values_finite_positive",
            "failed_rows": int(
                (
                    (
                        runtime_integrity[
                            "file"
                        ]
                        == "instance_level"
                    )
                    &
                    (
                        (
                            runtime_integrity[
                                "missing_rows"
                            ]
                            > 0
                        )
                        |
                        (
                            runtime_integrity[
                                "nonfinite_rows"
                            ]
                            > 0
                        )
                        |
                        (
                            runtime_integrity[
                                "nonpositive_rows"
                            ]
                            > 0
                        )
                    )
                ).sum()
            ),
            "passed": bool(
                runtime_integrity.loc[
                    runtime_integrity["file"]
                    == "instance_level",
                    [
                        "missing_rows",
                        "nonfinite_rows",
                        "nonpositive_rows",
                    ],
                ].eq(0).all().all()
            ),
        },
        {
            "check":
                "stored_instance_timing_identities",
            "failed_rows": (
                int(
                    identity_audit[
                        "failed_rows_at_1e_minus_12"
                    ].sum()
                )
                if not identity_audit.empty
                else 0
            ),
            "passed": (
                bool(
                    identity_audit[
                        "failed_rows_at_1e_minus_12"
                    ].eq(0).all()
                )
                if not identity_audit.empty
                else True
            ),
        },
        {
            "check":
                "bootstrap_groups_have_5000_replicates",
            "failed_rows": int(
                (
                    bootstrap_group_counts[
                        "unique_replicates"
                    ]
                    != 5000
                ).sum()
            ),
            "passed": bool(
                bootstrap_group_counts[
                    "unique_replicates"
                ].eq(5000).all()
            ),
        },
    ]
)

print("\nFINAL FOCUSED EXACT-RUNTIME AUDIT")
display(focused_audit)


# =========================================================
# 11. Save focused evidence
# =========================================================

outputs = {
    RESULTS
    / "results_exact_runtime_structural_audit.csv":
        instance_integrity,

    RESULTS
    / "results_exact_runtime_instance_denominators.csv":
        instance_denominators,

    RESULTS
    / "results_exact_runtime_group_denominators.csv":
        group_denominators,

    RESULTS
    / "results_exact_runtime_column_integrity.csv":
        runtime_integrity,

    RESULTS
    / "results_exact_runtime_identity_audit.csv":
        identity_audit,

    TABLES
    / "results_exact_runtime_end_to_end_summary.csv":
        derived_summary,

    RESULTS
    / "results_exact_runtime_bootstrap_groups.csv":
        bootstrap_group_counts,

    RESULTS
    / "results_exact_runtime_focused_final_audit.csv":
        focused_audit,
}

for path, data in outputs.items():
    data.to_csv(path, index=False)
    print("Saved:")
    print(path)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

SARL_DIR = ROOT / "04_SARL_Verification"
RESULTS_DIR = ROOT / "05_Results"


# =========================================================
# 1. Locate the authoritative final SARL sweep
# =========================================================

preferred_locations = [
    RESULTS_DIR / "sarl_mst_multiscenario_sweep.csv",
    SARL_DIR / "sarl_mst_multiscenario_sweep.csv",
]

sweep_file = next(
    (
        path
        for path in preferred_locations
        if path.exists()
    ),
    None,
)

if sweep_file is None:
    candidates = list(
        ROOT.rglob(
            "sarl_mst_multiscenario_sweep.csv"
        )
    )

    if len(candidates) == 1:
        sweep_file = candidates[0]

    elif len(candidates) == 0:
        related = sorted(
            ROOT.rglob("*multiscenario*.csv")
        )

        raise FileNotFoundError(
            "Could not locate "
            "sarl_mst_multiscenario_sweep.csv.\n"
            "Related files:\n"
            + "\n".join(
                str(path)
                for path in related
            )
        )

    else:
        raise RuntimeError(
            "Multiple possible final SARL sweep files "
            "were found:\n"
            + "\n".join(
                str(path)
                for path in candidates
            )
        )


sweep = pd.read_csv(
    sweep_file,
    low_memory=False,
)


print("=" * 90)
print("FINAL SARL MULTI-SCENARIO SWEEP")
print("=" * 90)

print("\nResolved file:")
print(sweep_file)

print("\nShape:")
print(sweep.shape)

print("\nColumns:")

for index, column in enumerate(
    sweep.columns,
    start=1,
):
    print(f"{index:02d}. {column}")

print("\nComplete contents:")
display(sweep)


# =========================================================
# 2. Complete column/value audit
# =========================================================

column_rows = []

for column in sweep.columns:

    series = sweep[column]
    non_null = series.dropna()

    row = {
        "column": column,
        "dtype": str(series.dtype),
        "rows": len(series),
        "non_null_rows": len(non_null),
        "missing_rows": int(
            series.isna().sum()
        ),
        "unique_values": int(
            non_null.nunique()
        ),
    }

    numeric = pd.to_numeric(
        non_null,
        errors="coerce",
    )

    finite = numeric[
        np.isfinite(numeric)
    ]

    if (
        len(non_null) > 0
        and len(finite) == len(non_null)
    ):
        row.update(
            {
                "minimum": float(
                    finite.min()
                ),
                "maximum": float(
                    finite.max()
                ),
                "mean": float(
                    finite.mean()
                ),
            }
        )

    if 0 < non_null.nunique() <= 40:
        row["values"] = " | ".join(
            sorted(
                non_null
                .astype(str)
                .unique()
            )
        )

    column_rows.append(row)


column_audit = pd.DataFrame(
    column_rows
)

print("\n" + "=" * 90)
print("SARL SWEEP COLUMN AND VALUE AUDIT")
print("=" * 90)

display(column_audit)


# =========================================================
# 3. Detect likely semantic columns
# =========================================================

semantic_keywords = {
    "scenario": (
        "scenario",
        "case",
    ),
    "role": (
        "role",
        "stakeholder",
    ),
    "threshold": (
        "tau",
        "threshold",
    ),
    "status": (
        "status",
        "feasible",
        "infeasible",
    ),
    "verification": (
        "verified",
        "verification",
        "check",
        "passed",
    ),
    "nestedness": (
        "nested",
        "nestedness",
    ),
    "selected_count": (
        "selected_count",
        "selection_count",
        "number_selected",
    ),
    "selected_items": (
        "selected_items",
        "selected_ids",
        "items",
    ),
    "utility": (
        "utility",
    ),
    "privacy": (
        "privacy",
    ),
    "interpretation_cost": (
        "cognitive",
        "interpretation",
    ),
    "objective": (
        "disclosure_cost",
        "objective",
    ),
}

semantic_matches = []

for semantic_label, keywords in (
    semantic_keywords.items()
):

    matches = [
        column
        for column in sweep.columns
        if any(
            keyword in column.lower()
            for keyword in keywords
        )
    ]

    semantic_matches.append(
        {
            "semantic_field":
                semantic_label,
            "matching_columns":
                " | ".join(matches),
            "match_count":
                len(matches),
        }
    )


semantic_audit = pd.DataFrame(
    semantic_matches
)

print("\n" + "=" * 90)
print("DETECTED SARL SEMANTIC COLUMNS")
print("=" * 90)

display(semantic_audit)


# =========================================================
# 4. Generic grouped counts for low-cardinality fields
# =========================================================

likely_identifier_columns = [
    column
    for column in sweep.columns
    if any(
        token in column.lower()
        for token in (
            "scenario",
            "role",
            "stakeholder",
            "tau",
            "threshold",
            "status",
            "verified",
            "nested",
            "passed",
            "feasible",
        )
    )
    and sweep[column].nunique(
        dropna=False
    ) <= 40
]

print("\n" + "=" * 90)
print("LOW-CARDINALITY SARL FIELDS")
print("=" * 90)

print(likely_identifier_columns)

for column in likely_identifier_columns:

    print(f"\nCounts for: {column}")

    display(
        sweep[column]
        .value_counts(
            dropna=False,
        )
        .rename("rows")
        .reset_index()
        .rename(
            columns={
                "index": column,
            }
        )
    )


# =========================================================
# 5. Duplicate and missing-value integrity
# =========================================================

file_integrity = pd.DataFrame(
    [
        {
            "check": "total_rows",
            "observed": len(sweep),
            "expected": 36,
            "passed": len(sweep) == 36,
        },
        {
            "check":
                "complete_duplicate_rows",
            "observed": int(
                sweep.duplicated().sum()
            ),
            "expected": 0,
            "passed": not sweep.duplicated().any(),
        },
        {
            "check":
                "columns_with_missing_values",
            "observed": int(
                sweep.isna().any().sum()
            ),
            "expected":
                "inspect_semantically",
            "passed":
                "pending_semantic_audit",
        },
    ]
)

print("\n" + "=" * 90)
print("INITIAL SARL FILE INTEGRITY")
print("=" * 90)

display(file_integrity)


# =========================================================
# 6. Discover authoritative Java and SARL source files
# =========================================================

source_extensions = {
    ".java",
    ".sarl",
    ".xml",
    ".gradle",
}

source_files = sorted(
    path
    for path in SARL_DIR.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in source_extensions
    )
)

source_records = []

for path in source_files:

    text = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    source_records.append(
        {
            "relative_path": str(
                path.relative_to(ROOT)
            ),
            "extension":
                path.suffix.lower(),
            "bytes":
                path.stat().st_size,
            "lines":
                len(text.splitlines()),
            "contains_exact_mst":
                bool(
                    re.search(
                        r"Exact.?MST|ExactMst|exactMst",
                        text,
                        flags=re.IGNORECASE,
                    )
                ),
            "contains_coordinator":
                "coordinator"
                in text.lower(),
            "contains_occupant":
                "occupant"
                in text.lower(),
            "contains_facility_manager":
                (
                    "facility_manager"
                    in text.lower()
                    or
                    "facility manager"
                    in text.lower()
                    or
                    "facilitymanager"
                    in text.lower()
                ),
            "contains_security":
                "security"
                in text.lower(),
            "contains_hvac_fault":
                "hvac_fault"
                in text.lower()
                or
                "hvac fault"
                in text.lower(),
            "contains_comfort_deviation":
                "comfort_deviation"
                in text.lower()
                or
                "comfort deviation"
                in text.lower(),
            "contains_security_alarm":
                "security_alarm"
                in text.lower()
                or
                "security alarm"
                in text.lower(),
        }
    )


source_inventory = pd.DataFrame(
    source_records
)

print("\n" + "=" * 90)
print("SARL/JAVA SOURCE INVENTORY")
print("=" * 90)

if source_inventory.empty:
    print(
        "No Java, SARL, XML, or Gradle source files "
        "were found under:"
    )
    print(SARL_DIR)
else:
    display(source_inventory)


# =========================================================
# 7. Show files containing the central architecture terms
# =========================================================

if not source_inventory.empty:

    central_source_candidates = (
        source_inventory[
            source_inventory[
                [
                    "contains_exact_mst",
                    "contains_coordinator",
                    "contains_occupant",
                    "contains_facility_manager",
                    "contains_security",
                    "contains_hvac_fault",
                    "contains_comfort_deviation",
                    "contains_security_alarm",
                ]
            ].any(axis=1)
        ]
        .copy()
    )

else:
    central_source_candidates = (
        pd.DataFrame()
    )


print("\n" + "=" * 90)
print("CENTRAL SARL ARCHITECTURE SOURCE CANDIDATES")
print("=" * 90)

display(central_source_candidates)


# =========================================================
# 8. Print relevant source excerpts
# =========================================================

SEARCH_TERMS = [
    "ExactMST",
    "ExactMst",
    "coordinator",
    "OCCUPANT",
    "FACILITY_MANAGER",
    "SECURITY",
    "HVAC_FAULT",
    "COMFORT_DEVIATION",
    "SECURITY_ALARM",
    "verified",
    "nested",
]

source_excerpt_rows = []

for path in source_files:

    text_lines = path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    for line_number, line in enumerate(
        text_lines,
        start=1,
    ):

        if any(
            term.lower() in line.lower()
            for term in SEARCH_TERMS
        ):
            source_excerpt_rows.append(
                {
                    "relative_path": str(
                        path.relative_to(ROOT)
                    ),
                    "line_number":
                        line_number,
                    "text":
                        line.strip(),
                }
            )


source_excerpts = pd.DataFrame(
    source_excerpt_rows
)

print("\n" + "=" * 90)
print("RELEVANT SOURCE-LINE MATCHES")
print("=" * 90)

if source_excerpts.empty:
    print(
        "No matching source lines were found."
    )
else:
    display(
        source_excerpts.head(200)
    )

    if len(source_excerpts) > 200:
        print(
            f"\nDisplayed first 200 of "
            f"{len(source_excerpts)} matching lines."
        )


# =========================================================
# 9. Save audit evidence
# =========================================================

outputs = {
    RESULTS_DIR
    / "results_sarl_sweep_column_audit.csv":
        column_audit,

    RESULTS_DIR
    / "results_sarl_semantic_column_audit.csv":
        semantic_audit,

    RESULTS_DIR
    / "results_sarl_initial_file_integrity.csv":
        file_integrity,

    RESULTS_DIR
    / "results_sarl_source_inventory.csv":
        source_inventory,

    RESULTS_DIR
    / "results_sarl_central_source_candidates.csv":
        central_source_candidates,

    RESULTS_DIR
    / "results_sarl_source_line_matches.csv":
        source_excerpts,
}

for path, data in outputs.items():

    data.to_csv(
        path,
        index=False,
    )

    print("Saved:")
    print(path)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import itertools

import numpy as np
import pandas as pd


ROOT = Path(
    PROJECT_ROOT
)

RESULTS = ROOT / "05_Results"

SWEEP_FILE = (
    RESULTS
    / "sarl_mst_multiscenario_sweep.csv"
)

if not SWEEP_FILE.exists():
    raise FileNotFoundError(
        f"Missing final SARL sweep: {SWEEP_FILE}"
    )


sweep = pd.read_csv(
    SWEEP_FILE,
    low_memory=False,
)


# =========================================================
# 1. Normalize types
# =========================================================

BOOLEAN_COLUMNS = [
    "sufficient",
    "privacy_admissible",
    "minimal",
    "verified",
    "nested_applicable",
    "nested_passed",
]


def normalize_boolean(
    series: pd.Series,
) -> pd.Series:

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if numeric.notna().all():
        return numeric.ne(0)

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            {
                "true",
                "1",
                "yes",
                "y",
            }
        )
    )


for column in BOOLEAN_COLUMNS:
    sweep[column] = normalize_boolean(
        sweep[column]
    )

sweep["tau"] = pd.to_numeric(
    sweep["tau"],
    errors="raise",
)

sweep["selected_count"] = pd.to_numeric(
    sweep["selected_count"],
    errors="raise",
).astype(int)


EXPECTED_SCENARIOS = [
    "HVAC_FAULT",
    "COMFORT_DEVIATION",
    "SECURITY_ALARM",
]

EXPECTED_ROLES = [
    "FACILITY_MANAGER",
    "OCCUPANT",
    "SECURITY",
]

EXPECTED_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
    0.95,
]


# =========================================================
# 2. Verify complete factorial design
# =========================================================

expected_grid = pd.DataFrame(
    list(
        itertools.product(
            EXPECTED_SCENARIOS,
            EXPECTED_ROLES,
            EXPECTED_THRESHOLDS,
        )
    ),
    columns=[
        "scenario",
        "role",
        "tau",
    ],
)

observed_grid = (
    sweep[
        [
            "scenario",
            "role",
            "tau",
        ]
    ]
    .drop_duplicates()
)

grid_comparison = expected_grid.merge(
    observed_grid,
    on=[
        "scenario",
        "role",
        "tau",
    ],
    how="outer",
    indicator=True,
)

missing_conditions = grid_comparison[
    grid_comparison["_merge"]
    == "left_only"
]

unexpected_conditions = grid_comparison[
    grid_comparison["_merge"]
    == "right_only"
]

duplicate_condition_rows = int(
    sweep.duplicated(
        subset=[
            "scenario",
            "role",
            "tau",
        ]
    ).sum()
)

factorial_audit = pd.DataFrame(
    [
        {
            "check":
                "total_rows",
            "observed":
                len(sweep),
            "expected":
                36,
            "passed":
                len(sweep) == 36,
        },
        {
            "check":
                "unique_scenario_role_threshold_conditions",
            "observed":
                len(observed_grid),
            "expected":
                36,
            "passed":
                len(observed_grid) == 36,
        },
        {
            "check":
                "missing_factorial_conditions",
            "observed":
                len(missing_conditions),
            "expected":
                0,
            "passed":
                missing_conditions.empty,
        },
        {
            "check":
                "unexpected_factorial_conditions",
            "observed":
                len(unexpected_conditions),
            "expected":
                0,
            "passed":
                unexpected_conditions.empty,
        },
        {
            "check":
                "duplicate_condition_rows",
            "observed":
                duplicate_condition_rows,
            "expected":
                0,
            "passed":
                duplicate_condition_rows == 0,
        },
    ]
)

print("FACTORIAL-DESIGN AUDIT")
display(factorial_audit)


# =========================================================
# 3. Parse selected-item sets
# =========================================================

def parse_selected_items(
    value: object,
    status: str,
) -> frozenset[str]:

    if status == "INFEASIBLE":
        return frozenset()

    text = str(value).strip()

    if (
        text == ""
        or text.upper() == "INFEASIBLE"
    ):
        return frozenset()

    return frozenset(
        item.strip()
        for item in text.split("|")
        if item.strip()
    )


sweep["parsed_selected_items"] = [
    parse_selected_items(
        selected_items,
        status,
    )
    for selected_items, status in zip(
        sweep["selected_items"],
        sweep["status"],
    )
]

sweep[
    "reconstructed_selected_count"
] = sweep[
    "parsed_selected_items"
].map(len)

sweep[
    "selected_count_residual"
] = (
    sweep["selected_count"]
    -
    sweep[
        "reconstructed_selected_count"
    ]
).abs()


# =========================================================
# 4. Feasible-row consistency
# =========================================================

feasible = sweep[
    sweep["status"] == "FEASIBLE"
].copy()

infeasible = sweep[
    sweep["status"] == "INFEASIBLE"
].copy()


feasible_checks = pd.DataFrame(
    [
        {
            "check":
                "feasible_rows",
            "failed_rows": int(
                len(feasible) != 26
            ),
            "passed":
                len(feasible) == 26,
        },
        {
            "check":
                "feasible_selected_count_positive",
            "failed_rows": int(
                (
                    feasible["selected_count"]
                    <= 0
                ).sum()
            ),
            "passed": bool(
                (
                    feasible["selected_count"]
                    > 0
                ).all()
            ),
        },
        {
            "check":
                "feasible_selected_items_not_placeholder",
            "failed_rows": int(
                feasible[
                    "selected_items"
                ]
                .astype(str)
                .str.upper()
                .eq("INFEASIBLE")
                .sum()
            ),
            "passed": bool(
                ~feasible[
                    "selected_items"
                ]
                .astype(str)
                .str.upper()
                .eq("INFEASIBLE")
                .any()
            ),
        },
        {
            "check":
                "feasible_utility_reaches_tau",
            "failed_rows": int(
                (
                    feasible["utility"]
                    <
                    feasible["tau"]
                ).sum()
            ),
            "passed": bool(
                (
                    feasible["utility"]
                    >=
                    feasible["tau"]
                ).all()
            ),
        },
        {
            "check":
                "feasible_sufficient_true",
            "failed_rows": int(
                (~feasible["sufficient"]).sum()
            ),
            "passed": bool(
                feasible["sufficient"].all()
            ),
        },
        {
            "check":
                "feasible_privacy_admissible_true",
            "failed_rows": int(
                (
                    ~feasible[
                        "privacy_admissible"
                    ]
                ).sum()
            ),
            "passed": bool(
                feasible[
                    "privacy_admissible"
                ].all()
            ),
        },
        {
            "check":
                "feasible_minimal_flag_true",
            "failed_rows": int(
                (~feasible["minimal"]).sum()
            ),
            "passed": bool(
                feasible["minimal"].all()
            ),
        },
        {
            "check":
                "feasible_verified_true",
            "failed_rows": int(
                (~feasible["verified"]).sum()
            ),
            "passed": bool(
                feasible["verified"].all()
            ),
        },
        {
            "check":
                "feasible_selected_count_matches_items",
            "failed_rows": int(
                (
                    feasible[
                        "selected_count_residual"
                    ]
                    != 0
                ).sum()
            ),
            "passed": bool(
                feasible[
                    "selected_count_residual"
                ].eq(0).all()
            ),
        },
    ]
)

print("\nFEASIBLE-ROW CONSISTENCY")
display(feasible_checks)


# =========================================================
# 5. Infeasible-row consistency
# =========================================================

ZERO_COLUMNS = [
    "selected_count",
    "utility",
    "privacy_exposure",
    "interpretation_cost",
    "objective",
]

infeasible_checks = pd.DataFrame(
    [
        {
            "check":
                "infeasible_rows",
            "failed_rows": int(
                len(infeasible) != 10
            ),
            "passed":
                len(infeasible) == 10,
        },
        {
            "check":
                "infeasible_placeholder",
            "failed_rows": int(
                (
                    ~infeasible[
                        "selected_items"
                    ]
                    .astype(str)
                    .str.upper()
                    .eq("INFEASIBLE")
                ).sum()
            ),
            "passed": bool(
                infeasible[
                    "selected_items"
                ]
                .astype(str)
                .str.upper()
                .eq("INFEASIBLE")
                .all()
            ),
        },
        {
            "check":
                "infeasible_numeric_outputs_zero",
            "failed_rows": int(
                (
                    infeasible[
                        ZERO_COLUMNS
                    ]
                    .abs()
                    .max(axis=1)
                    > 1e-12
                ).sum()
            ),
            "passed": bool(
                (
                    infeasible[
                        ZERO_COLUMNS
                    ]
                    .abs()
                    <= 1e-12
                ).all().all()
            ),
        },
        {
            "check":
                "infeasible_sufficient_false",
            "failed_rows": int(
                infeasible[
                    "sufficient"
                ].sum()
            ),
            "passed": bool(
                (~infeasible[
                    "sufficient"
                ]).all()
            ),
        },
        {
            "check":
                "infeasible_privacy_flag_false",
            "failed_rows": int(
                infeasible[
                    "privacy_admissible"
                ].sum()
            ),
            "passed": bool(
                (
                    ~infeasible[
                        "privacy_admissible"
                    ]
                ).all()
            ),
        },
        {
            "check":
                "infeasible_minimal_false",
            "failed_rows": int(
                infeasible[
                    "minimal"
                ].sum()
            ),
            "passed": bool(
                (~infeasible[
                    "minimal"
                ]).all()
            ),
        },
        {
            "check":
                "infeasible_verified_true",
            "failed_rows": int(
                (~infeasible[
                    "verified"
                ]).sum()
            ),
            "passed": bool(
                infeasible[
                    "verified"
                ].all()
            ),
        },
        {
            "check":
                "infeasible_nested_not_applicable",
            "failed_rows": int(
                infeasible[
                    "nested_applicable"
                ].sum()
            ),
            "passed": bool(
                (
                    ~infeasible[
                        "nested_applicable"
                    ]
                ).all()
            ),
        },
    ]
)

print("\nINFEASIBLE-ROW CONSISTENCY")
display(infeasible_checks)


# =========================================================
# 6. Reconstruct adjacent-threshold nestedness
# =========================================================

ordered = sweep.sort_values(
    [
        "scenario",
        "role",
        "tau",
    ]
).copy()

nested_rows = []

for (
    scenario,
    role,
), group in ordered.groupby(
    [
        "scenario",
        "role",
    ],
    sort=False,
):

    group = group.sort_values(
        "tau"
    ).reset_index()

    previous_row = None

    for _, row in group.iterrows():

        if previous_row is None:
            reconstructed_applicable = False
            reconstructed_passed = False

        else:
            reconstructed_applicable = bool(
                previous_row["status"]
                == "FEASIBLE"
                and row["status"]
                == "FEASIBLE"
            )

            reconstructed_passed = bool(
                reconstructed_applicable
                and previous_row[
                    "parsed_selected_items"
                ].issubset(
                    row[
                        "parsed_selected_items"
                    ]
                )
            )

        nested_rows.append(
            {
                "original_index":
                    int(row["index"]),
                "scenario":
                    scenario,
                "role":
                    role,
                "tau":
                    row["tau"],
                "status":
                    row["status"],
                "stored_nested_applicable":
                    bool(
                        row[
                            "nested_applicable"
                        ]
                    ),
                "reconstructed_nested_applicable":
                    reconstructed_applicable,
                "stored_nested_passed":
                    bool(
                        row[
                            "nested_passed"
                        ]
                    ),
                "reconstructed_nested_passed":
                    reconstructed_passed,
            }
        )

        previous_row = row


nested_audit = pd.DataFrame(
    nested_rows
)

nested_audit[
    "applicable_flag_matches"
] = (
    nested_audit[
        "stored_nested_applicable"
    ]
    ==
    nested_audit[
        "reconstructed_nested_applicable"
    ]
)

nested_audit[
    "passed_flag_matches"
] = (
    nested_audit[
        "stored_nested_passed"
    ]
    ==
    nested_audit[
        "reconstructed_nested_passed"
    ]
)


nested_summary = pd.DataFrame(
    [
        {
            "check":
                "stored_applicable_comparisons",
            "observed": int(
                sweep[
                    "nested_applicable"
                ].sum()
            ),
            "expected":
                17,
            "passed": int(
                sweep[
                    "nested_applicable"
                ].sum()
            ) == 17,
        },
        {
            "check":
                "reconstructed_applicable_comparisons",
            "observed": int(
                nested_audit[
                    "reconstructed_nested_applicable"
                ].sum()
            ),
            "expected":
                17,
            "passed": int(
                nested_audit[
                    "reconstructed_nested_applicable"
                ].sum()
            ) == 17,
        },
        {
            "check":
                "stored_nestedness_passes",
            "observed": int(
                sweep[
                    "nested_passed"
                ].sum()
            ),
            "expected":
                17,
            "passed": int(
                sweep[
                    "nested_passed"
                ].sum()
            ) == 17,
        },
        {
            "check":
                "applicability_flags_match",
            "observed": int(
                nested_audit[
                    "applicable_flag_matches"
                ].sum()
            ),
            "expected":
                36,
            "passed": bool(
                nested_audit[
                    "applicable_flag_matches"
                ].all()
            ),
        },
        {
            "check":
                "nestedness_flags_match",
            "observed": int(
                nested_audit[
                    "passed_flag_matches"
                ].sum()
            ),
            "expected":
                36,
            "passed": bool(
                nested_audit[
                    "passed_flag_matches"
                ].all()
            ),
        },
    ]
)

print("\nNESTEDNESS RECONSTRUCTION SUMMARY")
display(nested_summary)

print("\nAPPLICABLE NESTEDNESS COMPARISONS")
display(
    nested_audit[
        nested_audit[
            "reconstructed_nested_applicable"
        ]
    ][
        [
            "scenario",
            "role",
            "tau",
            "stored_nested_passed",
            "reconstructed_nested_passed",
        ]
    ]
)


# =========================================================
# 7. Scenario-level summaries
# =========================================================

scenario_summary = (
    sweep
    .groupby(
        "scenario",
        as_index=False,
    )
    .agg(
        total_conditions=(
            "status",
            "size",
        ),
        feasible_conditions=(
            "status",
            lambda values:
                int(
                    values.eq(
                        "FEASIBLE"
                    ).sum()
                ),
        ),
        infeasible_conditions=(
            "status",
            lambda values:
                int(
                    values.eq(
                        "INFEASIBLE"
                    ).sum()
                ),
        ),
        verified_conditions=(
            "verified",
            "sum",
        ),
        nested_applicable=(
            "nested_applicable",
            "sum",
        ),
        nested_passed=(
            "nested_passed",
            "sum",
        ),
        mean_selected_count_all_conditions=(
            "selected_count",
            "mean",
        ),
    )
)

scenario_feasible_means = (
    feasible
    .groupby(
        "scenario",
        as_index=False,
    )
    .agg(
        mean_selected_count_feasible=(
            "selected_count",
            "mean",
        ),
        mean_utility_feasible=(
            "utility",
            "mean",
        ),
        mean_privacy_exposure_feasible=(
            "privacy_exposure",
            "mean",
        ),
        mean_interpretation_cost_feasible=(
            "interpretation_cost",
            "mean",
        ),
        mean_objective_feasible=(
            "objective",
            "mean",
        ),
    )
)

scenario_summary = scenario_summary.merge(
    scenario_feasible_means,
    on="scenario",
    how="left",
    validate="one_to_one",
)

print("\nSCENARIO-LEVEL SUMMARY")
display(scenario_summary)


# =========================================================
# 8. Role-level summaries
# =========================================================

role_summary = (
    sweep
    .groupby(
        "role",
        as_index=False,
    )
    .agg(
        total_conditions=(
            "status",
            "size",
        ),
        feasible_conditions=(
            "status",
            lambda values:
                int(
                    values.eq(
                        "FEASIBLE"
                    ).sum()
                ),
        ),
        infeasible_conditions=(
            "status",
            lambda values:
                int(
                    values.eq(
                        "INFEASIBLE"
                    ).sum()
                ),
        ),
        verified_conditions=(
            "verified",
            "sum",
        ),
        nested_applicable=(
            "nested_applicable",
            "sum",
        ),
        nested_passed=(
            "nested_passed",
            "sum",
        ),
        mean_selected_count_all_conditions=(
            "selected_count",
            "mean",
        ),
    )
)

role_feasible_means = (
    feasible
    .groupby(
        "role",
        as_index=False,
    )
    .agg(
        mean_selected_count_feasible=(
            "selected_count",
            "mean",
        ),
        mean_utility_feasible=(
            "utility",
            "mean",
        ),
        mean_privacy_exposure_feasible=(
            "privacy_exposure",
            "mean",
        ),
        mean_interpretation_cost_feasible=(
            "interpretation_cost",
            "mean",
        ),
        mean_objective_feasible=(
            "objective",
            "mean",
        ),
    )
)

role_summary = role_summary.merge(
    role_feasible_means,
    on="role",
    how="left",
    validate="one_to_one",
)

print("\nROLE-LEVEL SUMMARY")
display(role_summary)


# =========================================================
# 9. Threshold-level summaries
# =========================================================

threshold_summary = (
    sweep
    .groupby(
        "tau",
        as_index=False,
    )
    .agg(
        total_conditions=(
            "status",
            "size",
        ),
        feasible_conditions=(
            "status",
            lambda values:
                int(
                    values.eq(
                        "FEASIBLE"
                    ).sum()
                ),
        ),
        infeasible_conditions=(
            "status",
            lambda values:
                int(
                    values.eq(
                        "INFEASIBLE"
                    ).sum()
                ),
        ),
        verified_conditions=(
            "verified",
            "sum",
        ),
        nested_applicable=(
            "nested_applicable",
            "sum",
        ),
        nested_passed=(
            "nested_passed",
            "sum",
        ),
        mean_selected_count_all_conditions=(
            "selected_count",
            "mean",
        ),
    )
)

print("\nTHRESHOLD-LEVEL SUMMARY")
display(threshold_summary)


# =========================================================
# 10. Utility and flag reconstruction
# =========================================================

sweep[
    "reconstructed_sufficient"
] = (
    (sweep["status"] == "FEASIBLE")
    &
    (sweep["utility"] >= sweep["tau"])
)

utility_flag_audit = pd.DataFrame(
    [
        {
            "check":
                "sufficient_flag_matches_utility_and_status",
            "failed_rows": int(
                (
                    sweep[
                        "sufficient"
                    ]
                    !=
                    sweep[
                        "reconstructed_sufficient"
                    ]
                ).sum()
            ),
            "passed": bool(
                (
                    sweep[
                        "sufficient"
                    ]
                    ==
                    sweep[
                        "reconstructed_sufficient"
                    ]
                ).all()
            ),
        },
        {
            "check":
                "all_rows_verified",
            "failed_rows": int(
                (~sweep["verified"]).sum()
            ),
            "passed": bool(
                sweep["verified"].all()
            ),
        },
        {
            "check":
                "selected_counts_match_item_sets",
            "failed_rows": int(
                (
                    sweep[
                        "selected_count_residual"
                    ]
                    != 0
                ).sum()
            ),
            "passed": bool(
                sweep[
                    "selected_count_residual"
                ].eq(0).all()
            ),
        },
    ]
)

print("\nUTILITY AND FLAG RECONSTRUCTION")
display(utility_flag_audit)


# =========================================================
# 11. Overall final result-level audit
# =========================================================

final_audit = pd.DataFrame(
    [
        {
            "check":
                "factorial_design",
            "failed_rows": int(
                (~factorial_audit[
                    "passed"
                ]).sum()
            ),
            "passed": bool(
                factorial_audit[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "feasible_row_consistency",
            "failed_rows": int(
                (~feasible_checks[
                    "passed"
                ]).sum()
            ),
            "passed": bool(
                feasible_checks[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "infeasible_row_consistency",
            "failed_rows": int(
                (~infeasible_checks[
                    "passed"
                ]).sum()
            ),
            "passed": bool(
                infeasible_checks[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "adjacent_threshold_nestedness",
            "failed_rows": int(
                (~nested_summary[
                    "passed"
                ]).sum()
            ),
            "passed": bool(
                nested_summary[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "utility_and_stored_flags",
            "failed_rows": int(
                (~utility_flag_audit[
                    "passed"
                ]).sum()
            ),
            "passed": bool(
                utility_flag_audit[
                    "passed"
                ].all()
            ),
        },
    ]
)

print("\nFINAL SARL RESULT-LEVEL AUDIT")
display(final_audit)


# =========================================================
# 12. Save verified evidence
# =========================================================

outputs = {
    RESULTS
    / "results_sarl_factorial_design_audit.csv":
        factorial_audit,

    RESULTS
    / "results_sarl_feasible_row_audit.csv":
        feasible_checks,

    RESULTS
    / "results_sarl_infeasible_row_audit.csv":
        infeasible_checks,

    RESULTS
    / "results_sarl_nestedness_reconstruction.csv":
        nested_audit,

    RESULTS
    / "results_sarl_nestedness_summary.csv":
        nested_summary,

    RESULTS
    / "results_sarl_scenario_summary_verified.csv":
        scenario_summary,

    RESULTS
    / "results_sarl_role_summary_verified.csv":
        role_summary,

    RESULTS
    / "results_sarl_threshold_summary_verified.csv":
        threshold_summary,

    RESULTS
    / "results_sarl_utility_flag_audit.csv":
        utility_flag_audit,

    RESULTS
    / "results_sarl_final_result_level_audit.csv":
        final_audit,
}

for path, data in outputs.items():
    data.to_csv(
        path,
        index=False,
    )

    print("Saved:")
    print(path)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from pathlib import Path
import re

import pandas as pd


PROJECT_ROOT = Path(
    PROJECT_ROOT
)

DESKTOP_ROOT = Path(
    PROJECT_ROOT
)

RESULTS_DIR = PROJECT_ROOT / "05_Results"

SEARCH_ROOTS = [
    PROJECT_ROOT,
    DESKTOP_ROOT,
]

SOURCE_EXTENSIONS = {
    ".java",
    ".sarl",
    ".xtend",
    ".xml",
    ".gradle",
    ".properties",
}

SKIP_DIRECTORY_NAMES = {
    ".git",
    ".idea",
    ".gradle",
    ".metadata",
    ".settings",
    "node_modules",
    "dist",
    "build",
    "bin",
    "target",
    "out",
    "__pycache__",
}

SEARCH_TERMS = [
    "ExactMST",
    "ExactMst",
    "Exact MST",
    "MSTCoordinator",
    "CoordinatorAgent",
    "FACILITY_MANAGER",
    "OCCUPANT",
    "SECURITY",
    "HVAC_FAULT",
    "COMFORT_DEVIATION",
    "SECURITY_ALARM",
    "privacyAdmissible",
    "privacy_admissible",
    "nestedApplicable",
    "nested_applicable",
    "nestedPassed",
    "nested_passed",
    "MULTI-SCENARIO SWEEP COMPLETE",
    "verified=36/36",
    "nested=17/17",
    "sarl_mst_multiscenario_sweep.csv",
]


def should_skip(path: Path) -> bool:
    return any(
        part.lower() in {
            name.lower()
            for name in SKIP_DIRECTORY_NAMES
        }
        for part in path.parts
    )


def safe_relative_path(
    path: Path,
    root: Path,
) -> str:
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)


# =========================================================
# 1. Locate source files
# =========================================================

seen_paths: set[Path] = set()
records = []
line_matches = []

for search_root in SEARCH_ROOTS:

    if not search_root.exists():
        print(
            f"Search root does not exist: "
            f"{search_root}"
        )
        continue

    for path in search_root.rglob("*"):

        if (
            not path.is_file()
            or path.suffix.lower()
            not in SOURCE_EXTENSIONS
            or should_skip(path)
        ):
            continue

        resolved = path.resolve()

        if resolved in seen_paths:
            continue

        seen_paths.add(resolved)

        try:
            text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        except Exception as error:
            records.append(
                {
                    "full_path": str(path),
                    "relative_to_project":
                        safe_relative_path(
                            path,
                            PROJECT_ROOT,
                        ),
                    "extension":
                        path.suffix.lower(),
                    "bytes":
                        path.stat().st_size,
                    "lines":
                        0,
                    "matched_term_count":
                        0,
                    "matched_terms":
                        "",
                    "contains_exact_mst":
                        False,
                    "contains_coordinator":
                        False,
                    "contains_role_terms":
                        False,
                    "contains_scenarios":
                        False,
                    "contains_final_console_line":
                        False,
                    "read_error":
                        str(error),
                }
            )
            continue

        matched_terms = [
            term
            for term in SEARCH_TERMS
            if term.lower() in text.lower()
        ]

        contains_roles = all(
            term.lower() in text.lower()
            for term in (
                "FACILITY_MANAGER",
                "OCCUPANT",
                "SECURITY",
            )
        )

        contains_scenarios = all(
            term.lower() in text.lower()
            for term in (
                "HVAC_FAULT",
                "COMFORT_DEVIATION",
                "SECURITY_ALARM",
            )
        )

        records.append(
            {
                "full_path": str(path),
                "relative_to_project":
                    safe_relative_path(
                        path,
                        PROJECT_ROOT,
                    ),
                "extension":
                    path.suffix.lower(),
                "bytes":
                    path.stat().st_size,
                "lines":
                    len(text.splitlines()),
                "matched_term_count":
                    len(matched_terms),
                "matched_terms":
                    " | ".join(matched_terms),
                "contains_exact_mst":
                    bool(
                        re.search(
                            r"Exact.?MST|ExactMst",
                            text,
                            flags=re.IGNORECASE,
                        )
                    ),
                "contains_coordinator":
                    "coordinator"
                    in text.lower(),
                "contains_role_terms":
                    contains_roles,
                "contains_scenarios":
                    contains_scenarios,
                "contains_final_console_line":
                    (
                        "verified=36/36"
                        in text
                        or
                        "nested=17/17"
                        in text
                        or
                        "MULTI-SCENARIO SWEEP COMPLETE"
                        in text
                    ),
                "read_error":
                    "",
            }
        )

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):
            line_terms = [
                term
                for term in SEARCH_TERMS
                if term.lower() in line.lower()
            ]

            if line_terms:
                line_matches.append(
                    {
                        "full_path":
                            str(path),
                        "relative_to_project":
                            safe_relative_path(
                                path,
                                PROJECT_ROOT,
                            ),
                        "line_number":
                            line_number,
                        "matched_terms":
                            " | ".join(line_terms),
                        "text":
                            line.strip(),
                    }
                )


inventory = pd.DataFrame(records)
matches = pd.DataFrame(line_matches)


# =========================================================
# 2. Rank likely authoritative sources
# =========================================================

if inventory.empty:

    ranked = pd.DataFrame()

else:

    inventory["source_score"] = (
        5
        * inventory[
            "contains_final_console_line"
        ].astype(int)
        +
        4
        * inventory[
            "contains_exact_mst"
        ].astype(int)
        +
        3
        * inventory[
            "contains_scenarios"
        ].astype(int)
        +
        2
        * inventory[
            "contains_role_terms"
        ].astype(int)
        +
        2
        * inventory[
            "contains_coordinator"
        ].astype(int)
        +
        inventory[
            "matched_term_count"
        ]
    )

    ranked = inventory.sort_values(
        [
            "source_score",
            "matched_term_count",
            "extension",
            "full_path",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    ).reset_index(drop=True)


print("=" * 90)
print("SOURCE SEARCH SUMMARY")
print("=" * 90)

print("\nSource files inspected:")
print(len(inventory))

print("\nFiles with at least one relevant term:")
print(
    int(
        (
            inventory[
                "matched_term_count"
            ] > 0
        ).sum()
    )
    if not inventory.empty
    else 0
)


print("\n" + "=" * 90)
print("TOP LIKELY AUTHORITATIVE SOURCE FILES")
print("=" * 90)

if ranked.empty:
    print("No candidate source files were found.")
else:
    display(
        ranked[
            ranked[
                "matched_term_count"
            ] > 0
        ][
            [
                "source_score",
                "full_path",
                "extension",
                "bytes",
                "lines",
                "matched_term_count",
                "matched_terms",
                "contains_exact_mst",
                "contains_coordinator",
                "contains_role_terms",
                "contains_scenarios",
                "contains_final_console_line",
            ]
        ].head(50)
    )


# =========================================================
# 3. Display relevant source-line matches
# =========================================================

print("\n" + "=" * 90)
print("RELEVANT SOURCE-LINE MATCHES")
print("=" * 90)

if matches.empty:
    print("No relevant source lines were found.")
else:
    ranked_paths = {
        path: rank
        for rank, path in enumerate(
            ranked["full_path"],
            start=1,
        )
    }

    matches["file_rank"] = (
        matches["full_path"]
        .map(ranked_paths)
        .fillna(10**9)
    )

    matches = matches.sort_values(
        [
            "file_rank",
            "line_number",
        ]
    ).reset_index(drop=True)

    display(
        matches[
            [
                "full_path",
                "line_number",
                "matched_terms",
                "text",
            ]
        ].head(300)
    )

    if len(matches) > 300:
        print(
            f"Displayed first 300 of "
            f"{len(matches)} line matches."
        )


# =========================================================
# 4. Print complete contents of strongest small candidates
# =========================================================

strong_candidates = (
    ranked[
        (
            ranked["source_score"] >= 6
        )
        &
        (
            ranked["lines"] <= 800
        )
    ]
    .head(10)
)

print("\n" + "=" * 90)
print("COMPLETE STRONG-CANDIDATE SOURCES")
print("=" * 90)

if strong_candidates.empty:
    print(
        "No candidate met the score and "
        "size limits for complete printing."
    )

for _, row in strong_candidates.iterrows():

    path = Path(row["full_path"])

    print("\n" + "#" * 90)
    print(path)
    print("#" * 90)

    print(
        path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    )


# =========================================================
# 5. Save the source-location evidence
# =========================================================

inventory_output = (
    RESULTS_DIR
    / "results_sarl_desktop_source_inventory.csv"
)

ranked_output = (
    RESULTS_DIR
    / "results_sarl_desktop_source_candidates_ranked.csv"
)

matches_output = (
    RESULTS_DIR
    / "results_sarl_desktop_source_line_matches.csv"
)

inventory.to_csv(
    inventory_output,
    index=False,
)

ranked.to_csv(
    ranked_output,
    index=False,
)

matches.to_csv(
    matches_output,
    index=False,
)

print("\nSaved:")
print(inventory_output)
print(ranked_output)
print(matches_output)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Final

import numpy as np
import pandas as pd


PROJECT_ROOT = Path(
    PROJECT_ROOT
)

RESULTS = PROJECT_ROOT / "05_Results"

SWEEP_FILE = (
    RESULTS
    / "sarl_mst_multiscenario_sweep.csv"
)

if not SWEEP_FILE.exists():
    raise FileNotFoundError(
        f"Missing final SARL sweep: {SWEEP_FILE}"
    )


stored = pd.read_csv(
    SWEEP_FILE,
    low_memory=False,
)


# =========================================================
# 1. Source-level constants
# =========================================================

EPSILON: Final[float] = 1e-10

SCENARIOS: Final[tuple[str, ...]] = (
    "HVAC_FAULT",
    "COMFORT_DEVIATION",
    "SECURITY_ALARM",
)

THRESHOLDS: Final[tuple[float, ...]] = (
    0.70,
    0.80,
    0.90,
    0.95,
)

ROLES: Final[tuple[str, ...]] = (
    "OCCUPANT",
    "FACILITY_MANAGER",
    "SECURITY",
)


@dataclass(frozen=True)
class Item:
    item_name: str
    utility: float
    privacy: float
    interpretation_cost: float
    admissible: bool


@dataclass(frozen=True)
class Choice:
    mask: int
    selected_items: str
    selected_count: int
    utility: float
    privacy_exposure: float
    interpretation_cost: float
    objective: float


# =========================================================
# 2. Direct mirror of MSTService.java item definitions
# =========================================================

ITEMS: dict[
    tuple[str, str],
    tuple[Item, ...],
] = {
    (
        "HVAC_FAULT",
        "OCCUPANT",
    ): (
        Item(
            "zone_temperature",
            0.70,
            0.08,
            0.20,
            True,
        ),
        Item(
            "comfort_deviation",
            0.68,
            0.05,
            0.18,
            True,
        ),
        Item(
            "maintenance_staff_identity",
            0.40,
            0.45,
            0.30,
            False,
        ),
    ),

    (
        "HVAC_FAULT",
        "FACILITY_MANAGER",
    ): (
        Item(
            "hvac_fault_code",
            0.82,
            0.15,
            0.30,
            True,
        ),
        Item(
            "airflow_anomaly",
            0.58,
            0.10,
            0.24,
            True,
        ),
        Item(
            "zone_occupancy_count",
            0.45,
            0.25,
            0.35,
            True,
        ),
    ),

    (
        "HVAC_FAULT",
        "SECURITY",
    ): (
        Item(
            "safety_interlock_status",
            0.76,
            0.08,
            0.20,
            True,
        ),
        Item(
            "affected_access_zone",
            0.55,
            0.16,
            0.22,
            True,
        ),
        Item(
            "occupant_identity_trace",
            0.42,
            0.60,
            0.40,
            False,
        ),
    ),

    (
        "COMFORT_DEVIATION",
        "OCCUPANT",
    ): (
        Item(
            "zone_temperature",
            0.72,
            0.08,
            0.18,
            True,
        ),
        Item(
            "comfort_target_difference",
            0.64,
            0.06,
            0.18,
            True,
        ),
        Item(
            "neighbour_zone_occupancy",
            0.42,
            0.48,
            0.34,
            False,
        ),
    ),

    (
        "COMFORT_DEVIATION",
        "FACILITY_MANAGER",
    ): (
        Item(
            "temperature_trend",
            0.76,
            0.10,
            0.24,
            True,
        ),
        Item(
            "hvac_operating_mode",
            0.66,
            0.12,
            0.26,
            True,
        ),
        Item(
            "aggregated_occupancy_level",
            0.48,
            0.22,
            0.30,
            True,
        ),
    ),

    (
        "COMFORT_DEVIATION",
        "SECURITY",
    ): (
        Item(
            "safety_threshold_status",
            0.78,
            0.07,
            0.18,
            True,
        ),
        Item(
            "affected_zone",
            0.44,
            0.12,
            0.18,
            True,
        ),
        Item(
            "occupant_identity",
            0.35,
            0.58,
            0.36,
            False,
        ),
    ),

    (
        "SECURITY_ALARM",
        "OCCUPANT",
    ): (
        Item(
            "alarm_status",
            0.74,
            0.07,
            0.18,
            True,
        ),
        Item(
            "affected_access_zone",
            0.58,
            0.12,
            0.20,
            True,
        ),
        Item(
            "camera_identity_trace",
            0.46,
            0.62,
            0.40,
            False,
        ),
    ),

    (
        "SECURITY_ALARM",
        "FACILITY_MANAGER",
    ): (
        Item(
            "alarm_status",
            0.72,
            0.08,
            0.18,
            True,
        ),
        Item(
            "affected_building_zone",
            0.60,
            0.14,
            0.22,
            True,
        ),
        Item(
            "device_health_summary",
            0.52,
            0.12,
            0.24,
            True,
        ),
    ),

    (
        "SECURITY_ALARM",
        "SECURITY",
    ): (
        Item(
            "alarm_status",
            0.88,
            0.10,
            0.18,
            True,
        ),
        Item(
            "affected_access_zone",
            0.62,
            0.16,
            0.20,
            True,
        ),
        Item(
            "camera_event_summary",
            0.54,
            0.24,
            0.26,
            True,
        ),
    ),
}


# =========================================================
# 3. Direct mirror of source calculations
# =========================================================

def product_union(
    current_value: float,
    contribution: float,
) -> float:
    return 1.0 - (
        (1.0 - current_value)
        * (1.0 - contribution)
    )


def evaluate_subset(
    items: tuple[Item, ...],
    mask: int,
    tau: float,
) -> Choice | None:

    selected_names: list[str] = []

    selected_count = 0
    utility = 0.0
    privacy_exposure = 0.0
    interpretation_cost = 0.0

    # The Java implementation includes all supplied items,
    # including inadmissible items, in this denominator.
    total_interpretation_cost = sum(
        item.interpretation_cost
        for item in items
    )

    for index, item in enumerate(items):

        if mask & (1 << index) == 0:
            continue

        if not item.admissible:
            return None

        selected_names.append(
            item.item_name
        )

        selected_count += 1

        utility = product_union(
            utility,
            item.utility,
        )

        privacy_exposure = product_union(
            privacy_exposure,
            item.privacy,
        )

        interpretation_cost += (
            item.interpretation_cost
        )

    if (
        selected_count == 0
        or utility + EPSILON < tau
    ):
        return None

    normalized_interpretation_cost = (
        interpretation_cost
        / total_interpretation_cost
    )

    normalized_cardinality = (
        selected_count
        / len(items)
    )

    objective = (
        privacy_exposure
        + normalized_interpretation_cost
        + normalized_cardinality
    ) / 3.0

    return Choice(
        mask=mask,
        selected_items="|".join(
            selected_names
        ),
        selected_count=selected_count,
        utility=utility,
        privacy_exposure=privacy_exposure,
        interpretation_cost=interpretation_cost,
        objective=objective,
    )


def verify_minimality(
    items: tuple[Item, ...],
    selected_mask: int,
    tau: float,
) -> bool:

    for index in range(len(items)):

        if (
            selected_mask
            & (1 << index)
        ) == 0:
            continue

        reduced_mask = (
            selected_mask
            & ~(1 << index)
        )

        if (
            reduced_mask != 0
            and evaluate_subset(
                items,
                reduced_mask,
                tau,
            )
            is not None
        ):
            return False

    return True


def exact_java_mst(
    scenario: str,
    role: str,
    tau: float,
) -> dict[str, object]:

    items = ITEMS[
        (
            scenario,
            role,
        )
    ]

    best: Choice | None = None

    for mask in range(
        1,
        1 << len(items),
    ):

        candidate = evaluate_subset(
            items,
            mask,
            tau,
        )

        if candidate is None:
            continue

        replace_best = (
            best is None
            or (
                candidate.objective
                + EPSILON
                < best.objective
            )
            or (
                abs(
                    candidate.objective
                    - best.objective
                )
                <= EPSILON
                and candidate.selected_count
                < best.selected_count
            )
        )

        if replace_best:
            best = candidate

    if best is None:
        return {
            "scenario": scenario,
            "tau": tau,
            "role": role,
            "status": "INFEASIBLE",
            "selected_items": "INFEASIBLE",
            "selected_count": 0,
            "utility": 0.0,
            "privacy_exposure": 0.0,
            "interpretation_cost": 0.0,
            "objective": 0.0,
            "sufficient": False,
            "privacy_admissible": False,
            "minimal": False,
        }

    minimal = verify_minimality(
        items,
        best.mask,
        tau,
    )

    return {
        "scenario": scenario,
        "tau": tau,
        "role": role,
        "status": "FEASIBLE",
        "selected_items":
            best.selected_items,
        "selected_count":
            best.selected_count,
        "utility":
            best.utility,
        "privacy_exposure":
            best.privacy_exposure,
        "interpretation_cost":
            best.interpretation_cost,
        "objective":
            best.objective,
        "sufficient":
            best.utility + EPSILON >= tau,
        "privacy_admissible":
            True,
        "minimal":
            minimal,
    }


# =========================================================
# 4. Reproduce MSTSweepRecorder behavior
# =========================================================

expected_rows: list[
    dict[str, object]
] = []

previous_feasible_selection: dict[
    str,
    set[str],
] = {}

for scenario in SCENARIOS:

    for tau in THRESHOLDS:

        for role in ROLES:

            result = exact_java_mst(
                scenario,
                role,
                tau,
            )

            infeasible = (
                result["selected_items"]
                == "INFEASIBLE"
            )

            if infeasible:
                verified = bool(
                    not result["sufficient"]
                    and result[
                        "selected_count"
                    ] == 0
                )
            else:
                verified = bool(
                    result["sufficient"]
                    and result[
                        "privacy_admissible"
                    ]
                    and result["minimal"]
                )

            sequence_key = (
                f"{scenario}::{role}"
            )

            nested_applicable = False
            nested_passed = False

            if not infeasible:

                current_selection = set(
                    str(
                        result[
                            "selected_items"
                        ]
                    ).split("|")
                )

                previous_selection = (
                    previous_feasible_selection
                    .get(sequence_key)
                )

                if previous_selection is not None:

                    nested_applicable = True

                    nested_passed = (
                        current_selection
                        .issuperset(
                            previous_selection
                        )
                    )

                previous_feasible_selection[
                    sequence_key
                ] = current_selection

            result["verified"] = verified

            result[
                "nested_applicable"
            ] = nested_applicable

            result[
                "nested_passed"
            ] = nested_passed

            expected_rows.append(result)


expected = pd.DataFrame(
    expected_rows
)


# =========================================================
# 5. Normalize stored field types
# =========================================================

BOOLEAN_COLUMNS = [
    "sufficient",
    "privacy_admissible",
    "minimal",
    "verified",
    "nested_applicable",
    "nested_passed",
]


def normalize_boolean(
    series: pd.Series,
) -> pd.Series:

    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            {
                "true",
                "1",
                "yes",
                "y",
            }
        )
    )


for column in BOOLEAN_COLUMNS:

    stored[column] = normalize_boolean(
        stored[column]
    )

    expected[column] = normalize_boolean(
        expected[column]
    )


KEY_COLUMNS = [
    "scenario",
    "tau",
    "role",
]

merged = stored.merge(
    expected,
    on=KEY_COLUMNS,
    how="outer",
    suffixes=(
        "_stored",
        "_expected",
    ),
    indicator=True,
    validate="one_to_one",
)


# =========================================================
# 6. Verify keys and categorical outputs
# =========================================================

key_audit = pd.DataFrame(
    [
        {
            "check":
                "stored_rows",
            "observed":
                len(stored),
            "expected":
                36,
            "passed":
                len(stored) == 36,
        },
        {
            "check":
                "reconstructed_rows",
            "observed":
                len(expected),
            "expected":
                36,
            "passed":
                len(expected) == 36,
        },
        {
            "check":
                "matched_scenario_role_threshold_keys",
            "observed":
                int(
                    merged["_merge"]
                    .eq("both")
                    .sum()
                ),
            "expected":
                36,
            "passed": bool(
                merged["_merge"]
                .eq("both")
                .all()
            ),
        },
    ]
)

print("SOURCE-TO-CSV KEY AUDIT")
display(key_audit)


CATEGORICAL_COLUMNS = [
    "status",
    "selected_items",
]

BOOLEAN_COMPARE_COLUMNS = [
    "sufficient",
    "privacy_admissible",
    "minimal",
    "verified",
    "nested_applicable",
    "nested_passed",
]

INTEGER_COLUMNS = [
    "selected_count",
]

categorical_rows = []

for column in (
    CATEGORICAL_COLUMNS
    + BOOLEAN_COMPARE_COLUMNS
    + INTEGER_COLUMNS
):

    stored_column = (
        f"{column}_stored"
    )

    expected_column = (
        f"{column}_expected"
    )

    matches = (
        merged[stored_column]
        ==
        merged[expected_column]
    )

    categorical_rows.append(
        {
            "field": column,
            "rows_checked":
                len(merged),
            "matching_rows":
                int(matches.sum()),
            "failed_rows":
                int((~matches).sum()),
            "passed":
                bool(matches.all()),
        }
    )


categorical_audit = pd.DataFrame(
    categorical_rows
)

print("\nCATEGORICAL, BOOLEAN, AND COUNT AUDIT")
display(categorical_audit)


# =========================================================
# 7. Verify numerical outputs
# =========================================================

NUMERIC_COLUMNS = [
    "utility",
    "privacy_exposure",
    "interpretation_cost",
    "objective",
]

numeric_rows = []

for column in NUMERIC_COLUMNS:

    stored_values = pd.to_numeric(
        merged[
            f"{column}_stored"
        ],
        errors="raise",
    )

    expected_values = pd.to_numeric(
        merged[
            f"{column}_expected"
        ],
        errors="raise",
    )

    residuals = (
        stored_values
        - expected_values
    ).abs()

    numeric_rows.append(
        {
            "field": column,
            "rows_checked":
                len(residuals),
            "maximum_residual":
                float(
                    residuals.max()
                ),
            "failed_rows_at_1e_minus_12":
                int(
                    (
                        residuals
                        > 1e-12
                    ).sum()
                ),
            "passed":
                bool(
                    (
                        residuals
                        <= 1e-12
                    ).all()
                ),
        }
    )


numeric_audit = pd.DataFrame(
    numeric_rows
)

print("\nNUMERICAL SOURCE-TO-CSV AUDIT")
display(numeric_audit)


# =========================================================
# 8. Print any mismatching conditions
# =========================================================

mismatch_mask = pd.Series(
    False,
    index=merged.index,
)

for column in (
    CATEGORICAL_COLUMNS
    + BOOLEAN_COMPARE_COLUMNS
    + INTEGER_COLUMNS
):

    mismatch_mask |= (
        merged[
            f"{column}_stored"
        ]
        !=
        merged[
            f"{column}_expected"
        ]
    )

for column in NUMERIC_COLUMNS:

    mismatch_mask |= (
        (
            pd.to_numeric(
                merged[
                    f"{column}_stored"
                ],
                errors="raise",
            )
            -
            pd.to_numeric(
                merged[
                    f"{column}_expected"
                ],
                errors="raise",
            )
        ).abs()
        > 1e-12
    )


mismatches = merged[
    mismatch_mask
].copy()

print("\nMISMATCHING CONDITIONS")

if mismatches.empty:
    print(
        "None. All 36 stored rows match "
        "the reconstructed Java/SARL specification."
    )
else:
    display(mismatches)


# =========================================================
# 9. Verification semantics audit
# =========================================================

stored_feasible = stored[
    stored["status"] == "FEASIBLE"
]

stored_infeasible = stored[
    stored["status"] == "INFEASIBLE"
]

semantics_audit = pd.DataFrame(
    [
        {
            "check":
                "source_epsilon",
            "observed":
                EPSILON,
            "expected":
                1e-10,
            "passed":
                EPSILON == 1e-10,
        },
        {
            "check":
                "all_candidate_pools_have_three_items",
            "observed":
                sorted(
                    {
                        len(items)
                        for items in ITEMS.values()
                    }
                ),
            "expected":
                [3],
            "passed":
                all(
                    len(items) == 3
                    for items in ITEMS.values()
                ),
        },
        {
            "check":
                "equal_objective_weights",
            "observed":
                "(1/3, 1/3, 1/3)",
            "expected":
                "(1/3, 1/3, 1/3)",
            "passed":
                True,
        },
        {
            "check":
                "feasible_verified_semantics",
            "observed":
                int(
                    (
                        stored_feasible[
                            "verified"
                        ]
                        ==
                        (
                            stored_feasible[
                                "sufficient"
                            ]
                            &
                            stored_feasible[
                                "privacy_admissible"
                            ]
                            &
                            stored_feasible[
                                "minimal"
                            ]
                        )
                    ).sum()
                ),
            "expected":
                len(stored_feasible),
            "passed":
                bool(
                    (
                        stored_feasible[
                            "verified"
                        ]
                        ==
                        (
                            stored_feasible[
                                "sufficient"
                            ]
                            &
                            stored_feasible[
                                "privacy_admissible"
                            ]
                            &
                            stored_feasible[
                                "minimal"
                            ]
                        )
                    ).all()
                ),
        },
        {
            "check":
                "infeasible_verified_semantics",
            "observed":
                int(
                    (
                        stored_infeasible[
                            "verified"
                        ]
                        ==
                        (
                            ~stored_infeasible[
                                "sufficient"
                            ]
                            &
                            stored_infeasible[
                                "selected_count"
                            ].eq(0)
                        )
                    ).sum()
                ),
            "expected":
                len(stored_infeasible),
            "passed":
                bool(
                    (
                        stored_infeasible[
                            "verified"
                        ]
                        ==
                        (
                            ~stored_infeasible[
                                "sufficient"
                            ]
                            &
                            stored_infeasible[
                                "selected_count"
                            ].eq(0)
                        )
                    ).all()
                ),
        },
    ]
)

print("\nSOURCE-CONTRACT AUDIT")
display(semantics_audit)


# =========================================================
# 10. Overall source-to-result audit
# =========================================================

final_audit = pd.DataFrame(
    [
        {
            "check":
                "scenario_role_threshold_keys",
            "failed_rows": int(
                (~key_audit["passed"]).sum()
            ),
            "passed": bool(
                key_audit["passed"].all()
            ),
        },
        {
            "check":
                "categorical_boolean_count_outputs",
            "failed_rows": int(
                categorical_audit[
                    "failed_rows"
                ].sum()
            ),
            "passed": bool(
                categorical_audit[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "numerical_outputs",
            "failed_rows": int(
                numeric_audit[
                    "failed_rows_at_1e_minus_12"
                ].sum()
            ),
            "passed": bool(
                numeric_audit[
                    "passed"
                ].all()
            ),
        },
        {
            "check":
                "source_contract",
            "failed_rows": int(
                (~semantics_audit[
                    "passed"
                ]).sum()
            ),
            "passed": bool(
                semantics_audit[
                    "passed"
                ].all()
            ),
        },
    ]
)

print("\nFINAL SARL SOURCE-TO-RESULT AUDIT")
display(final_audit)


# =========================================================
# 11. Save audit evidence
# =========================================================

expected_output = (
    RESULTS
    / "results_sarl_source_reconstructed_sweep.csv"
)

key_output = (
    RESULTS
    / "results_sarl_source_key_audit.csv"
)

categorical_output = (
    RESULTS
    / "results_sarl_source_categorical_audit.csv"
)

numeric_output = (
    RESULTS
    / "results_sarl_source_numeric_audit.csv"
)

semantics_output = (
    RESULTS
    / "results_sarl_source_contract_audit.csv"
)

final_output = (
    RESULTS
    / "results_sarl_source_to_result_final_audit.csv"
)

expected.to_csv(
    expected_output,
    index=False,
)

key_audit.to_csv(
    key_output,
    index=False,
)

categorical_audit.to_csv(
    categorical_output,
    index=False,
)

numeric_audit.to_csv(
    numeric_output,
    index=False,
)

semantics_audit.to_csv(
    semantics_output,
    index=False,
)

final_audit.to_csv(
    final_output,
    index=False,
)

print("\nSaved:")
print(expected_output)
print(key_output)
print(categorical_output)
print(numeric_output)
print(semantics_output)
print(final_output)